### Initialisation

In [1]:
import pandas as pd
import duckdb
from pathlib import Path
import pyarrow as pa

In [2]:
# Helpers for safely quoting column names in SQL
def quote_ident(name):
    return '"' + name.replace('"', '""') + '"'

In [3]:
CSV_FILE   = Path.cwd() / "usdot_insurance_events_2026-10-12.csv"
CACHE_DIR  = Path.cwd() / "cache"
CACHE_DIR.mkdir(exist_ok=True)
DUCKDB_PATH = CACHE_DIR / "analysis.duckdb"

# Flip to True to invalidate all caches (DuckDB file + parquet + pickle) on
# next run. Also useful during development after upstream SQL/feature changes.
FORCE_REBUILD = True

if not CSV_FILE.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_FILE}")

print(f"Dataset:    {CSV_FILE}")
print(f"File size:  {CSV_FILE.stat().st_size / 1024**3:.2f} GB")
print(f"Cache dir:  {CACHE_DIR}  (FORCE_REBUILD={FORCE_REBUILD})")

# Managed cache files — only these get wiped on FORCE_REBUILD. Any other
# files you've dropped into cache/ (prior-run pickles, side experiments,
# user artefacts) are preserved. Note that analysis.duckdb is NOT in this
# list — individual cells handle their own DuckDB table lifecycle via
# DROP TABLE statements guarded by FORCE_REBUILD.
_MANAGED_CACHE_FILES = {
    'vintage_and_provider_map.pkl', 'dot_df.parquet', 'line_tenure.pkl',
    # Model + scoring caches (per backend variant)
    'models_classifier.pkl',          'models_ranker.pkl',
    'scored_obs_rank_classifier.pkl', 'scored_obs_rank_ranker.pkl',
    'models_weighted_classifier.pkl',          'models_weighted_ranker.pkl',
    'scored_obs_rank_weighted_classifier.pkl', 'scored_obs_rank_weighted_ranker.pkl',
    # Classifier+ranker ensemble (built in Part 5A)
    'scored_obs_rank_ensemble.pkl',
}

if FORCE_REBUILD:
    _n_cleared = _n_preserved = 0
    for p in CACHE_DIR.iterdir():
        if not p.is_file():
            continue
        if p.name in _MANAGED_CACHE_FILES:
            p.unlink(); _n_cleared += 1
        else:
            _n_preserved += 1
    print(f"  cleared {_n_cleared} managed cache files; preserved {_n_preserved} other files")

# File-backed DuckDB — tables persist across kernel restarts so OOM in a
# downstream cell doesn't force a rebuild of everything upstream.
con = duckdb.connect(str(DUCKDB_PATH))
con.execute("SET memory_limit = '6GB'")
# 2 threads keeps per-thread hash-table partitions larger (less OOM risk on
# big GROUP BYs) and avoids 4-way contention over a tight 6 GB cap.
con.execute("SET threads = 2")
# Don't buffer rows for insertion-order preservation — saves memory on big
# aggregations where row order is irrelevant.
con.execute("SET preserve_insertion_order = false")

# Separate dir for DuckDB spill files (temporary, not the persistent DB).
temp_dir = Path.cwd() / "duckdb_temp"
temp_dir.mkdir(exist_ok=True)
con.execute(f"SET temp_directory = '{temp_dir.as_posix()}'")

csv_path = CSV_FILE.resolve().as_posix().replace("'", "''")

# `raw` is a view over the CSV — cheap to re-create every session; always refresh.
con.execute(f"""
    CREATE OR REPLACE VIEW raw AS
    SELECT * FROM read_csv_auto(
        '{csv_path}',
        sample_size = 100000
    )
""")

# Cache helpers used by downstream cells for idempotent builds.
def _table_exists(name):
    """True if a DuckDB table with the given name already exists on disk."""
    return bool(con.execute(
        f"SELECT 1 FROM duckdb_tables() WHERE table_name = '{name}'"
    ).fetchall())

def _cache_df(name):
    """Parquet path for a named pandas DataFrame cache."""
    return CACHE_DIR / f"{name}.parquet"

def _cache_pkl(name):
    """Pickle path for a named object cache (models, scored dicts, etc.)."""
    return CACHE_DIR / f"{name}.pkl"


Dataset:    c:\Users\hi\Documents\ROSELLA ASSIGNMENT\usdot_insurance_events_2026-10-12.csv
File size:  43.87 GB
Cache dir:  c:\Users\hi\Documents\ROSELLA ASSIGNMENT\cache  (FORCE_REBUILD=True)
  cleared 0 managed cache files; preserved 8 other files


## Exploratory Data Analysis

### 1. Schema: column names and inferred data types

In [4]:
print("\n" + "=" * 80)
print("1. SCHEMA")
print("=" * 80)

schema = con.execute("DESCRIBE raw").df()
print(schema.to_string(index=False))

columns = schema["column_name"].tolist()
numeric_types = {
    "TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT",
    "UTINYINT", "USMALLINT", "UINTEGER", "UBIGINT",
    "FLOAT", "REAL", "DOUBLE",
    "DECIMAL", "NUMERIC"
}

numeric_columns = [
    row["column_name"]
    for _, row in schema.iterrows()
    if any(
        str(row["column_type"]).upper().startswith(t)
        for t in numeric_types
    )
]

print(f"\nTotal columns: {len(columns)}")
print(f"Numeric columns: {len(numeric_columns)}")


1. SCHEMA
                                     column_name column_type null  key default extra
                                           usdot      BIGINT  YES None    None  None
                                    company_name     VARCHAR  YES None    None  None
                                        dba_name     VARCHAR  YES None    None  None
                                           state     VARCHAR  YES None    None  None
                             has_insurance_event     BOOLEAN  YES None    None  None
                                    event_source     VARCHAR  YES None    None  None
                          event_usdot_matched_by     VARCHAR  YES None    None  None
                                    event_status     VARCHAR  YES None    None  None
                                     policy_line     VARCHAR  YES None    None  None
                                         insurer     VARCHAR  YES None    None  None
                                   insurer_group     V

### 2. Sample records

In [5]:
print("\n" + "=" * 80)
print("2. FIRST 10 ROWS")
print("=" * 80)

print(con.execute("SELECT * FROM raw LIMIT 10").df().to_string(index=False))


2. FIRST 10 ROWS
 usdot                                company_name                  dba_name state  has_insurance_event event_source event_usdot_matched_by event_status policy_line insurer insurer_group coverage_class form_code class_code policy_number docket_or_certificate_number effective_date expiration_date cancellation_or_end_date end_reason cancellation_form_code filed_or_received_date rejection_reason  event_year years_covered all_insurers_this_usdot_in_event_year  insurer_count_this_usdot_in_event_year  coverage_amount_usd  underlying_limit_usd  min_required_coverage_usd  premium_usd rating_state  vehicle_count insurer_branch agent_name  is_peo  residual_market event_details_json event_company_signature  insurance_event_count  insurance_distinct_insurers insurance_first_event_date insurance_latest_event_date insurance_all_insurers insurance_active_bipd_insurers_as_of_2026_05_26 insurance_active_cargo_insurers_as_of_2026_05_26 insurance_wc_insurers  insurance_cancelled_count  

### 3. Total row count

In [6]:
print("\n" + "=" * 80)
print("3. TOTAL ROWS")
print("=" * 80)

row_count = con.execute(
    "SELECT COUNT(*) FROM raw"
).fetchone()[0]

print(f"Rows: {row_count:,}")


3. TOTAL ROWS
Rows: 12,243,741


In [7]:
expressions = []

for i, col in enumerate(columns):
    c = quote_ident(col)
    expressions.append(f"COUNT({c}) AS nonnull_{i}")

query = "SELECT " + ", ".join(expressions) + " FROM raw"
result = con.execute(query).fetchone()

profile = []

for i, col in enumerate(columns):
    nonnull = result[i]
    profile.append({
        "column": col,
        "null_count": row_count - nonnull,
        "null_pct": (
            (row_count - nonnull) / row_count * 100
            if row_count else 0
        )
    })

profile_df = pd.DataFrame(profile).sort_values(
    "null_pct", ascending=False
)

print(profile_df.to_string(index=False))

                                          column  null_count   null_pct
                                    co_sic_codes    12243741 100.000000
                                       co_county    12243741 100.000000
                               census_trplimo_16    12243641  99.999183
                              census_mail_barrio    12243601  99.998857
                           census_trpschool_9_15    12243594  99.998799
                            census_trpschool_1_8    12243568  99.998587
                               census_total_cars    12243558  99.998505
                             census_trplimo_9_15    12243482  99.997885
                             census_trpschool_16    12243400  99.997215
                              census_trplimo_1_8    12243384  99.997084
                               census_phy_barrio    12243310  99.996480
                            census_trmschool_1_8    12243202  99.995598
                  census_prior_revoke_dot_number    12243157  99

In [8]:
key_columns = [
    # Company identity
    "usdot",
    "company_name",
    "state",

    # Insurance event details
    "has_insurance_event",
    "event_source",
    "event_usdot_matched_by",
    "event_status",
    "event_year",
    "policy_line",
    "insurer",
    "insurer_group",
    "coverage_class",
    "policy_number",

    # Coverage and policy lifecycle
    "effective_date",
    "expiration_date",
    "cancellation_or_end_date",
    "end_reason",
    "filed_or_received_date",
    "rejection_reason",
    "coverage_amount_usd",
    "min_required_coverage_usd",
    "premium_usd",

    # Insurance history summaries
    "insurance_event_count",
    "insurance_distinct_insurers",
    "insurance_cancelled_count",
    "insurance_replaced_count",
    "insurance_rejected_count",

    # Fleet and safety profile
    "census_power_units",
    "census_total_drivers",
    "crash_count",
    "inspection_count",
    "revocation_count",
    "oos_order_count",

    # Business enrichment
    "co_primary_naics",
    "co_years_in_business",
    "co_employee_band",
    "co_match_confidence",
]

expressions = [
    f"APPROX_COUNT_DISTINCT({quote_ident(col)}) AS {quote_ident(col)}"
    for col in key_columns
]

query = "SELECT " + ", ".join(expressions) + " FROM raw"
print(con.execute(query).fetchdf())

     usdot  company_name  state  has_insurance_event  event_source  \
0  4205801       4222370    117                    2            13   

   event_usdot_matched_by  event_status  event_year  policy_line  insurer  \
0                       3            12          85            9     6453   

   ...  census_power_units  census_total_drivers  crash_count  \
0  ...                2577                  1722          266   

   inspection_count  revocation_count  oos_order_count  co_primary_naics  \
0               932                27                6              1420   

   co_years_in_business  co_employee_band  co_match_confidence  
0                   112                10                    2  

[1 rows x 37 columns]


### 5. DuckDB's built-in descriptive summary

Includes statistics appropriate to each column type

In [9]:
print("\n" + "=" * 80)
print("5. DESCRIPTIVE SUMMARY")
print("=" * 80)

summary = con.execute("SUMMARIZE raw").df()
print(summary.to_string(index=False))


5. DESCRIPTIVE SUMMARY
                                     column_name column_type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        min                         

### 6. Numeric statistics

In [10]:
if numeric_columns:
    print("\n" + "=" * 80)
    print("6. NUMERIC STATISTICS")
    print("=" * 80)

    expressions = []

    for i, col in enumerate(numeric_columns):
        c = quote_ident(col)

        expressions.extend([
            f"MIN({c}) AS min_{i}",
            f"MAX({c}) AS max_{i}",
            f"AVG(TRY_CAST({c} AS DOUBLE)) AS mean_{i}",
            f"STDDEV_SAMP(TRY_CAST({c} AS DOUBLE)) AS std_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.25) AS p25_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.50) AS median_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.75) AS p75_{i}"
        ])

    values = con.execute(
        "SELECT " + ", ".join(expressions) + " FROM raw"
    ).fetchone()

    numeric_profile = []

    for i, col in enumerate(numeric_columns):
        numeric_profile.append({
            "column": col,
            "min": values[i * 7],
            "max": values[i * 7 + 1],
            "mean": values[i * 7 + 2],
            "std": values[i * 7 + 3],
            "p25": values[i * 7 + 4],
            "median": values[i * 7 + 5],
            "p75": values[i * 7 + 6]
        })

    print(pd.DataFrame(numeric_profile).to_string(index=False))


6. NUMERIC STATISTICS
                                column           min          max          mean          std           p25        median           p75
                                 usdot  1.000000e+00 9.002022e+09  2.141387e+06 4.661490e+06  1.005510e+06  2.025180e+06  3.164565e+06
                            event_year  1.936000e+03 2.027000e+03  2.013818e+03 9.028065e+00  2.007000e+03  2.015421e+03  2.022000e+03
insurer_count_this_usdot_in_event_year  1.000000e+00 4.200000e+01  1.901650e+00 1.569668e+00  1.000000e+00  2.000000e+00  2.000000e+00
                   coverage_amount_usd  0.000000e+00 2.000000e+09  7.593126e+05 1.124660e+06  5.023452e+05  7.500000e+05  1.000000e+06
                  underlying_limit_usd  0.000000e+00 9.285800e+07  1.022706e+04 1.412732e+05  0.000000e+00  0.000000e+00  0.000000e+00
             min_required_coverage_usd  0.000000e+00 1.000000e+07  8.745397e+05 7.346829e+05  7.500000e+05  7.500000e+05  7.500000e+05
                           premi

In [12]:
query = """
WITH values_long AS (
    SELECT 'insurer' AS column_name,
           COALESCE(NULLIF(TRIM(CAST(insurer AS VARCHAR)), ''), '<NULL/BLANK>') AS value
    FROM raw

    UNION ALL
    SELECT 'insurer_group',
           COALESCE(NULLIF(TRIM(CAST(insurer_group AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'event_source',
           COALESCE(NULLIF(TRIM(CAST(event_source AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'event_status',
           COALESCE(NULLIF(TRIM(CAST(event_status AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'policy_line',
           COALESCE(NULLIF(TRIM(CAST(policy_line AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'coverage_class',
           COALESCE(NULLIF(TRIM(CAST(coverage_class AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'form_code',
           COALESCE(NULLIF(TRIM(CAST(form_code AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'class_code',
           COALESCE(NULLIF(TRIM(CAST(class_code AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'rating_state',
           COALESCE(NULLIF(TRIM(CAST(rating_state AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'all_insurers_this_usdot_in_event_year',
           COALESCE(NULLIF(TRIM(CAST(all_insurers_this_usdot_in_event_year AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_all_insurers',
           COALESCE(NULLIF(TRIM(CAST(insurance_all_insurers AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_active_bipd_insurers_as_of_2026_05_26',
           COALESCE(NULLIF(TRIM(CAST(insurance_active_bipd_insurers_as_of_2026_05_26 AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_active_cargo_insurers_as_of_2026_05_26',
           COALESCE(NULLIF(TRIM(CAST(insurance_active_cargo_insurers_as_of_2026_05_26 AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_wc_insurers',
           COALESCE(NULLIF(TRIM(CAST(insurance_wc_insurers AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_carriers_by_year',
           COALESCE(NULLIF(TRIM(CAST(insurance_carriers_by_year AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw
),
freq AS (
    SELECT
        column_name,
        value,
        COUNT(*) AS frequency
    FROM values_long
    GROUP BY column_name, value
)
SELECT
    column_name,
    value,
    frequency,
    ROUND(
        100.0 * frequency /
        SUM(frequency) OVER (PARTITION BY column_name),
        3
    ) AS pct_of_rows
FROM freq
ORDER BY column_name, frequency DESC
"""

# Execute using your existing DuckDB connection
result = con.execute(query).fetchdf()

# Display the full distribution
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 150)

display(result)

RuntimeError: Query interrupted

In [13]:
categorical_columns = [
    "insurer",
    "insurer_group",
    "event_source",
    "event_status",
    "policy_line",
    "coverage_class",
    "form_code",
    "class_code",
    "rating_state",
]

values_clause = ",\n".join(
    f"('{col}', CAST({quote_ident(col)} AS VARCHAR))"
    for col in categorical_columns
)

query = f"""
WITH values_long AS (
    SELECT
        v.column_name,
        COALESCE(
            NULLIF(TRIM(v.value), ''),
            '<NULL/BLANK>'
        ) AS value
    FROM raw
    CROSS JOIN LATERAL (
        VALUES
            {values_clause}
    ) AS v(column_name, value)
),
freq AS (
    SELECT
        column_name,
        value,
        COUNT(*) AS frequency
    FROM values_long
    GROUP BY column_name, value
),
ranked AS (
    SELECT
        column_name,
        value,
        frequency,
        ROUND(
            100.0 * frequency /
            SUM(frequency) OVER (
                PARTITION BY column_name
            ),
            3
        ) AS pct_of_rows,
        ROW_NUMBER() OVER (
            PARTITION BY column_name
            ORDER BY frequency DESC
        ) AS rn
    FROM freq
)
SELECT
    column_name,
    value,
    frequency,
    pct_of_rows
FROM ranked
WHERE rn <= 50
ORDER BY column_name, frequency DESC
"""

result = con.execute(query).fetchdf()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 150)

display(result)

,column_name,value,frequency,pct_of_rows
0,class_code,<NULL/BLANK>,5953477,48.625
1,class_code,P,5905367,48.232
2,class_code,E,60480,0.494
3,class_code,5190,19960,0.163
4,class_code,3724,17804,0.145
5,class_code,5606,13566,0.111
6,class_code,7219,12226,0.100
7,class_code,5183,12133,0.099
8,class_code,8810,9713,0.079
9,class_code,5213,9686,0.079


### Discovery queries (for labelling scheme + scoping)

Four follow-up queries to settle open questions before modelling:

- **Q1** — does `end_reason` cleanly split carrier-initiated endings from insured-initiated ones?
- **Q2** — how many distinct DOTs sit in each "current-placement" cohort (Tier A pool size)?
- **Q3** — are there enough short-tenure cancellations to serve as a wrong-fit signal?
- **Q4** — are the `residual_market` flags populated with usable values for validation?


### Discovery Q1: end_reason x event_status breakdown

Does end_reason distinguish carrier-initiated non-renewal from
insured-initiated changes? If so, carrier-initiated endings
become usable hard-negative labels for the fit model; insured-
initiated endings should NOT be labelled wrong-fit.

In [11]:
query = """
WITH e AS (
    SELECT
        COALESCE(NULLIF(TRIM(CAST(end_reason   AS VARCHAR)), ''), '<NULL>') AS end_reason,
        COALESCE(NULLIF(TRIM(CAST(event_status AS VARCHAR)), ''), '<NULL>') AS event_status
    FROM raw
    WHERE end_reason IS NOT NULL
)
SELECT
    end_reason,
    event_status,
    COUNT(*) AS n,
    ROUND(
        100.0 * COUNT(*)
        / SUM(COUNT(*)) OVER (PARTITION BY end_reason),
        2
    ) AS pct_within_end_reason
FROM e
GROUP BY end_reason, event_status
ORDER BY end_reason, n DESC
"""

pd.set_option("display.max_rows", 500)
display(con.execute(query).fetchdf())


,end_reason,event_status,n,pct_within_end_reason
0,CANC/CANCL,Cancelled,6379,100.00
1,CANCEL,Cancelled,3862192,100.00
2,CANCEL,Name Changed,2,0.00
3,CANCEL,Transferred,2,0.00
4,DUPLICATE,Cancelled,449,100.00
5,NAMECHG,Name Changed,68241,100.00
6,REVOKED,Cancelled,1610,100.00
7,TERM/CANCL,Cancelled,914988,89.87
8,TERM/CANCL,Name Changed,102148,10.03
9,TERM/CANCL,Transferred,1038,0.10


### Discovery Q2: Distinct-DOT counts per 'current-placement' field

Establishes how many DOTs we can actually score for mismatch
(Tier A), vs how many have authority but no active placement
(Tier B / 'uncovered' candidates we can't trust without
knowing data completeness).

In [12]:
query = """
SELECT 'total_distinct_dots' AS cohort,
       COUNT(DISTINCT usdot) AS n_dots
FROM raw

UNION ALL

SELECT 'has_insurance_event = TRUE',
       COUNT(DISTINCT usdot)
FROM raw
WHERE has_insurance_event = TRUE

UNION ALL

SELECT 'any_insurance_active_bipd_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_active_bipd_insurers_as_of_2026_05_26 IS NOT NULL

UNION ALL

SELECT 'any_insurance_active_cargo_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_active_cargo_insurers_as_of_2026_05_26 IS NOT NULL

UNION ALL

SELECT 'any_insurance_wc_insurers_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_wc_insurers IS NOT NULL

UNION ALL

SELECT 'any_co_wc_carrier_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE co_wc_carrier IS NOT NULL

UNION ALL

SELECT 'any_active_bipd_OR_cargo_OR_wc',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_active_bipd_insurers_as_of_2026_05_26  IS NOT NULL
   OR insurance_active_cargo_insurers_as_of_2026_05_26 IS NOT NULL
   OR insurance_wc_insurers                            IS NOT NULL

UNION ALL

SELECT 'any_authority_filing',
       COUNT(DISTINCT usdot)
FROM raw
WHERE authority_common_stat   IS NOT NULL
   OR authority_contract_stat IS NOT NULL
   OR authority_broker_stat   IS NOT NULL

UNION ALL

SELECT 'has_authority_but_no_active_bipd',
       COUNT(DISTINCT usdot)
FROM raw
WHERE (authority_common_stat   IS NOT NULL
    OR authority_contract_stat IS NOT NULL
    OR authority_broker_stat   IS NOT NULL)
  AND insurance_active_bipd_insurers_as_of_2026_05_26 IS NULL

ORDER BY n_dots DESC
"""

display(con.execute(query).fetchdf())


,cohort,n_dots
0,total_distinct_dots,4517323
1,any_authority_filing,1679119
2,has_insurance_event = TRUE,1497586
3,has_authority_but_no_active_bipd,1293780
4,any_active_bipd_OR_cargo_OR_wc,461960
5,any_insurance_active_bipd_set,388724
6,any_insurance_wc_insurers_set,70303
7,any_co_wc_carrier_set,70212
8,any_insurance_active_cargo_set,35076


### Discovery Q3: Time-to-end distribution for ended policies

Short-tenure endings (<90 days in force) are much more likely
to indicate wrong-fit (mid-term declines, misrepresentation
discovered, early non-renewal) than full-term replacements.
This tells us whether we have a usable short-tenure signal.

In [13]:
query = """
WITH t AS (
    SELECT
        DATE_DIFF('day', effective_date, cancellation_or_end_date) AS days_in_force,
        COALESCE(NULLIF(TRIM(CAST(event_status AS VARCHAR)), ''), '<NULL>') AS event_status
    FROM raw
    WHERE effective_date IS NOT NULL
      AND cancellation_or_end_date IS NOT NULL
)
SELECT
    CASE
        WHEN days_in_force < 0    THEN '<0 (data issue)'
        WHEN days_in_force < 30   THEN '00-29 days'
        WHEN days_in_force < 90   THEN '30-89 days'
        WHEN days_in_force < 180  THEN '90-179 days'
        WHEN days_in_force < 365  THEN '180-364 days'
        WHEN days_in_force < 730  THEN '365-729 days'
        ELSE                           '730+ days'
    END AS tenure_bucket,
    event_status,
    COUNT(*) AS n
FROM t
GROUP BY tenure_bucket, event_status
ORDER BY event_status, tenure_bucket
"""

pd.set_option("display.max_rows", 200)
display(con.execute(query).fetchdf())


,tenure_bucket,event_status,n
0,00-29 days,<NULL>,10810
1,180-364 days,<NULL>,6519
2,30-89 days,<NULL>,5380
3,365-729 days,<NULL>,19424
4,730+ days,<NULL>,3276
5,90-179 days,<NULL>,5428
6,<0 (data issue),<NULL>,39
7,180-364 days,Active,1040
8,30-89 days,Active,1018
9,365-729 days,Active,2640


### Discovery Q4: residual-market indicators

Where populated, these are hard-labelled wrong-fit signals
(the DOT couldn't find a voluntary carrier). Confirms whether
the values distinguish anything useful for validation negatives.

In [14]:
query = """
SELECT 'residual_market' AS field,
       CAST(residual_market AS VARCHAR) AS value,
       COUNT(*) AS n
FROM raw
WHERE residual_market IS NOT NULL
GROUP BY 1, 2

UNION ALL

SELECT 'co_wc_residual_market' AS field,
       CAST(co_wc_residual_market AS VARCHAR) AS value,
       COUNT(*) AS n
FROM raw
WHERE co_wc_residual_market IS NOT NULL
GROUP BY 1, 2

ORDER BY field, value
"""

display(con.execute(query).fetchdf())


,field,value,n
0,co_wc_residual_market,false,268684
1,co_wc_residual_market,true,26536
2,residual_market,false,265843
3,residual_market,true,9312


### Mismatch model — binary contrastive fit, 6 policy lines, 4-dimension feature set

**Framing:** binary contrastive. For each `(DOT, carrier, state)` pair, model learns `P(real pair | DOT profile, carrier aggregate profile)`. Observed carrier's rank among all in-state candidates = **appetite outlier score** (reframed from "mismatch" to be honest about PU contamination).

**Grain:** DOT × policy_line × active carrier. Scorable pool is the 462k DOTs with any active placement across the 6 lines.

**Six per-line models (one per policy line):**
- BIPD (auto liability) — ~388k DOTs, pre-computed snapshot
- Cargo — ~35k DOTs, pre-computed snapshot
- WC — ~70k DOTs, pre-computed snapshot
- Broker surety bond — ~30-50k DOTs, event-reconstructed active placement
- Surety (state-level) — ~2-3k DOTs, thin
- GL — ~2-3k DOTs, thin

**Carrier identity:** `(carrier_group, state)` with per-line min-N thresholds (BIPD/Cargo/WC = 20, surety_bond = 15, surety/gl = 10). Rule-based consolidation on top insurer families.

**Carrier aggregates:** from all events in last 24 months (not just currently-active placements), state-partitioned. Captures current appetite, not decade-old patterns.

**Labelling (clean positives + hard negatives):**
- Positives (y=1): observed placements where `is_residual_market=0 AND is_cancellation_pending=0`
- Hard negatives (y=0): observed placements with `is_residual_market=1` + events with `end_reason='REVOKED'`
- Soft negatives (y=0): K=3 random carriers per positive from same state's pool, anti-joined vs real placements

**Features across 4 fit dimensions** (as features, not label weights):

| dimension | DOT-side | carrier-side (per state) |
|---|---|---|
| appetite | fleet, commodity, hazmat, operation, state, authority, radius | fleet size dist (mean/median/p90), commodity mix, radius mix |
| price proxies | `is_residual_market`, rejected_count, coverage_vs_min, has_layered_coverage, wc_rate_multiplier | — |
| coverage | coverage_amount, authority_min, underlying_limit | — |
| service | — | retention/short-cancel/voluntary-exit/rejection composite |
| timing | days_since_effective, days_until_expiration, years_with_current_carrier | — |

**Baselines:**
- B1: random-in-state
- B2: popularity by state × fleet-size-band

**Validation:**
- Grouped 3-fold CV on `usdot`
- Top-K hit rate (K=1, 5, 10) + MRR per line
- Segmented by state, fleet band
- Long-tenure diagnostic (placements with ≥3 years tenure should score HIGH; replaces residual-market diagnostic since residual-market now trains as hard negative)

**Sub-entity breakdown (Option B — weighted Euclidean distance):** within each recommended group, rank sub-entities by appetite distance to the DOT (not by volume share). Fleet size, risk level, commodity mix, operating radius. Weights derived empirically from within-group variance.

**Output artifacts:**
- Wide CSV: top 10k DOTs × top-20 recommended groups (with best-fit sub-entity per group)
- Long CSV: top-100 full ranking per DOT per line (with `is_observed` flag)

**Compute backend:** LightGBM where available (2-3× faster); falls back to `HistGradientBoostingClassifier`.

**Known limitations (methodology section):**
- PU contamination — positives contain inertia placements we can't identify; output reframed as "appetite outliers"
- No ground truth for mismatch — primary validation is ranking consistency vs baselines, not direct
- Rosella appointment list unknown — some top-K recommendations may name carriers Rosella can't appoint
- Agent/channel effects confounded with appetite (99.9% null)
- Snapshot staleness (May 2026 vs Oct 2026)
- Service composite is a behavioural proxy, not measured service quality
- Price-proxy features are weak (premium 98.4% null)
- MCS-150 fleet attributes can be stale
- No carrier state licensing data — approximated by "has recent placements in state"
- Carrier state footprint changes only partially handled by 24-month filter (recent entries may be dropped, recent exits may be falsely recommended)
- Hand-tuned priority formula (fleet weight = 0.1)


## Phase 1 — Setup, Carrier Consolidation & Feature Build

Config, carrier-consolidation rules (hybrid: data-provider map + regex rules + raw fallback), data vintage computation (anchored to CSV event dates, sanitised against year-9xxx outliers), sanity check against the snapshot column, and the single-table `dot_features` build (chunked into two passes to avoid OOM).

### Part 1A — Imports, config, logging utilities

Run order dependency: this cell must run before everything else.

In [4]:
import re
import gc
import time
import pickle
import joblib
import numpy as np
import pandas as pd
from datetime import date
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import lightgbm as lgb

# Elapsed-time logger — call _log() at every stage boundary
_T0 = time.time()
def _elapsed():
    return f"{time.time() - _T0:.1f}s"
def _log(msg):
    print(f"[t={_elapsed()}] {msg}")

# Memory logger
try:
    import psutil
    _proc = psutil.Process()
    def mem_gb(): return _proc.memory_info().rss / 1024**3
except ImportError:
    def mem_gb(): return 0.0
def _log_mem(label):
    print(f"  [mem] {label}: {mem_gb():.2f} GB")

# Config constants
RANDOM_SEED              = 42
N_NEGATIVES_PER_POSITIVE = 3
RECENCY_WINDOW_MONTHS    = 24
TOP_N_OUTPUT             = 10_000
TOP_K_PER_DOT            = 20
TOP_K_FULL_RANKING       = 100

# NOTE: 'surety' and 'gl' intentionally excluded (see LINES_IN_SCOPE comment
# below). surety_bond is the FMCSA-tracked BMC-84 broker bond and has clean
# event_status data; 'surety' (non-BMC-84 bonds) and 'gl' do not.
MIN_CARRIER_PLACEMENTS = {
    'bipd': 20, 'cargo': 20, 'wc': 20,
    'surety_bond': 15,
}
MIN_SUB_ENTITY_PLACEMENTS = 10
# Carriers with >= this share of residual-market events are excluded from the
# recommendation pool. They remain in training as hard negatives, so the model
# still learns to avoid them for voluntary-market DOTs.
RESIDUAL_MARKET_CARRIER_THRESHOLD = 0.5
# Intra-state DOT batching for cell 4A (streaming scoring). A state's
# cross-join is len(state_dots) * len(state_cells) rows; big states with
# tens of thousands of DOTs blow past 6 GB with the full cross. Batching
# the DOTs caps peak memory per iteration.
SCORING_BATCH_DOTS = 3000

# Weighted-strict labelling (Parts 4E/4F). Rather than filter positives on
# tenure (which excludes younger DOTs), we tier them by confidence and pass
# sample weights to LightGBM. Younger DOTs stay in training but are
# down-weighted. Per-line thresholds reflect how fast each line's placements
# "mature" (BIPD annual renewal cycle ~1.5y; cargo/surety less data available).
STRICT_TENURE_MIN_PER_LINE = {
    'bipd':        1.5,
    'wc':          1.0,
    'cargo':       0.5,
    'surety_bond': 0.5,
}
STRICT_CANCEL_RATE_MAX = 0.3
POS_WEIGHT_STRONG = 1.0   # tenure >= threshold AND cancel_rate OK AND clean
POS_WEIGHT_MEDIUM = 0.6   # tenure in [0.5y, threshold) AND clean
POS_WEIGHT_WEAK   = 0.3   # tenure < 0.5y OR tenure unknown (still clean)
NEG_WEIGHT_HARD   = 1.0
NEG_WEIGHT_SOFT   = 0.3

# Phase 7 (Part 7A) temporal validation: hold out last N months of placements
# as a true out-of-sample test set. Was 3m; raised to 6m because 3m gave BIPD
# n=111 (unusable for a point estimate). 6m yields BIPD n~5-10K while still
# leaving 75% of events for training.
TEMPORAL_SPLIT_MONTHS = 6

# Short-window carrier aggregates: in addition to the 24m `carr_*` features,
# we compute 6m variants (e.g. `carr_mean_power_6m`). The 24m window smooths
# over carrier appetite drift; a 6m variant exposes recent shifts. Set to
# None to disable the short-window features entirely.
SHORT_WINDOW_MONTHS = 6

# Model backends trained in a single run — classifier (LGBMClassifier,
# binary cross-entropy) and ranker (LGBMRanker, lambdarank objective).
# Separate caches per backend; downstream aliases default to 'classifier'.
# MODEL_BACKEND_TYPE is a module-level variable that training/scoring cells
# temporarily set during each per-backend pass.
BACKEND_TYPES = ['classifier', 'ranker']
MODEL_BACKEND_TYPE = BACKEND_TYPES[0]

# Which model's output to use for the Phase 5 deliverable CSV.
#   'base'     = base labelling (clean_mask only)            — cell 36 trains MODELS
#   'weighted' = tiered weights by line-tenure / cancel-rate — cell 42 trains MODELS_WEIGHTED
# Weighted is more directly aligned with the task framing ("appropriately placed")
# but costs 0.3-4.5 pts on standard metrics. See Part 6B for the comparison.
MODEL_FOR_DELIVERABLE = 'weighted'

APPETITE_DIST_FEATURES = [
    'power_units_log', 'crash_count', 'inspection_oos_viol',
    'crgo_genfreight', 'crgo_bldgmat', 'crgo_livestock',
    'crgo_oilfield', 'crgo_intermodal', 'crgo_chem', 'crgo_passengers',
]

# Four lines retained in modelling scope:
#   - auto liability / auto  → mapped to 'bipd' (FMCSA core: motor-carrier BIPD)
#   - cargo                  → cargo liability
#   - wc                     → workers compensation
#   - broker surety bond     → mapped to 'surety_bond' (BMC-84 freight-broker bond)
#
# Two excluded (see Part 3B diagnostic for evidence):
#   - 'surety' (non-BMC-84 bonds): 3,274 DOTs but event_status is systematically
#     NULL in the raw data, so we can't distinguish active from historical
#     placements. Likely sourced from a different upstream feed than the
#     FMCSA-filed lines above. Treating latest-event-as-current would silently
#     mislabel ex-customers as positives.
#   - 'gl' (general liability): 7,659 DOTs, same NULL-status issue. GL is not
#     mandated by FMCSA for motor carriers, so filings come from a secondary
#     source without status tracking.
# BIPD/cargo/WC/surety_bond together cover the FMCSA-mandated trucking
# insurance stack and 92%+ of DOT-line pairs with usable data.
LINES_IN_SCOPE = ['auto liability', 'auto', 'cargo', 'wc', 'broker surety bond']

SNAPSHOT_DATE = pd.Timestamp('2026-05-26')  # from insurance_active_bipd_insurers_as_of_...

rng = np.random.default_rng(RANDOM_SEED)
_log("Part 1A complete — config loaded")


[t=0.0s] Part 1A complete — config loaded


### Part 1B — Carrier consolidation rules (tiered)

78 rules organized by confidence tier. See `rule_vs_provider_disagreements.csv`
(written in Part 1C) and `carrier_consolidation_report.csv` (Part 2A) to spot-check.

In [5]:
_log("starting Part 1B")

CARRIER_RULES = [
    # 3-tuples: (regex, group_name, confidence_tier)
    #
    # Tier 1 — High confidence. Industry-well-known structure, regex is
    #          specific. Safe to trust.
    # Tier 2 — Moderate confidence. Known acquisitions / documented
    #          sub-entities. Verified from training data but not web.
    # Tier 3 — Lower confidence. Regex may over-match, OR sub-entity
    #          ownership details are less certain. VERIFY.
    # Tier 4 — Speculative. Added without careful verification of regex
    #          specificity or exact sub-entity ownership. VERIFY.
    #
    # User action: Review Tier 3/4 rules in `rule_vs_provider_disagreements.csv`
    # (runtime diagnostic). Where the data provider disagrees with our rule,
    # the provider's answer typically wins (edit CARRIER_RULES accordingly).

    # ================= TIER 1 — HIGH CONFIDENCE =================
    # Progressive Corporation family (well-documented commercial trucking book)
    (r'PROGRESSIVE',              'PROGRESSIVE GRP', 1),
    (r'ARTISAN\s*&?\s*TRUCKERS',  'PROGRESSIVE GRP', 1),
    (r'DRIVE NEW JERSEY',         'PROGRESSIVE GRP', 1),
    (r'UNITED FINANCIAL CASUALTY','PROGRESSIVE GRP', 1),
    (r'NATIONAL CONTINENTAL',     'PROGRESSIVE GRP', 1),

    # Berkshire Hathaway core (NICO + NLF + Homestate + Direct + GEICO Marine)
    (r'BERKSHIRE HATHAWAY',       'BERKSHIRE HATHAWAY GRP', 1),
    (r'NATIONAL INDEMNITY',       'BERKSHIRE HATHAWAY GRP', 1),
    (r'NATIONAL LIABILITY',       'BERKSHIRE HATHAWAY GRP', 1),
    (r'GEICO MARINE',             'BERKSHIRE HATHAWAY GRP', 1),

    # Canal Insurance Group
    (r'CANAL',                    'CANAL GRP', 1),

    # Travelers (incl. Northland commercial trucking book, acquired 2001)
    (r'NORTHLAND',                'TRAVELERS GRP', 1),
    (r'TRAVELERS',                'TRAVELERS GRP', 1),

    # Zurich North America
    (r'ZURICH',                   'ZURICH INS GRP', 1),
    (r'EMPIRE FIRE',              'ZURICH INS GRP', 1),

    # Standalone/well-known trucking carriers
    (r'GREAT WEST CASUALTY',      'GREAT WEST', 1),
    (r'HALLMARK',                 'HALLMARK GRP', 1),
    (r'OOIDA',                    'OOIDA RRG', 1),
    # NOTE: removed broad LLOYD'?S rule. In Texas, "Lloyds" is a corporate
    # structure (not actual Lloyd's of London), so e.g. HARTFORD LLOYDS INS CO
    # is a Hartford subsidiary, not Lloyd's. Those are better left to the
    # provider map / raw fallback.
    (r'UNDERWRITERS AT LLOYD',    'LLOYDS LONDON', 1),
    (r'SENTRY',                   'SENTRY INS GRP', 1),
    (r'TRISURA',                  'TRISURA', 1),

    # IAT main (Occidental Fire + explicit Harco)
    (r'OCCIDENTAL FIRE',          'IAT GRP', 1),

    # AmTrust main brand + Wesco (its main US arm)
    (r'WESCO',                    'AMTRUST NGH GRP', 1),
    (r'AMTRUST',                  'AMTRUST NGH GRP', 1),

    # W.R. Berkley (Carolina Casualty is well-known subsidiary)
    (r'CAROLINA CASUALTY',        'WR BERKLEY CORP GRP', 1),
    (r'W\.?\s*R\.?\s*BERKLEY',    'WR BERKLEY CORP GRP', 1),

    # Fairfax Financial (Hudson + Crum & Forster are well-known subsidiaries)
    (r'HUDSON',                   'FAIRFAX FIN GRP', 1),
    (r'CRUM.{0,3}FORSTER',        'FAIRFAX FIN GRP', 1),

    # American Financial Group / Great American + National Interstate (AFG subsidiaries)
    (r'NATIONAL INTERSTATE',      'AMERICAN FINANCIAL GRP', 1),
    (r'GREAT AMERICAN',           'AMERICAN FINANCIAL GRP', 1),

    # Kemper (rebranded from Unitrin in 2011)
    (r'UNITRIN',                  'KEMPER GRP', 1),
    (r'KEMPER',                   'KEMPER GRP', 1),

    # Specialty standalone
    (r'PACIFIC FINANCIAL',        'PACIFIC FIN', 1),
    (r'LINCOLN GENERAL',          'LINCOLN GENERAL', 1),

    # Nationwide core (National Casualty is Nationwide subsidiary)
    # Anchored so "OMAHA NATIONAL CASUALTY COMPANY" (independent Nebraska
    # insurer) doesn't collapse into Nationwide.
    (r'^NATIONAL CASUALTY',       'NATIONWIDE CORP GRP', 1),
    (r'NATIONWIDE',               'NATIONWIDE CORP GRP', 1),

    # Allstate core
    (r'ALLSTATE',                 'ALLSTATE GRP', 1),

    # Liberty Mutual core
    # NOTE: removed LIBERTY INS catch-all. It matched e.g. AMERICAN LIBERTY
    # INS CO (actually Benchmark Holdings). Specific Liberty subs that provider
    # doesn't map will fall through to raw-fallback.
    (r'LIBERTY MUTUAL',           'LIBERTY MUT GRP', 1),

    # State WC funds
    (r'TEXAS MUTUAL',             'TEXAS MUTUAL INS CO', 1),
    (r'PINNACOL',                 'PINNACOL ASSURANCE', 1),
    (r'STATE FUND',               'STATE FUND', 1),

    # Hartford — anchored so "NATIONAL FIRE INS CO OF HARTFORD" (actually
    # a CNA subsidiary) doesn't get swept in by the trailing-HARTFORD match.
    (r'^HARTFORD\b',              'HARTFORD FIRE & CAS GRP', 1),

    # State Auto — removed; Liberty Mutual acquired State Auto Financial in
    # 2022, so provider map (which routes to LIBERTY MUT GRP) is correct now.

    # Independent / well-known standalone
    (r'ACUITY',                   'ACUITY', 1),
    # GEICO — three variants show up in the raw data (COUNTY MUTUAL INSURANCE
    # COMPANY, same with CO. suffix, GENERAL INSURANCE COMPANY). All are
    # Berkshire Hathaway subsidiaries. Combined: ~32K events.
    (r'\bGEICO\b',                'BERKSHIRE HATHAWAY GRP', 1),
    (r'AUTO.{0,2}OWNERS',         'AUTO OWNERS GRP', 1),
    (r'CLEAR BLUE',               'CLEAR BLUE', 1),
    (r'QUALITAS',                 'QUALITAS', 1),
    (r'DB INSURANCE',             'DB INS GRP', 1),
    (r'MERITZ',                   'MERITZ GRP', 1),

    # Builders Mutual — restricted so later T4 rules can match
    # BUILDERS INS / MIDWEST BUILDERS (which route to their own groups)
    (r'BUILDERS MUT',             'BUILDERS GRP', 1),

    # ================= TIER 2 — MODERATE CONFIDENCE =================
    # Known acquisitions, documented sub-entities

    # IAT subsidiaries beyond Occidental Fire
    (r'WILSHIRE',                 'IAT GRP', 2),
    (r'HARCO NATIONAL',           'IAT GRP', 2),

    # Lancer → Core Specialty (2022 acquisition)
    (r'LANCER',                   'CORE SPECIALTY GRP', 2),

    # Markel / State National (Markel acquired State National 2017)
    (r'UNITED SPECIALTY',         'MARKEL CORP GRP', 2),
    (r'STATE NATIONAL',           'MARKEL CORP GRP', 2),
    (r'MARKEL',                   'MARKEL CORP GRP', 2),

    # Accredited (post Randall & Quilter spinoff; now Onex-owned)
    (r'ACCREDITED',               'RANDALL & QUILTER INVESTMENT GRP', 2),

    # National General / Integon (Allstate acquired 2021 — kept separate for appetite distinction)
    (r'INTEGON',                  'NATIONAL GENERAL / INTEGON', 2),
    (r'NATIONAL GENERAL',         'NATIONAL GENERAL / INTEGON', 2),

    # Nationwide E&S arm
    (r'SCOTTSDALE',               'NATIONWIDE CORP GRP', 2),

    # Liberty Mutual acquisitions
    (r'SAFECO',                   'LIBERTY MUT GRP', 2),   # acquired 2008
    (r'IRONSHORE',                'LIBERTY MUT GRP', 2),   # acquired 2017

    # Chubb (post ACE merger 2016; Federal Insurance is Chubb subsidiary)
    (r'ACE AMERICAN',             'CHUBB LTD GRP', 2),
    (r'ACE PROPERTY',             'CHUBB LTD GRP', 2),
    (r'CHUBB',                    'CHUBB LTD GRP', 2),
    (r'FEDERAL INSURANCE',        'CHUBB LTD GRP', 2),

    # CNA (Continental Casualty + Transportation Ins are CNA subsidiaries)
    (r'\bCNA\b',                 'CNA INS GRP', 2),
    (r'CONTINENTAL CASUALTY',     'CNA INS GRP', 2),
    (r'TRANSPORTATION INS',       'CNA INS GRP', 2),

    # Everest Re
    (r'EVEREST',                  'EVEREST REINS HOLDINGS GRP', 2),

    # Allstate acquisitions
    (r'ENCOMPASS',                'ALLSTATE GRP', 2),
    (r'ESURANCE',                 'ALLSTATE GRP', 2),

    # Nationwide acquisitions
    (r'HARLEYSVILLE',             'NATIONWIDE CORP GRP', 2),   # acquired 2012

    # Baldwin & Lyons / Protective / Sagamore — provider still groups these
    # under BALDWIN & LYONS GRP (not Progressive), so follow provider.
    (r'PROTECTIVE INSURANCE',     'BALDWIN & LYONS GRP', 2),
    (r'BALDWIN.{0,3}LYONS',       'BALDWIN & LYONS GRP', 2),
    (r'SAGAMORE INS',             'BALDWIN & LYONS GRP', 2),

    # ================= TIER 3 — LOWER CONFIDENCE (verify) =================
    # Regex may over-match OR sub-entity ownership less certain

    # Berkshire — Columbia Insurance may collide with Columbia Mutual (independent)
    (r'COLUMBIA INSURANCE',       'BERKSHIRE HATHAWAY GRP', 3),
    (r'GUARD INSURANCE',          'BERKSHIRE HATHAWAY GRP', 3),

    # Fairfax — Zenith, Allied World
    (r'ZENITH',                   'FAIRFAX FIN GRP', 3),
    (r'ALLIED WORLD',             'FAIRFAX FIN GRP', 3),

    # Nationwide — "Allied" could over-match
    (r'ALLIED PROPERTY',          'NATIONWIDE CORP GRP', 3),

    # Hanover subsidiary
    (r'MASSACHUSETTS BAY INS',    'THE HANOVER INS GRP', 3),

    # Argo Group (Colony Specialty is Argo E&S)
    (r'COLONY (SPECIALTY|INS)',   'ARGONAUT GRP', 3),

    # Tokio Marine subsidiaries
    (r'PHILADELPHIA (CONSOL|INDEM|INS)', 'TOKIO MARINE HOLDINGS INC GRP', 3),
    (r'HOUSTON CASUALTY',         'TOKIO MARINE HOLDINGS INC GRP', 3),

    # AIG subsidiaries (specific sub-entity names)
    (r'LEXINGTON INSURANCE',      'AMERICAN INTL GRP', 3),
    (r'NATIONAL UNION FIRE',      'AMERICAN INTL GRP', 3),
    (r'NEW HAMPSHIRE INS',        'AMERICAN INTL GRP', 3),
    (r'\bAIG\b',                  'AMERICAN INTL GRP', 3),
    (r'AMERICAN INTERNATIONAL',   'AMERICAN INTL GRP', 3),

    # Old Republic subsidiary
    (r'BITCO INS',                'OLD REPUBLIC GRP', 3),

    # XL / AXA XL subsidiaries
    (r'INDIAN HARBOR',            'XL AMER GRP', 3),
    (r'GREENWICH INS',            'XL AMER GRP', 3),

    # Other sub-entities
    (r'EXPLORER INS',             'ICW GRP ASSETS INC GRP', 3),
    (r'MT\.? HAWLEY',              'RLI INS GRP', 3),
    (r'ENDURANCE (SPECIALTY|ASSURANCE)', 'SOMPO GRP', 3),
    (r'ATLANTIC STATES INS',      'DONEGAL GRP', 3),

    # ================= TIER 4 — SPECULATIVE (verify) =================
    # Added in bulk based on data-provider insurer_group values; regex specificity
    # and exact sub-entity membership not individually verified

    # Erie
    (r'ERIE INSURANCE',           'ERIE INS GRP', 4),
    (r'ERIE INDEMNITY',           'ERIE INS GRP', 4),

    # Cincinnati Financial
    (r'CINCINNATI INSURANCE',     'CINCINNATI FIN GRP', 4),
    (r'CINCINNATI CASUALTY',      'CINCINNATI FIN GRP', 4),
    (r'CINCINNATI SPECIALTY',     'CINCINNATI FIN GRP', 4),

    # Selective
    (r'SELECTIVE',                'SELECTIVE INS GRP', 4),

    # Federated Mutual
    (r'FEDERATED MUTUAL',         'FEDERATED MUT GRP', 4),
    (r'FEDERATED SERVICE',        'FEDERATED MUT GRP', 4),

    # FCCI
    (r'FCCI',                     'FCCI MUT INS GRP', 4),

    # EMC Insurance
    (r'EMC INSURANCE',            'EMC INS CO GRP', 4),
    (r'EMC PROPERTY',             'EMC INS CO GRP', 4),
    (r'EMCASCO',                  'EMC INS CO GRP', 4),

    # Donegal
    (r'DONEGAL',                  'DONEGAL GRP', 4),

    # ProAssurance
    (r'PROASSURANCE',             'PROASSURANCE CORP GRP', 4),

    # Pennsylvania National
    (r'PENNSYLVANIA NATIONAL',    'PENNSYLVANIA NATL INS GRP', 4),

    # Tokio Marine parent name match
    (r'TOKIO MARINE',             'TOKIO MARINE HOLDINGS INC GRP', 4),

    # QBE
    (r'\bQBE\b',                  'QBE INS GRP', 4),

    # Starr Companies
    (r'STARR INDEMNITY',          'STARR GRP', 4),
    (r'STARR SPECIALTY',          'STARR GRP', 4),
    (r'STARR SURPLUS',            'STARR GRP', 4),

    # Argo parent name match
    (r'ARGONAUT',                 'ARGONAUT GRP', 4),

    # Hanover parent name match
    (r'HANOVER INSURANCE',        'THE HANOVER INS GRP', 4),

    # Old Republic parent name match
    (r'OLD REPUBLIC',             'OLD REPUBLIC GRP', 4),

    # AmeriSure
    (r'AMERISURE',                'AMERISURE CO GRP', 4),

    # Arch
    (r'ARCH INSURANCE',           'ARCH INS GRP', 4),
    (r'ARCH SPECIALTY',           'ARCH INS GRP', 4),

    # AmeriSafe
    (r'AMERISAFE',                'AMERISAFE GRP', 4),

    # XL parent name matches
    (r'XL INSURANCE',             'XL AMER GRP', 4),
    (r'XL SPECIALTY',             'XL AMER GRP', 4),
    (r'AXA XL',                   'XL AMER GRP', 4),

    # ICW parent name match
    (r'INSURANCE COMPANY OF THE WEST', 'ICW GRP ASSETS INC GRP', 4),

    # Mercury
    (r'MERCURY (GENERAL|INSURANCE|CASUALTY|INDEMNITY)', 'MERCURY GRP', 4),

    # Independents (standalone names)
    (r'GRINNELL',                 'GRINNELL MUT GRP', 4),
    (r'PEKIN',                    'PEKIN INS GRP', 4),
    (r'HISCOX',                   'HISCOX', 4),

    # RLI parent name match
    (r'\bRLI\b',                  'RLI INS GRP', 4),

    # Sompo parent name match
    (r'SOMPO',                    'SOMPO GRP', 4),

    # AXIS
    (r'\bAXIS\b (INSURANCE|SPECIALTY|SURPLUS)', 'AXIS CAPITAL', 4),

    # Specialty / WC funds
    (r'MAINE EMPLOYERS',          'MAINE EMPLOYERS MUT INS GRP', 4),
    (r'WORKERS COMP FUND',        'WORKERS COMP FUND GRP', 4),
    (r'AMERICAN CONTRACTORS',     'AMERICAN CONTRACTORS INS GRP', 4),
    (r'MIDWEST BUILDERS',         'MIDWEST BUILDERS CAS GRP', 4),
    # NOTE: removed BUILDERS INS catch-all — matched Utah / National / Nevada
    # entities that are independent of The Builders Insurance Group of Georgia.

    # Group 1001
    (r'GROUP ONE THOUSAND',       'GROUP ONE THOUSAND ONE GRP', 4),

    # Summitpoint
    (r'SUMMITPOINT',              'SUMMITPOINT INSURANCE CO', 4),

    # Mapfre
    (r'MAPFRE',                   'MAPFRE GRP', 4),

    # Americas Insurance
    (r'AMERICAS INSURANCE',       'AMERICAS INS', 4),
]

CARRIER_RULES_C = [(re.compile(p), g, t) for p, g, t in CARRIER_RULES]
_MULTI_SPLIT    = re.compile(r'[|;]|,(?=\s*[A-Z0-9])')

def consolidate_multi(raw):
    """insurance_active_* can carry multiple carriers in one string.
    Split on separators and consolidate each. Returns deduped list."""
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    out, seen = [], set()
    for part in _MULTI_SPLIT.split(str(raw)):
        g = consolidate_one(part)  # hybrid: provider map + rules (defined in Part 1C)
        if g and g not in seen:
            seen.add(g)
            out.append(g)
    return out

# Pure-rules consolidation (used by consolidate_multi before DATA_PROVIDER_MAP exists)
def _consolidate_one_raw(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    s = str(raw).strip().upper()
    if not s or s in ('NULL', 'NONE', 'N/A'):
        return None
    for pat, grp, _tier in CARRIER_RULES_C:
        if pat.search(s):
            return grp
    return s  # raw-fallback

_log(f"Part 1B complete — {len(CARRIER_RULES)} rules compiled")

[t=1.5s] starting Part 1B
[t=1.5s] Part 1B complete — 142 rules compiled


### Part 1C — Data vintage + Hybrid consolidation + Diagnostic

Dynamic data vintage (fixes the "today's date" bug) + DATA_PROVIDER_MAP from
the populated 2.7% of insurer_group rows + runtime disagreement check.

In [6]:
_log("starting Part 1C")

# Data vintage: use the file's actual event date range, not today.
# Raw CSV has entries like 9001-01-01 (data entry errors), so clip
# before taking MIN/MAX. Past-only columns (effective_date, crash
# dates, etc.) are capped at CURRENT_DATE; a 2028 effective_date is
# as much of a data error as a 9001 one. Policy *expiration* /
# scheduled-cancellation dates can legitimately sit in the future,
# so those get a + 2 year buffer in cells 27 and 30.
_VALID_DATE_LO_SQL      = "DATE '1990-01-01'"
_VALID_DATE_HI_PAST_SQL = "CURRENT_DATE"
# Future-allowed cap is a hard "obviously-a-data-error" ceiling at year 2100;
# multi-decade surety bonds / long-term contracts (even 50 years out) are kept.
_VALID_DATE_HI_FUT_SQL  = "DATE '2100-01-01'"
_VALID_DATE_LO          = pd.Timestamp('1990-01-01')
_VALID_DATE_HI_PAST     = pd.Timestamp.today().normalize()
_VALID_DATE_HI_FUT      = pd.Timestamp('2100-01-01')

# These two queries each do a full scan of the 43 GB CSV (`raw` is a
# view, not a table). Cache the small outputs (dates + ~1-2K dict
# entries) so cold-starts don't take 10 minutes.
_VINTAGE_CACHE = _cache_pkl('vintage_and_provider_map')
if not FORCE_REBUILD and _VINTAGE_CACHE.exists():
    with open(_VINTAGE_CACHE, 'rb') as f:
        _cached = pickle.load(f)
    _DATA_VINTAGE  = _cached['vintage']
    _EARLIEST_DATE = _cached['earliest']
    _WINDOW_START  = _cached['window_start']
    DATA_PROVIDER_MAP = _cached['provider_map']
    print(f"  [cache] loaded vintage ({_DATA_VINTAGE.date()}) + "
          f"DATA_PROVIDER_MAP ({len(DATA_PROVIDER_MAP):,} entries) from pickle")
else:
    _vintage_row = con.execute(f"""
        SELECT MIN(effective_date) AS min_d,
               MAX(effective_date) AS max_d
        FROM raw
        WHERE effective_date IS NOT NULL
          AND effective_date BETWEEN {_VALID_DATE_LO_SQL} AND {_VALID_DATE_HI_PAST_SQL}
    """).fetchone()
    _DATA_VINTAGE  = pd.Timestamp(_vintage_row[1])
    _EARLIEST_DATE = pd.Timestamp(_vintage_row[0])
    _WINDOW_START  = _DATA_VINTAGE - pd.DateOffset(months=RECENCY_WINDOW_MONTHS)

    # Report any raw effective_date values that fell outside the valid window.
    _outlier_rows = con.execute(f"""
        SELECT COUNT(*) AS n,
               MIN(effective_date) AS lo,
               MAX(effective_date) AS hi
        FROM raw
        WHERE effective_date IS NOT NULL
          AND (effective_date < {_VALID_DATE_LO_SQL} OR effective_date > {_VALID_DATE_HI_PAST_SQL})
    """).fetchone()
    if _outlier_rows[0] and _outlier_rows[0] > 0:
        print(f"  WARNING: {_outlier_rows[0]:,} effective_date values outside "
              f"[1990, today] (min={_outlier_rows[1]}, max={_outlier_rows[2]}) — "
              f"excluded from vintage computation")

    print("\nBuilding data-provider insurer_group mapping...")
    _provider_df = con.execute("""
        WITH votes AS (
            SELECT UPPER(TRIM(CAST(insurer AS VARCHAR))) AS insurer_raw,
                   UPPER(TRIM(CAST(insurer_group AS VARCHAR))) AS provider_group,
                   COUNT(*) AS n
            FROM raw
            WHERE insurer IS NOT NULL
              AND insurer_group IS NOT NULL
              AND TRIM(CAST(insurer_group AS VARCHAR)) <> ''
            GROUP BY 1, 2
        ),
        ranked AS (
            SELECT insurer_raw, provider_group,
                   ROW_NUMBER() OVER (PARTITION BY insurer_raw ORDER BY n DESC) AS rn
            FROM votes
        )
        SELECT insurer_raw, provider_group FROM ranked WHERE rn = 1
    """).df()
    DATA_PROVIDER_MAP = dict(zip(_provider_df['insurer_raw'], _provider_df['provider_group']))
    print(f"  {len(DATA_PROVIDER_MAP):,} raw_insurer -> group mappings from data provider")

    with open(_VINTAGE_CACHE, 'wb') as f:
        pickle.dump({
            'vintage': _DATA_VINTAGE,
            'earliest': _EARLIEST_DATE,
            'window_start': _WINDOW_START,
            'provider_map': DATA_PROVIDER_MAP,
        }, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  cached vintage + provider map to {_VINTAGE_CACHE}")

print(f"  data event-date range (sanitized): {_EARLIEST_DATE.date()} → {_DATA_VINTAGE.date()}")
print(f"  24-month window: {_WINDOW_START.date()} → {_DATA_VINTAGE.date()}")
print(f"  snapshot column date: {SNAPSHOT_DATE.date()}")

# Temporal cutoff — anchors ALL training/aggregates to effective_date < T_SPLIT.
# This is the pipeline's honest-validation setup: every carrier aggregate, every
# training positive, every feature is derived from events strictly before this
# cutoff. The post-cutoff slice (effective_date >= T_SPLIT) is the held-out
# evaluation set that produces the honest headline metrics.
T_SPLIT = _DATA_VINTAGE - pd.DateOffset(months=TEMPORAL_SPLIT_MONTHS)

# Redefine the aggregate window relative to T_SPLIT (not vintage). recent_events
# spans [T_SPLIT - RECENCY_WINDOW_MONTHS, _DATA_VINTAGE] so we cover both the
# pre-cutoff aggregate window AND the post-cutoff holdout — a 30-month window
# relative to vintage.
_WINDOW_START = T_SPLIT - pd.DateOffset(months=RECENCY_WINDOW_MONTHS)

print(f"  T_SPLIT = {T_SPLIT.date()}  (vintage - {TEMPORAL_SPLIT_MONTHS}m)")
print(f"  Aggregate window: [{_WINDOW_START.date()}, {T_SPLIT.date()})  "
      f"(= {RECENCY_WINDOW_MONTHS}m ending at cutoff)")
print(f"  Holdout window:   [{T_SPLIT.date()}, {_DATA_VINTAGE.date()}]")

# Convert to SQL date literals for subsequent queries
_VINTAGE_SQL      = f"DATE '{_DATA_VINTAGE.date()}'"
_WINDOW_START_SQL = f"DATE '{_WINDOW_START.date()}'"
T_SPLIT_SQL       = f"DATE '{T_SPLIT.date()}'"
_SNAPSHOT_SQL     = f"DATE '{SNAPSHOT_DATE.date()}'"

# Hybrid consolidate_one: provider map first (authoritative — they have the
# most recent snapshot of carrier ownership), then rules, then raw fallback.
# If provider stored the raw name as the group, that's still an explicit
# decision from the data provider to keep that insurer standalone.
def consolidate_one(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    s = str(raw).strip().upper()
    if not s or s in ('NULL', 'NONE', 'N/A'):
        return None
    g = DATA_PROVIDER_MAP.get(s)
    if g: return g
    for pat, grp, _tier in CARRIER_RULES_C:
        if pat.search(s):
            return grp
    return s

# Rule-vs-provider disagreement diagnostic
def _apply_rules_only(raw):
    if raw is None: return (None, None)
    s = str(raw).strip().upper()
    if not s: return (None, None)
    for pat, grp, tier in CARRIER_RULES_C:
        if pat.search(s):
            return (grp, tier)
    return (None, None)

print("\nRule-vs-provider disagreement diagnostic...")
_disagreements = []
for _ins_raw, _prov_grp in DATA_PROVIDER_MAP.items():
    _rule_grp, _rule_tier = _apply_rules_only(_ins_raw)
    if _rule_grp is not None and _rule_grp != _prov_grp:
        _disagreements.append({'insurer_raw': _ins_raw, 'rule_output': _rule_grp,
                               'rule_tier': _rule_tier, 'provider_output': _prov_grp})

if _disagreements:
    _disagree_df = pd.DataFrame(_disagreements).sort_values(['rule_tier', 'rule_output'])
    _disagree_df.to_csv('rule_vs_provider_disagreements.csv', index=False)
    print(f"  Found {len(_disagree_df):,} disagreements -> rule_vs_provider_disagreements.csv")
    print(f"  Breakdown by tier: {_disagree_df['rule_tier'].value_counts().to_dict()}")
    print("\n  First 20 disagreements:")
    print(_disagree_df.head(20).to_string(index=False))
else:
    print("  No disagreements — rules agree with data provider on all overlapping insurers.")

_log("Part 1C complete")


[t=6.5s] starting Part 1C

Building data-provider insurer_group mapping...
  773 raw_insurer -> group mappings from data provider
  cached vintage + provider map to c:\Users\hi\Documents\ROSELLA ASSIGNMENT\cache\vintage_and_provider_map.pkl
  data event-date range (sanitized): 1990-01-01 → 2026-10-29
  24-month window: 2024-10-29 → 2026-10-29
  snapshot column date: 2026-05-26
  T_SPLIT = 2026-04-29  (vintage - 6m)
  Aggregate window: [2024-04-29, 2026-04-29)  (= 24m ending at cutoff)
  Holdout window:   [2026-04-29, 2026-10-29]

Rule-vs-provider disagreement diagnostic...
  Found 4 disagreements -> rule_vs_provider_disagreements.csv
  Breakdown by tier: {1: 2, 3: 1, 4: 1}

  First 20 disagreements:
                                       insurer_raw         rule_output  rule_tier                                    provider_output
                            ACUITY A MUTUAL INS CO              ACUITY          1                             ACUITY A MUTUAL INS CO
          NATIONWIDE PROP

### Part 1D — Build dot_features table (one row per DOT)

Chunked into two CREATE TEMP TABLE passes (core + feature-expansion) then
LEFT JOIN on usdot. Each pass's hash state is roughly half the single-pass
version that was OOMing. Idempotent: skips if dot_features already exists
(file-backed DuckDB preserves it across kernel restarts).

In [7]:
_log("starting Part 1D — building dot_features")

if not FORCE_REBUILD and _table_exists('dot_features'):
    _n = con.execute("SELECT COUNT(*) FROM dot_features").fetchone()[0]
    print(f"  [cache] dot_features already exists ({_n:,} rows); skipping rebuild")
    _log("Part 1D complete (cached)")
else:
    con.execute("DROP TABLE IF EXISTS dot_features")
    con.execute("DROP TABLE IF EXISTS _dot_features_a")
    con.execute("DROP TABLE IF EXISTS _dot_features_b")

    # Pass A: core aggregates (~75 columns)
    print("  pass A: core aggregates...")
    con.execute(f"""
    CREATE TEMPORARY TABLE _dot_features_a AS
    SELECT
        usdot,
        ANY_VALUE(company_name)             AS company_name,
        UPPER(TRIM(ANY_VALUE(state)))           AS state,
        UPPER(TRIM(ANY_VALUE(census_phy_state))) AS phy_state,

        -- fleet shape
        MAX(census_power_units)             AS power_units,
        MAX(census_total_drivers)           AS total_drivers,
        MAX(census_total_cdl)               AS total_cdl,
        MAX(census_truck_units)             AS truck_units,
        ANY_VALUE(census_fleetsize)         AS fleetsize_band,

        -- operation
        ANY_VALUE(census_carrier_operation) AS carrier_operation,
        ANY_VALUE(census_classdef)          AS classdef,
        ANY_VALUE(census_hm_ind)            AS hazmat_ind,
        MAX(census_mcs150_mileage)          AS mcs150_mileage,

        -- radius
        MAX(census_interstate_beyond_100_miles) AS inter_beyond_100,
        MAX(census_interstate_within_100_miles) AS inter_within_100,
        MAX(census_intrastate_beyond_100_miles) AS intra_beyond_100,
        MAX(census_intrastate_within_100_miles) AS intra_within_100,

        -- authority
        ANY_VALUE(authority_common_stat)    AS authority_common_stat,
        ANY_VALUE(authority_contract_stat)  AS authority_contract_stat,
        ANY_VALUE(authority_broker_stat)    AS authority_broker_stat,
        ANY_VALUE(authority_bipd_file)      AS authority_bipd_file,
        ANY_VALUE(authority_cargo_file)     AS authority_cargo_file,
        ANY_VALUE(authority_bond_file)      AS authority_bond_file,
        CASE WHEN ANY_VALUE(authority_common_stat)   IS NOT NULL
               OR ANY_VALUE(authority_contract_stat) IS NOT NULL
               OR ANY_VALUE(authority_broker_stat)   IS NOT NULL
             THEN 1 ELSE 0 END              AS has_authority,
        MAX(TRY_CAST(authority_min_cov_amount AS DOUBLE)) AS authority_min_cov_amount,

        -- commodity flags (18 primary)
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_genfreight, '')) <> '' THEN 1 ELSE 0 END) AS crgo_genfreight,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_bldgmat,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_bldgmat,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_construct,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_construct,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_beverages,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_beverages,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_paperprod,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_paperprod,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_produce,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_produce,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_meat,       '')) <> '' THEN 1 ELSE 0 END) AS crgo_meat,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_drybulk,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_drybulk,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_coldfood,   '')) <> '' THEN 1 ELSE 0 END) AS crgo_coldfood,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_intermodal, '')) <> '' THEN 1 ELSE 0 END) AS crgo_intermodal,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_grainfeed,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_grainfeed,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_livestock,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_livestock,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_motoveh,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_motoveh,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_household,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_household,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_passengers, '')) <> '' THEN 1 ELSE 0 END) AS crgo_passengers,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_oilfield,   '')) <> '' THEN 1 ELSE 0 END) AS crgo_oilfield,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_chem,       '')) <> '' THEN 1 ELSE 0 END) AS crgo_chem,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_logpole,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_logpole,

        -- risk record (structural zero)
        COALESCE(MAX(crash_count), 0)                 AS crash_count,
        COALESCE(MAX(crash_count_last_24_months), 0)  AS crash_count_24m,
        COALESCE(MAX(crash_fatalities), 0)            AS crash_fatalities,
        COALESCE(MAX(crash_injuries), 0)              AS crash_injuries,
        COALESCE(MAX(inspection_count), 0)            AS inspection_count,
        COALESCE(MAX(inspection_oos_violations), 0)   AS inspection_oos_viol,
        COALESCE(MAX(inspections_with_oos), 0)        AS inspections_with_oos,
        COALESCE(MAX(oos_order_count), 0)             AS oos_order_count,
        COALESCE(MAX(revocation_count), 0)            AS revocation_count,

        -- SMS core
        MAX(sms_unsafe_driv_measure) AS sms_unsafe_driv,
        MAX(sms_hos_driv_measure)    AS sms_hos_driv,
        MAX(sms_driv_fit_measure)    AS sms_driv_fit,
        MAX(sms_veh_maint_measure)   AS sms_veh_maint,
        MAX(sms_contr_subst_measure) AS sms_contr_subst,
        CASE WHEN MAX(sms_insp_total) IS NOT NULL THEN 1 ELSE 0 END AS has_sms,

        -- business profile
        ANY_VALUE(co_primary_naics)     AS primary_naics,
        MAX(co_years_in_business)       AS years_in_business,
        ANY_VALUE(co_organization_type) AS org_type,

        -- insurance history
        COALESCE(MAX(insurance_event_count), 0)         AS ins_event_count,
        COALESCE(MAX(insurance_distinct_insurers), 0)   AS ins_distinct_insurers,
        COALESCE(MAX(insurance_cancelled_count), 0)     AS ins_cancelled_count,
        COALESCE(MAX(insurance_replaced_count), 0)      AS ins_replaced_count,
        COALESCE(MAX(insurance_rejected_count), 0)      AS ins_rejected_count,
        COALESCE(MAX(insurance_years_with_coverage), 0) AS ins_years_coverage,
        MAX(insurance_first_year)                        AS ins_first_year,
        MAX(insurance_last_year)                         AS ins_last_year,

        -- current placements per line (raw; consolidation applied in pandas)
        ANY_VALUE(insurance_active_bipd_insurers_as_of_2026_05_26)  AS active_bipd_raw,
        ANY_VALUE(insurance_active_cargo_insurers_as_of_2026_05_26) AS active_cargo_raw,
        ANY_VALUE(insurance_wc_insurers)                             AS active_wc_raw,
        ANY_VALUE(co_wc_carrier)                                     AS wc_carrier_raw,

        -- coverage
        MAX(TRY_CAST(coverage_amount_usd AS DOUBLE))      AS coverage_amount_usd,
        MAX(TRY_CAST(underlying_limit_usd AS DOUBLE))     AS underlying_limit_usd,
        MAX(TRY_CAST(min_required_coverage_usd AS DOUBLE)) AS min_required_coverage_usd,

        -- PRICE PROXIES / SATISFACTION SIGNALS (label-definers; not features)
        -- co_wc_residual_market is a DOT-level company flag (same across all
        -- events), so it must fire regardless of recency; residual_market is
        -- event-level and windowed.
        MAX(CASE WHEN co_wc_residual_market = TRUE THEN 1
                 WHEN residual_market = TRUE AND effective_date >= {_WINDOW_START_SQL} THEN 1
                 ELSE 0 END) AS is_residual_market,
        MAX(CASE WHEN LOWER(COALESCE(event_status, '')) = 'active (cancellation pending)'
                       AND effective_date >= {_WINDOW_START_SQL}
                  THEN 1 ELSE 0 END) AS is_cancellation_pending,
        SUM(CASE WHEN LOWER(COALESCE(event_status, '')) = 'rejected'
                      AND effective_date >= {_WINDOW_START_SQL}
                 THEN 1 ELSE 0 END) AS rejected_filing_count,
        MAX(co_wc_rate_multiplier) AS wc_rate_multiplier,

        -- TIMING (effective/expiration of current active BIPD placement)
        MAX(CASE
              WHEN LOWER(COALESCE(event_status, '')) IN ('active', 'active (cancellation pending)')
               AND LOWER(COALESCE(policy_line, '')) IN ('auto liability', 'auto')
              THEN effective_date
            END) AS current_bipd_effective_date,
        MAX(CASE
              WHEN LOWER(COALESCE(event_status, '')) IN ('active', 'active (cancellation pending)')
               AND LOWER(COALESCE(policy_line, '')) IN ('auto liability', 'auto')
              THEN expiration_date
            END) AS current_bipd_expiration_date
    FROM raw
    WHERE usdot IS NOT NULL
      AND effective_date < {T_SPLIT_SQL}
    GROUP BY usdot
    """)
    _log_mem("after pass A")

    # Pass B: feature-expansion aggregates (~85 columns)
    print("  pass B: feature-expansion aggregates...")
    con.execute(f"""
    CREATE TEMPORARY TABLE _dot_features_b AS
    SELECT
        usdot,
        -- business/industry
        ANY_VALUE(co_lifecycle_status)      AS co_lifecycle_status,

        -- census characteristics
        ANY_VALUE(census_carship)           AS carship,
        ANY_VALUE(census_status_code)       AS census_status_code,
        ANY_VALUE(census_safety_rating)     AS safety_rating,
        MAX(census_recordable_crash_rate)   AS crash_rate_per_mil,
        MAX(census_total_intrastate_drivers) AS total_intrastate_drivers,
        MAX(census_driver_inter_total)      AS driver_inter_total,
        MAX(census_bus_units)               AS bus_units,
        MAX(CASE WHEN UPPER(TRIM(census_prior_revoke_flag)) = 'Y' THEN 1 ELSE 0 END) AS prior_revoke_flag,
        MAX(CASE WHEN census_mcsipstep IS NOT NULL THEN 1 ELSE 0 END) AS has_mcsip,

        -- remaining commodity flags (12)
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_cargoothr, '')) <> '' THEN 1 ELSE 0 END) AS crgo_cargoothr,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_machlrg, ''))   <> '' THEN 1 ELSE 0 END) AS crgo_machlrg,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_mobilehome, '')) <> '' THEN 1 ELSE 0 END) AS crgo_mobilehome,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_coalcoke, ''))  <> '' THEN 1 ELSE 0 END) AS crgo_coalcoke,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_garbage, ''))   <> '' THEN 1 ELSE 0 END) AS crgo_garbage,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_metalsheet, '')) <> '' THEN 1 ELSE 0 END) AS crgo_metalsheet,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_drivetow, ''))  <> '' THEN 1 ELSE 0 END) AS crgo_drivetow,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_waterwell, '')) <> '' THEN 1 ELSE 0 END) AS crgo_waterwell,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_usmail, ''))    <> '' THEN 1 ELSE 0 END) AS crgo_usmail,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_utility, ''))   <> '' THEN 1 ELSE 0 END) AS crgo_utility,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_farmsupp, ''))  <> '' THEN 1 ELSE 0 END) AS crgo_farmsupp,
        MAX(CASE WHEN TRIM(COALESCE(census_crgo_liqgas, ''))    <> '' THEN 1 ELSE 0 END) AS crgo_liqgas,

        -- risk detail
        COALESCE(MAX(crash_tow_away_count), 0)       AS crash_tow_away_count,
        COALESCE(MAX(inspection_violations), 0)      AS inspection_violations,
        COALESCE(MAX(inspection_driver_oos), 0)      AS inspection_driver_oos,
        COALESCE(MAX(inspection_vehicle_oos), 0)     AS inspection_vehicle_oos,
        COALESCE(MAX(inspection_hazmat_oos), 0)      AS inspection_hazmat_oos,
        MAX(crash_latest_date)                       AS crash_latest_date,
        MAX(inspection_latest_date)                  AS inspection_latest_date,
        MAX(oos_latest_date)                         AS oos_latest_date,
        MAX(revocation_latest_date)                  AS revocation_latest_date,

        -- SMS detail (totals + violations + alert status per BASIC)
        COALESCE(MAX(sms_insp_total), 0)             AS sms_insp_total,
        COALESCE(MAX(sms_driver_insp_total), 0)      AS sms_driver_insp_total,
        COALESCE(MAX(sms_vehicle_insp_total), 0)     AS sms_vehicle_insp_total,
        COALESCE(MAX(sms_driver_oos_insp_total), 0)  AS sms_driver_oos_insp_total,
        COALESCE(MAX(sms_vehicle_oos_insp_total), 0) AS sms_vehicle_oos_insp_total,
        COALESCE(MAX(sms_violation_count), 0)        AS sms_violation_count,
        COALESCE(MAX(sms_violation_oos_count), 0)    AS sms_violation_oos_count,
        COALESCE(MAX(sms_violation_severity_weight), 0) AS sms_violation_severity,
        COALESCE(MAX(sms_unsafe_driv_insp_w_viol), 0) AS sms_unsafe_driv_viol,
        COALESCE(MAX(sms_hos_driv_insp_w_viol), 0)    AS sms_hos_driv_viol,
        COALESCE(MAX(sms_driv_fit_insp_w_viol), 0)    AS sms_driv_fit_viol,
        COALESCE(MAX(sms_contr_subst_insp_w_viol), 0) AS sms_contr_subst_viol,
        COALESCE(MAX(sms_veh_maint_insp_w_viol), 0)   AS sms_veh_maint_viol,
        ANY_VALUE(sms_unsafe_driv_ac)                AS sms_unsafe_driv_ac,
        ANY_VALUE(sms_hos_driv_ac)                   AS sms_hos_driv_ac,
        ANY_VALUE(sms_driv_fit_ac)                   AS sms_driv_fit_ac,
        ANY_VALUE(sms_contr_subst_ac)                AS sms_contr_subst_ac,
        ANY_VALUE(sms_veh_maint_ac)                  AS sms_veh_maint_ac,

        -- insurance recency
        MAX(insurance_first_event_date)              AS insurance_first_event_date,
        MAX(insurance_latest_event_date)             AS insurance_latest_event_date,

        -- authority detail
        ANY_VALUE(authority_bus_state_code)          AS authority_bus_state_code,
        COALESCE(MAX(authority_history_count), 0)    AS authority_history_count,
        MAX(authority_first_served_date)             AS authority_first_served_date,
        ANY_VALUE(authority_undeliverable_mail)      AS authority_undeliverable_mail,
        ANY_VALUE(authority_property_chk)            AS authority_property_chk,
        ANY_VALUE(authority_passenger_chk)           AS authority_passenger_chk,
        ANY_VALUE(authority_hhg_chk)                 AS authority_hhg_chk,
        ANY_VALUE(authority_private_auth_chk)        AS authority_private_auth_chk,
        ANY_VALUE(authority_enterprise_chk)          AS authority_enterprise_chk,
        ANY_VALUE(authority_cargo_req)               AS authority_cargo_req,
        ANY_VALUE(authority_bond_req)                AS authority_bond_req,
        ANY_VALUE(authority_common_app_pend)         AS authority_common_app_pend,
        ANY_VALUE(authority_contract_app_pend)       AS authority_contract_app_pend,
        ANY_VALUE(authority_broker_app_pend)         AS authority_broker_app_pend,
        ANY_VALUE(authority_common_rev_pend)         AS authority_common_rev_pend,
        ANY_VALUE(authority_contract_rev_pend)       AS authority_contract_rev_pend,
        ANY_VALUE(authority_broker_rev_pend)         AS authority_broker_rev_pend,

        -- company sophistication
        MAX(CASE WHEN is_peo = TRUE THEN 1 ELSE 0 END) AS is_peo_flag,
        MAX(co_usdot_count)                          AS co_usdot_count,
        MAX(co_phone_count)                          AS co_phone_count,
        MAX(co_officer_count)                        AS co_officer_count,
        MAX(co_source_count)                         AS co_source_count,
        MAX(co_resolved_company_count)               AS co_resolved_company_count,
        MAX(CASE WHEN co_has_retirement_plan = TRUE THEN 1 ELSE 0 END) AS co_has_retirement_plan,
        MAX(CASE WHEN co_has_health_plan     = TRUE THEN 1 ELSE 0 END) AS co_has_health_plan,
        MAX(CASE WHEN co_primary_domain IS NOT NULL THEN 1 ELSE 0 END) AS has_domain,
        ANY_VALUE(co_match_confidence)               AS co_match_confidence,

        -- fleet date for recency computation
        ANY_VALUE(census_mcs150_date)                AS census_mcs150_date,

        MAX(has_insurance_event) AS has_insurance_event
    FROM raw
    WHERE usdot IS NOT NULL
      AND effective_date < {T_SPLIT_SQL}
    GROUP BY usdot
    """)
    _log_mem("after pass B")

    # Join passes into final dot_features
    print("  joining passes A and B into dot_features...")
    con.execute("""
    CREATE TABLE dot_features AS
    SELECT a.*, b.* EXCLUDE (usdot)
    FROM _dot_features_a a
    LEFT JOIN _dot_features_b b USING (usdot)
    """)
    con.execute("DROP TABLE _dot_features_a")
    con.execute("DROP TABLE _dot_features_b")
    _log_mem("after join")

    n_dots = con.execute("SELECT COUNT(*) FROM dot_features").fetchone()[0]
    n_scorable = con.execute("""
        SELECT COUNT(*) FROM dot_features
        WHERE active_bipd_raw IS NOT NULL
           OR active_cargo_raw IS NOT NULL
           OR active_wc_raw IS NOT NULL
           OR wc_carrier_raw IS NOT NULL
    """).fetchone()[0]
    print(f"  dot_features: {n_dots:,} DOTs ({n_scorable:,} scorable with any active placement)")
    _log("Part 1D complete")


[t=404.3s] starting Part 1D — building dot_features
  pass A: core aggregates...
  [mem] after pass A: 1.43 GB
  pass B: feature-expansion aggregates...
  [mem] after pass B: 2.46 GB
  joining passes A and B into dot_features...
  [mem] after join: 0.44 GB
  dot_features: 1,490,198 DOTs (458,128 scorable with any active placement)
[t=731.7s] Part 1D complete


## Phase 2 — Aggregates, Centroids & Scorable DOTs

Carrier consolidation lookup, 24-month event window, (carrier, state, line) aggregates + leave-one-out contributions, sub-entity centroids for Option B distance, and loading the scorable DOT universe into pandas with sanitised derived features.

### Part 2A — Carrier lookup + consolidation report CSV

In [8]:
_log("starting Part 2A")

# NOTE: CSV is written to the project root (not CACHE_DIR). The earlier
# version of this check looked in CACHE_DIR, which always missed → forced
# a 3-min full rebuild every session. Fixed to check only the DuckDB table
# — the CSV is a diagnostic artifact that persists in the repo.
if not FORCE_REBUILD and _table_exists('carrier_lookup'):
    _n = con.execute("SELECT COUNT(*) FROM carrier_lookup").fetchone()[0]
    print(f"  [cache] carrier_lookup already exists ({_n:,} rows); skipping rebuild")
    _log("Part 2A complete (cached)")
else:
    print("Building carrier consolidation lookup (over LINES_IN_SCOPE)...")
    distinct_insurers = con.execute(f"""
        SELECT DISTINCT UPPER(TRIM(CAST(insurer AS VARCHAR))) AS insurer_raw
        FROM raw
        WHERE insurer IS NOT NULL
          AND effective_date >= {_WINDOW_START_SQL}
          AND effective_date <= {_VINTAGE_SQL}
          AND LOWER(TRIM(CAST(policy_line AS VARCHAR))) IN
              ({', '.join("'" + L + "'" for L in LINES_IN_SCOPE)})
    """).df()
    distinct_insurers['carrier_group'] = distinct_insurers['insurer_raw'].map(consolidate_one)
    distinct_insurers = distinct_insurers.dropna(subset=['carrier_group'])
    print(f"  {len(distinct_insurers):,} distinct insurer strings → {distinct_insurers['carrier_group'].nunique():,} consolidated groups")

    con.register('carrier_lookup_df', distinct_insurers)
    con.execute("DROP TABLE IF EXISTS carrier_lookup")
    con.execute("CREATE TABLE carrier_lookup AS SELECT * FROM carrier_lookup_df")
    con.unregister('carrier_lookup_df')

    # Consolidation report with per-row mapping source + tier
    print("\nWriting carrier_consolidation_report.csv...")
    consolidation_report = con.execute(f"""
    WITH event_counts AS (
        SELECT UPPER(TRIM(CAST(r.insurer AS VARCHAR))) AS insurer_raw,
               COUNT(*) AS n_events_24m
        FROM raw r
        WHERE r.insurer IS NOT NULL
          AND r.effective_date >= {_WINDOW_START_SQL}
          AND r.effective_date <= {_VINTAGE_SQL}
          AND LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) IN
              ({', '.join("'" + L + "'" for L in LINES_IN_SCOPE)})
        GROUP BY 1
    )
    SELECT cl.carrier_group, cl.insurer_raw,
           COALESCE(ec.n_events_24m, 0) AS n_events_24m
    FROM carrier_lookup cl
    LEFT JOIN event_counts ec USING (insurer_raw)
    ORDER BY cl.carrier_group, n_events_24m DESC, cl.insurer_raw
    """).df()

    def _mapping_source(ins_raw, grp):
        if DATA_PROVIDER_MAP.get(ins_raw) == grp:
            return ('provider', None)
        rule_grp, rule_tier = _apply_rules_only(ins_raw)
        if rule_grp == grp:
            return (f'rule_T{rule_tier}', rule_tier)
        if grp == ins_raw:
            return ('raw_fallback', None)
        return ('other', None)

    _sources = consolidation_report.apply(
        lambda r: _mapping_source(r['insurer_raw'], r['carrier_group']), axis=1
    )
    consolidation_report['mapping_source'] = _sources.str[0]
    consolidation_report['rule_tier']      = _sources.str[1]

    grp_total = consolidation_report.groupby('carrier_group')['n_events_24m'].transform('sum')
    consolidation_report['group_total_events']  = grp_total
    consolidation_report['pct_of_group'] = (
        100.0 * consolidation_report['n_events_24m'] / consolidation_report['group_total_events'].replace(0, np.nan)
    ).round(2)
    consolidation_report['n_entities_in_group'] = consolidation_report.groupby('carrier_group')['insurer_raw'].transform('nunique')

    consolidation_report.to_csv('carrier_consolidation_report.csv', index=False)
    print(f"  Wrote carrier_consolidation_report.csv ({len(consolidation_report):,} rows, "
          f"{consolidation_report['carrier_group'].nunique():,} groups)")

    grp_summary = (consolidation_report.drop_duplicates('carrier_group')
                   [['carrier_group','n_entities_in_group','group_total_events']]
                   .sort_values('group_total_events', ascending=False).head(20))
    print("\n  Top 20 groups by event volume:")
    print(grp_summary.to_string(index=False))

    _log("Part 2A complete")


[t=731.7s] starting Part 2A
Building carrier consolidation lookup (over LINES_IN_SCOPE)...
  1,935 distinct insurer strings → 1,225 consolidated groups

Writing carrier_consolidation_report.csv...
  Wrote carrier_consolidation_report.csv (1,935 rows, 1,225 groups)

  Top 20 groups by event volume:
                            carrier_group  n_entities_in_group  group_total_events
                          PROGRESSIVE GRP                   27              220368
                   BERKSHIRE HATHAWAY GRP                   47              120275
                            TRAVELERS GRP                   48               41293
         RANDALL & QUILTER INVESTMENT GRP                    4               37888
                                CANAL GRP                    2               36739
                      TEXAS MUTUAL INS CO                    3               35464
                               GREAT WEST                    2               30669
                           ZURICH INS

### Part 2B — recent_events (24mo window anchored at data vintage, state normalised)

In [9]:
_log("starting Part 2B - recent_events")

if not FORCE_REBUILD and _table_exists('recent_events'):
    _n = con.execute("SELECT COUNT(*) FROM recent_events").fetchone()[0]
    print(f"  [cache] recent_events already exists ({_n:,} rows); skipping rebuild")
    _log("Part 2B complete (cached)")
else:
    con.execute("DROP TABLE IF EXISTS recent_events")
    con.execute(f"""
    CREATE TABLE recent_events AS
    SELECT r.usdot,
           cl.insurer_raw,
           cl.carrier_group,
           CASE
             WHEN LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) IN ('auto liability','auto') THEN 'bipd'
             WHEN LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) = 'broker surety bond'       THEN 'surety_bond'
             ELSE LOWER(TRIM(CAST(r.policy_line AS VARCHAR)))
           END AS line,
           LOWER(TRIM(CAST(r.event_status AS VARCHAR))) AS event_status,
           LOWER(TRIM(CAST(r.end_reason   AS VARCHAR))) AS end_reason,
           r.effective_date,
           -- Validate cancellation_or_end_date: null only year-9xxx entry errors.
           -- The year-2100 ceiling is a sanity cap; multi-decade surety bonds and
           -- other long-term contracts are kept.
           CASE WHEN r.cancellation_or_end_date BETWEEN DATE '1990-01-01'
                                                    AND DATE '2100-01-01'
                THEN r.cancellation_or_end_date END AS cancellation_or_end_date,
           DATE_DIFF('day', r.effective_date,
               CASE WHEN r.cancellation_or_end_date BETWEEN DATE '1990-01-01'
                                                        AND DATE '2100-01-01'
                    THEN r.cancellation_or_end_date END) AS days_in_force,
           UPPER(TRIM(COALESCE(r.census_phy_state, r.state))) AS state,
           CASE WHEN r.residual_market = TRUE OR r.co_wc_residual_market = TRUE
                THEN 1 ELSE 0 END AS is_residual_market_event
    FROM raw r
    JOIN carrier_lookup cl
      ON UPPER(TRIM(CAST(r.insurer AS VARCHAR))) = cl.insurer_raw
    WHERE r.insurer IS NOT NULL
      AND r.effective_date >= {_WINDOW_START_SQL}
      AND r.effective_date <= {_VINTAGE_SQL}
      AND LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) IN
          ({', '.join("'" + L + "'" for L in LINES_IN_SCOPE)})
    """)

    n_recent = con.execute("SELECT COUNT(*) FROM recent_events").fetchone()[0]
    per_line = con.execute("SELECT line, COUNT(*) AS n FROM recent_events GROUP BY line ORDER BY n DESC").df()
    print(f"  {n_recent:,} recent events in [{_WINDOW_START.date()}, {_DATA_VINTAGE.date()}]")
    print(per_line.to_string(index=False))

    _log_mem("after recent_events")
    _log("Part 2B complete")


[t=1006.8s] starting Part 2B - recent_events
  1,184,740 recent events in [2024-04-29, 2026-10-29]
       line      n
       bipd 922146
         wc 227582
surety_bond  25169
      cargo   9843
  [mem] after recent_events: 0.47 GB
[t=1149.7s] Part 2B complete


### Part 2C — Service composite + (carrier, state, line) aggregates + momentum

Adds carr_placements_6m, carr_placements_6to18m, carr_state_momentum
to capture carrier state entry/exit trajectory

In [10]:
_log("starting Part 2C")

# Schema-aware cache check: if the existing tables pre-date the 6m
# short-window columns, force a rebuild so downstream feature code works.
def _has_6m_columns(table, col='carr_mean_power_6m'):
    try:
        q = f"SELECT {col} FROM {table} LIMIT 1"
        con.execute(q).fetchone()
        return True
    except Exception:
        return False

_needs_6m_rebuild = (_table_exists('carrier_state_agg')
                     and not _has_6m_columns('carrier_state_agg', 'carr_mean_power_6m'))
if _needs_6m_rebuild:
    print("  [schema] carrier_state_agg predates 6m short-window columns — forcing rebuild")
    con.execute("DROP TABLE IF EXISTS service_composite")
    con.execute("DROP TABLE IF EXISTS carrier_state_agg")
    con.execute("DROP TABLE IF EXISTS dot_cell_contrib")

# Service composite
if not FORCE_REBUILD and _table_exists('service_composite') and _has_6m_columns('service_composite', 'svc_rejection_rate_6m'):
    print("  [cache] service_composite already exists (with 6m cols); skipping rebuild")
else:
    print("Computing service composite per carrier...")
    con.execute("DROP TABLE IF EXISTS service_composite")
    # Short-window variants anchored at T_SPLIT (not vintage) — captures recent-
    # era carrier appetite drift as of the training cutoff, not post-cutoff state.
    _sw_cutoff_sql = f"CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '{SHORT_WINDOW_MONTHS} months'"
    con.execute(f"""
    CREATE TABLE service_composite AS
    WITH stats AS (
        SELECT carrier_group,
               COUNT(*) AS svc_n_events,
               AVG(CASE WHEN days_in_force < 90 AND event_status = 'cancelled' THEN 1.0 ELSE 0.0 END)
                 AS svc_short_tenure_cancel_rate,
               AVG(CASE WHEN event_status = 'rejected' THEN 1.0 ELSE 0.0 END) AS svc_rejection_rate,
               AVG(CASE WHEN days_in_force >= 0 THEN days_in_force END) AS svc_mean_tenure_days,
               -- 6m-window variants (short-window features)
               AVG(CASE WHEN days_in_force < 90 AND event_status = 'cancelled' THEN 1.0 ELSE 0.0 END)
                   FILTER (WHERE effective_date >= {_sw_cutoff_sql})
                 AS svc_short_tenure_cancel_rate_6m,
               AVG(CASE WHEN event_status = 'rejected' THEN 1.0 ELSE 0.0 END)
                   FILTER (WHERE effective_date >= {_sw_cutoff_sql})
                 AS svc_rejection_rate_6m
        FROM recent_events
        WHERE effective_date < {T_SPLIT_SQL}
        GROUP BY carrier_group
    ),
    vol_exit AS (
        SELECT carrier_group,
               AVG(CASE WHEN event_status = 'replaced' THEN 1.0 ELSE 0.0 END) AS svc_voluntary_exit_ratio
        FROM recent_events
        WHERE event_status IN ('cancelled','replaced')
          AND effective_date < {T_SPLIT_SQL}
        GROUP BY carrier_group
    )
    SELECT s.*, COALESCE(v.svc_voluntary_exit_ratio, 0.5) AS svc_voluntary_exit_ratio
    FROM stats s LEFT JOIN vol_exit v USING (carrier_group)
    """)

min_thresh_sql = ' '.join(f"WHEN line = '{line}' THEN {thresh}" for line, thresh in MIN_CARRIER_PLACEMENTS.items())

if not FORCE_REBUILD and _table_exists('carrier_state_agg'):
    _n = con.execute("SELECT COUNT(*) FROM carrier_state_agg").fetchone()[0]
    print(f"  [cache] carrier_state_agg already exists ({_n:,} cells); skipping rebuild")
else:
    print("\nComputing (carrier, state, line) aggregates with momentum features...")
    con.execute("DROP TABLE IF EXISTS carrier_state_agg")
    con.execute(f"""
    CREATE TABLE carrier_state_agg AS
    WITH enriched AS (
        SELECT re.carrier_group, re.state, re.line, re.usdot, re.effective_date,
               re.is_residual_market_event,
               df.power_units, df.crash_count, df.inspection_oos_viol,
               df.revocation_count, df.oos_order_count,
               df.crgo_genfreight, df.crgo_bldgmat, df.crgo_construct,
               df.crgo_livestock, df.crgo_oilfield, df.crgo_passengers,
               df.crgo_chem, df.crgo_meat, df.crgo_intermodal, df.crgo_motoveh,
               CASE WHEN UPPER(TRIM(df.hazmat_ind)) IN ('Y','X') THEN 1.0 ELSE 0.0 END AS hazmat_bin,
               df.inter_beyond_100, df.intra_within_100
        FROM recent_events re
        JOIN dot_features df USING (usdot)
        WHERE re.effective_date < {T_SPLIT_SQL}
    ),
    agg AS (
        SELECT carrier_group, state, line,
               COUNT(DISTINCT usdot) AS carr_n_placements,
               COUNT(*) AS carr_n_events,
               -- Means (overwritten by LOO at train/score time; see apply_loo_adjustment)
               AVG(power_units) AS carr_mean_power,
               MEDIAN(power_units) AS carr_median_power,
               QUANTILE_CONT(power_units, 0.9) AS carr_p90_power,
               AVG(crash_count) AS carr_mean_crash,
               AVG(inspection_oos_viol) AS carr_mean_insp_oos,
               AVG(revocation_count) AS carr_mean_revoc,
               AVG(oos_order_count) AS carr_mean_oos,
               AVG(hazmat_bin) AS carr_pct_hazmat,
               AVG(crgo_genfreight) AS carr_pct_genfreight,
               AVG(crgo_bldgmat) AS carr_pct_bldgmat,
               AVG(crgo_construct) AS carr_pct_construct,
               AVG(crgo_livestock) AS carr_pct_livestock,
               AVG(crgo_oilfield) AS carr_pct_oilfield,
               AVG(crgo_passengers) AS carr_pct_passengers,
               AVG(crgo_chem) AS carr_pct_chem,
               AVG(crgo_meat) AS carr_pct_meat,
               AVG(crgo_intermodal) AS carr_pct_intermodal,
               AVG(crgo_motoveh) AS carr_pct_motoveh,
               AVG(inter_beyond_100) AS carr_mean_inter_long,
               AVG(intra_within_100) AS carr_mean_intra_short,
               -- Share of residual-market events (for filtering recommendations)
               AVG(CAST(is_residual_market_event AS DOUBLE)) AS carr_pct_residual_market,
               -- 6m-window variants of core features (short-window drift signal).
               -- Single scan via FILTER aggregates — same memory profile as the 24m version.
               AVG(power_units)         FILTER (WHERE effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '{SHORT_WINDOW_MONTHS} months') AS carr_mean_power_6m,
               AVG(crash_count)         FILTER (WHERE effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '{SHORT_WINDOW_MONTHS} months') AS carr_mean_crash_6m,
               AVG(inspection_oos_viol) FILTER (WHERE effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '{SHORT_WINDOW_MONTHS} months') AS carr_mean_insp_oos_6m,
               AVG(oos_order_count)     FILTER (WHERE effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '{SHORT_WINDOW_MONTHS} months') AS carr_mean_oos_6m,
               AVG(hazmat_bin)          FILTER (WHERE effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '{SHORT_WINDOW_MONTHS} months') AS carr_pct_hazmat_6m,
               AVG(CAST(is_residual_market_event AS DOUBLE))
                                        FILTER (WHERE effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '{SHORT_WINDOW_MONTHS} months') AS carr_pct_residual_market_6m,
               -- Sums (for LOO adjustment at train/score time)
               SUM(power_units)        AS carr_power_sum,
               SUM(crash_count)        AS carr_crash_sum,
               SUM(inspection_oos_viol) AS carr_insp_oos_sum,
               SUM(revocation_count)   AS carr_revoc_sum,
               SUM(oos_order_count)    AS carr_oos_sum,
               SUM(hazmat_bin)         AS carr_hazmat_sum,
               SUM(crgo_genfreight)    AS carr_genfreight_sum,
               SUM(crgo_bldgmat)       AS carr_bldgmat_sum,
               SUM(crgo_construct)     AS carr_construct_sum,
               SUM(crgo_livestock)     AS carr_livestock_sum,
               SUM(crgo_oilfield)      AS carr_oilfield_sum,
               SUM(crgo_passengers)    AS carr_passengers_sum,
               SUM(crgo_chem)          AS carr_chem_sum,
               SUM(crgo_meat)          AS carr_meat_sum,
               SUM(crgo_intermodal)    AS carr_intermodal_sum,
               SUM(crgo_motoveh)       AS carr_motoveh_sum,
               SUM(inter_beyond_100)   AS carr_inter_long_sum,
               SUM(intra_within_100)   AS carr_intra_short_sum,
               COUNT(DISTINCT CASE WHEN effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '6 months'
                                   THEN usdot END) AS carr_placements_6m,
               COUNT(DISTINCT CASE WHEN effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '18 months'
                                    AND effective_date <  CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '6 months'
                                   THEN usdot END) AS carr_placements_6to18m
        FROM enriched
        GROUP BY carrier_group, state, line
    )
    SELECT a.*,
           sc.svc_short_tenure_cancel_rate, sc.svc_rejection_rate,
           sc.svc_voluntary_exit_ratio, sc.svc_mean_tenure_days,
           CAST(a.carr_placements_6m AS DOUBLE) / NULLIF(a.carr_placements_6to18m / 2.0 + 1, 0)
             AS carr_state_momentum
    FROM agg a
    LEFT JOIN service_composite sc USING (carrier_group)
    WHERE a.carr_n_placements >= CASE {min_thresh_sql} ELSE 20 END
    """)

n_cells = con.execute("SELECT COUNT(*) FROM carrier_state_agg").fetchone()[0]
print(f"  {n_cells:,} (carrier, state, line) cells after per-line min-N")

# Load into pandas (always — cheap, needed by downstream cells)
carrier_state_agg = con.execute("SELECT * FROM carrier_state_agg").df()
for c in carrier_state_agg.columns:
    if pd.api.types.is_integer_dtype(carrier_state_agg[c]):
        carrier_state_agg[c] = pd.to_numeric(carrier_state_agg[c], downcast='integer')
    elif pd.api.types.is_float_dtype(carrier_state_agg[c]):
        carrier_state_agg[c] = pd.to_numeric(carrier_state_agg[c], downcast='float')
print(f"  cells per line: {carrier_state_agg.groupby('line').size().to_dict()}")

# dot_cell_contrib — per (usdot, carrier, state, line), the DOT's event
# count in that cell. Used for leave-one-out adjustment of carrier aggregates.
if not FORCE_REBUILD and _table_exists('dot_cell_contrib'):
    _n = con.execute("SELECT COUNT(*) FROM dot_cell_contrib").fetchone()[0]
    print(f"  [cache] dot_cell_contrib already exists ({_n:,} rows); skipping rebuild")
else:
    con.execute(f"""
    CREATE OR REPLACE TABLE dot_cell_contrib AS
    SELECT re.usdot, re.carrier_group, re.state, re.line,
           COUNT(*) AS dot_n_events_in_cell,
           MAX(CASE WHEN re.effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '6 months'
                    THEN 1 ELSE 0 END) AS dot_in_6m,
           MAX(CASE WHEN re.effective_date >= CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '18 months'
                     AND re.effective_date <  CAST({T_SPLIT_SQL} AS DATE) - INTERVAL '6 months'
                    THEN 1 ELSE 0 END) AS dot_in_6to18m
    FROM recent_events re
    WHERE re.effective_date < {T_SPLIT_SQL}
    GROUP BY re.usdot, re.carrier_group, re.state, re.line
    """)

dot_cell_contrib = con.execute("SELECT * FROM dot_cell_contrib").df()
# Keep usdot as int64 for consistent merge dtype across all tables
dot_cell_contrib['usdot'] = dot_cell_contrib['usdot'].astype('int64')
for c in ['dot_n_events_in_cell','dot_in_6m','dot_in_6to18m']:
    dot_cell_contrib[c] = pd.to_numeric(dot_cell_contrib[c], downcast='integer')
print(f"  dot_cell_contrib: {len(dot_cell_contrib):,} DOT-cell pairs (for LOO)")

_log("Part 2C complete")


[t=1149.8s] starting Part 2C
Computing service composite per carrier...

Computing (carrier, state, line) aggregates with momentum features...
  2,554 (carrier, state, line) cells after per-line min-N
  cells per line: {'bipd': 1554, 'cargo': 78, 'surety_bond': 274, 'wc': 648}
  dot_cell_contrib: 603,351 DOT-cell pairs (for LOO)
[t=1151.8s] Part 2C complete


### Part 2D — Sub-entity centroids + Option B feature weights

In [11]:
_log("starting Part 2D")

if not FORCE_REBUILD and _table_exists('sub_entity_centroids'):
    _n = con.execute("SELECT COUNT(*) FROM sub_entity_centroids").fetchone()[0]
    print(f"  [cache] sub_entity_centroids already exists ({_n:,} rows); skipping rebuild")
else:
    con.execute("DROP TABLE IF EXISTS sub_entity_centroids")
    con.execute(f"""
    CREATE TABLE sub_entity_centroids AS
    WITH enriched AS (
        SELECT re.carrier_group, re.state, re.line, re.insurer_raw, re.usdot,
               LN(1.0 + COALESCE(df.power_units, 0)) AS power_units_log,
               COALESCE(df.crash_count, 0)           AS crash_count,
               COALESCE(df.inspection_oos_viol, 0)   AS inspection_oos_viol,
               COALESCE(df.crgo_genfreight, 0)       AS crgo_genfreight,
               COALESCE(df.crgo_bldgmat, 0)          AS crgo_bldgmat,
               COALESCE(df.crgo_livestock, 0)        AS crgo_livestock,
               COALESCE(df.crgo_oilfield, 0)         AS crgo_oilfield,
               COALESCE(df.crgo_intermodal, 0)       AS crgo_intermodal,
               COALESCE(df.crgo_chem, 0)             AS crgo_chem,
               COALESCE(df.crgo_passengers, 0)       AS crgo_passengers
        FROM recent_events re
        JOIN dot_features df USING (usdot)
        WHERE re.effective_date < {T_SPLIT_SQL}
    )
    SELECT carrier_group, state, line, insurer_raw,
           COUNT(DISTINCT usdot) AS sub_n_placements,
           COUNT(*)              AS sub_n_events,
           -- Means (LOO-adjusted at distance time; see best_subentity_for_dot)
           AVG(power_units_log)      AS sub_power_units_log,
           AVG(crash_count)          AS sub_crash_count,
           AVG(inspection_oos_viol)  AS sub_inspection_oos_viol,
           AVG(crgo_genfreight)      AS sub_crgo_genfreight,
           AVG(crgo_bldgmat)         AS sub_crgo_bldgmat,
           AVG(crgo_livestock)       AS sub_crgo_livestock,
           AVG(crgo_oilfield)        AS sub_crgo_oilfield,
           AVG(crgo_intermodal)      AS sub_crgo_intermodal,
           AVG(crgo_chem)            AS sub_crgo_chem,
           AVG(crgo_passengers)      AS sub_crgo_passengers,
           -- Sums (for LOO adjustment in Option B)
           SUM(power_units_log)      AS sub_power_units_log_sum,
           SUM(crash_count)          AS sub_crash_count_sum,
           SUM(inspection_oos_viol)  AS sub_inspection_oos_viol_sum,
           SUM(crgo_genfreight)      AS sub_crgo_genfreight_sum,
           SUM(crgo_bldgmat)         AS sub_crgo_bldgmat_sum,
           SUM(crgo_livestock)       AS sub_crgo_livestock_sum,
           SUM(crgo_oilfield)        AS sub_crgo_oilfield_sum,
           SUM(crgo_intermodal)      AS sub_crgo_intermodal_sum,
           SUM(crgo_chem)            AS sub_crgo_chem_sum,
           SUM(crgo_passengers)      AS sub_crgo_passengers_sum
    FROM enriched
    GROUP BY carrier_group, state, line, insurer_raw
    HAVING COUNT(DISTINCT usdot) >= {MIN_SUB_ENTITY_PLACEMENTS}
    """)

sub_entity_centroids = con.execute("SELECT * FROM sub_entity_centroids").df()
for c in sub_entity_centroids.columns:
    if pd.api.types.is_float_dtype(sub_entity_centroids[c]):
        sub_entity_centroids[c] = sub_entity_centroids[c].astype('float32')
print(f"  {len(sub_entity_centroids):,} sub-entity centroids")

# Within-cell variance -> feature weights for Option B distance
print("\nComputing within-cell feature variances for distance weighting...")
dist_feat_cols = [f'sub_{f}' for f in APPETITE_DIST_FEATURES]
cell_sizes = sub_entity_centroids.groupby(['carrier_group','state','line']).size()
multi_cells = cell_sizes[cell_sizes >= 2].index
if len(multi_cells) > 0:
    v = (sub_entity_centroids
         .merge(pd.DataFrame(multi_cells.tolist(), columns=['carrier_group','state','line']),
                on=['carrier_group','state','line'], how='inner')
         .groupby(['carrier_group','state','line'])[dist_feat_cols]
         .var().fillna(0))
    feat_var = v.mean(axis=0)
    FEATURE_WEIGHTS = (feat_var / feat_var.sum()).to_dict() if feat_var.sum() > 0 else {}
else:
    FEATURE_WEIGHTS = {c: 1.0 / len(dist_feat_cols) for c in dist_feat_cols}

print("  Appetite-distance feature weights (sum to 1):")
for k, v in sorted(FEATURE_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"    {k:<30s}  {v:.4f}")

# dot_subent_contrib — per (usdot, carrier, state, line, insurer_raw),
# the DOT's event count in that sub-entity. Used for LOO in Option B distance.
if not FORCE_REBUILD and _table_exists('dot_subent_contrib'):
    _n = con.execute("SELECT COUNT(*) FROM dot_subent_contrib").fetchone()[0]
    print(f"  [cache] dot_subent_contrib already exists ({_n:,} rows); skipping rebuild")
else:
    con.execute(f"""
    CREATE OR REPLACE TABLE dot_subent_contrib AS
    SELECT re.usdot, re.carrier_group, re.state, re.line, re.insurer_raw,
           COUNT(*) AS dot_n_events_in_subent
    FROM recent_events re
    WHERE re.effective_date < {T_SPLIT_SQL}
    GROUP BY re.usdot, re.carrier_group, re.state, re.line, re.insurer_raw
    """)
dot_subent_contrib = con.execute("SELECT * FROM dot_subent_contrib").df()
# Keep usdot as int64 for consistent merge dtype across all tables
dot_subent_contrib['usdot'] = dot_subent_contrib['usdot'].astype('int64')
dot_subent_contrib['dot_n_events_in_subent'] = pd.to_numeric(
    dot_subent_contrib['dot_n_events_in_subent'], downcast='integer')
print(f"  dot_subent_contrib: {len(dot_subent_contrib):,} DOT-subentity pairs")

_log("Part 2D complete")


[t=1151.8s] starting Part 2D
  4,759 sub-entity centroids

Computing within-cell feature variances for distance weighting...
  Appetite-distance feature weights (sum to 1):
    sub_crash_count                 0.5475
    sub_inspection_oos_viol         0.4446
    sub_power_units_log             0.0060
    sub_crgo_genfreight             0.0006
    sub_crgo_bldgmat                0.0006
    sub_crgo_passengers             0.0003
    sub_crgo_intermodal             0.0002
    sub_crgo_chem                   0.0002
    sub_crgo_oilfield               0.0001
    sub_crgo_livestock              0.0000
  dot_subent_contrib: 636,587 DOT-subentity pairs
[t=1153.6s] Part 2D complete


### Part 2E — Load scorable DOTs into pandas + derived features

Includes: timing, coverage ratio, log fleet, NAICS bucketing, days_since_*,
          NEW ratio features, NEW rare-event flags, fleet_band
Cached to parquet: a kernel restart doesn't force re-running all the
Python-side feature derivation.

In [12]:
_log("starting Part 2E")

# Outlier-nulling bounds (data-quality fix for raw CSV).
# Past-only columns capped at today — a 2028 crash_latest_date is a
# data error. Future-allowed columns capped at year 2100 — hard
# "obviously-a-data-error" ceiling that catches 9999-01-01 entries
# but keeps multi-decade surety bonds / long-term contracts untouched.
_VALID_DATE_MIN      = pd.Timestamp('1990-01-01')
_VALID_DATE_MAX_PAST = pd.Timestamp.today().normalize()
_VALID_DATE_MAX_FUT  = pd.Timestamp('2100-01-01')

# (col, max_bound) — most dates are past-only; a few can be future-dated.
_DATE_COLS_PAST = [
    'crash_latest_date', 'inspection_latest_date', 'oos_latest_date',
    'revocation_latest_date', 'insurance_first_event_date',
    'insurance_latest_event_date', 'authority_first_served_date',
    'census_mcs150_date', 'current_bipd_effective_date',
]
_DATE_COLS_FUT = [
    'current_bipd_expiration_date',  # policies legitimately expire 1-2 years ahead
]
_NUMERIC_BOUNDS = {
    # Fleet / drivers
    'power_units': (0, 100_000), 'total_drivers': (0, 100_000),
    'total_cdl': (0, 100_000), 'truck_units': (0, 100_000),
    'bus_units': (0, 100_000), 'total_intrastate_drivers': (0, 100_000),
    'driver_inter_total': (0, 100_000), 'mcs150_mileage': (0, 1_000_000_000),
    # Crashes / inspections / enforcement
    'crash_count': (0, 10_000), 'crash_count_24m': (0, 10_000),
    'crash_fatalities': (0, 1_000), 'crash_injuries': (0, 10_000),
    'crash_tow_away_count': (0, 10_000),
    'inspection_count': (0, 1_000_000), 'inspection_violations': (0, 1_000_000),
    'inspection_oos_viol': (0, 100_000), 'inspections_with_oos': (0, 100_000),
    'inspection_driver_oos': (0, 100_000), 'inspection_vehicle_oos': (0, 100_000),
    'inspection_hazmat_oos': (0, 10_000),
    'oos_order_count': (0, 10_000), 'revocation_count': (0, 1_000),
    # Insurance history
    'ins_event_count': (0, 10_000), 'ins_distinct_insurers': (0, 1_000),
    'ins_cancelled_count': (0, 10_000), 'ins_replaced_count': (0, 10_000),
    'ins_rejected_count': (0, 10_000), 'ins_years_coverage': (0, 100),
    'ins_first_year': (1960, 2035), 'ins_last_year': (1960, 2035),
    # Coverage / price
    'coverage_amount_usd': (0, 100_000_000_000),
    'underlying_limit_usd': (0, 100_000_000_000),
    'min_required_coverage_usd': (0, 100_000_000_000),
    'authority_min_cov_amount': (0, 100_000_000_000),
    'authority_history_count': (0, 10_000),
    # Business profile
    'years_in_business': (0, 150),
    'co_usdot_count': (0, 100_000), 'co_phone_count': (0, 100_000),
    'co_officer_count': (0, 10_000), 'co_source_count': (0, 10_000),
    'co_resolved_company_count': (0, 10_000),
    'crash_rate_per_mil': (0, 10_000), 'wc_rate_multiplier': (0, 100),
}

_DOT_DF_CACHE = _cache_df('dot_df')
if not FORCE_REBUILD and _DOT_DF_CACHE.exists():
    dot_df = pd.read_parquet(_DOT_DF_CACHE)
    # Enforce int64 usdot — parquet round-trip can return nullable Int32,
    # which clashes with int64 usdot in dot_cell_contrib / dot_subent_contrib
    # at merge time (pandas Int64Factorizer dtype mismatch).
    dot_df['usdot'] = dot_df['usdot'].astype('int64')
    print(f"  [cache] loaded dot_df from parquet: {len(dot_df):,} rows, {dot_df.shape[1]} cols")
    _log_mem("after dot_df cache load")
    _log("Part 2E complete (cached)")
else:
    print("Loading scorable DOTs into pandas...")
    dot_df = con.execute("""
        WITH event_active AS (
            SELECT DISTINCT usdot FROM recent_events
            WHERE line IN ('surety_bond','surety','gl')
              AND event_status IN ('active','active (cancellation pending)')
        )
        SELECT * FROM dot_features df
        WHERE df.active_bipd_raw IS NOT NULL
           OR df.active_cargo_raw IS NOT NULL
           OR df.active_wc_raw IS NOT NULL
           OR df.wc_carrier_raw IS NOT NULL
           OR df.usdot IN (SELECT usdot FROM event_active)
    """).df()
    print(f"  {len(dot_df):,} scorable DOTs")
    _log_mem("after dot_df load")

    # Downcast dtypes (but keep usdot as int64 — all merges elsewhere use it
    # as the join key, and mixing int32/int64/Int64 across tables triggers a
    # dtype-mismatch factorize error on some pandas versions.)
    for c in dot_df.columns:
        if c == 'usdot':
            dot_df[c] = dot_df[c].astype('int64')
            continue
        if pd.api.types.is_integer_dtype(dot_df[c]):
            dot_df[c] = pd.to_numeric(dot_df[c], downcast='integer')
        elif pd.api.types.is_float_dtype(dot_df[c]):
            dot_df[c] = pd.to_numeric(dot_df[c], downcast='float')
    _log_mem("after dot_df downcast")

    # Null outlier dates (raw has entries like 9001-01-01 and 2028-xx).
    # Set to NaT so downstream days_since_* becomes NaN rather than
    # millions-of-days (or large-negative for "future" dates).
    print("Sanitizing outlier dates in dot_df...")
    _date_oob_summary = []
    for col, cap in ([(c, _VALID_DATE_MAX_PAST) for c in _DATE_COLS_PAST]
                     + [(c, _VALID_DATE_MAX_FUT) for c in _DATE_COLS_FUT]):
        if col not in dot_df.columns: continue
        s = pd.to_datetime(dot_df[col], errors='coerce')
        oob = (s < _VALID_DATE_MIN) | (s > cap)
        n_oob = int(oob.sum())
        if n_oob:
            _date_oob_summary.append((col, n_oob, s[oob].min(), s[oob].max(), cap))
        dot_df[col] = s.mask(oob)
    if _date_oob_summary:
        print(f"  Nulled {sum(x[1] for x in _date_oob_summary):,} out-of-range dates across "
              f"{len(_date_oob_summary)} columns:")
        for col, n, lo, hi, cap in _date_oob_summary:
            print(f"    {col:38s} {n:>7,} values  (observed {lo} .. {hi}; cap {cap.date()})")
    else:
        print("  No date outliers detected.")

    # Null outlier numerics (raw has entries like power_units in the millions).
    # Clipping would pretend a 1e9 is a legit 100k and poison the LOO sums.
    # Nulling is honest ("we don't know"); DuckDB SUM/AVG skip NULL, so LOO
    # aggregates stay clean, and LightGBM handles NaN natively.
    print("Nulling outlier numerics in dot_df...")
    _num_oob_summary = []
    for col, (lo, hi) in _NUMERIC_BOUNDS.items():
        if col not in dot_df.columns: continue
        s = pd.to_numeric(dot_df[col], errors='coerce')
        oob = (s < lo) | (s > hi)
        n_oob = int(oob.sum())
        if n_oob:
            _num_oob_summary.append((col, n_oob, float(s[oob].min()), float(s[oob].max())))
        dot_df[col] = s.mask(oob)
    if _num_oob_summary:
        print(f"  Nulled {sum(x[1] for x in _num_oob_summary):,} out-of-range values across "
              f"{len(_num_oob_summary)} columns:")
        for col, n, lo_, hi_ in _num_oob_summary:
            print(f"    {col:38s} {n:>7,} values  (observed range {lo_:.2f} .. {hi_:.2f})")
    else:
        print("  No numeric outliers detected.")

    # Timing features — anchored at T_SPLIT (training cutoff), not vintage.
    # Consistent with dot_features having been built on pre-cutoff events only;
    # days_since_* measures elapsed time from the latest pre-cutoff event to
    # the cutoff date. This keeps feature derivation internally consistent
    # with training-time and holdout-time scoring.
    today = T_SPLIT
    dot_df['current_bipd_effective_date']  = pd.to_datetime(dot_df['current_bipd_effective_date'], errors='coerce')
    dot_df['current_bipd_expiration_date'] = pd.to_datetime(dot_df['current_bipd_expiration_date'], errors='coerce')
    dot_df['days_since_bipd_effective']  = ((today - dot_df['current_bipd_effective_date']).dt.days).astype('float32')
    dot_df['days_until_bipd_expiration'] = ((dot_df['current_bipd_expiration_date'] - today).dt.days).astype('float32')
    dot_df['years_with_current_carrier'] = (dot_df['days_since_bipd_effective'] / 365.25).astype('float32')

    # Coverage-fit derived
    dot_df['coverage_vs_minimum_ratio']  = (dot_df['coverage_amount_usd']
                                            / dot_df['authority_min_cov_amount'].replace(0, np.nan)).astype('float32')
    dot_df['has_layered_coverage']       = dot_df['underlying_limit_usd'].notna().astype('int8')
    dot_df['power_units_log']            = np.log1p(dot_df['power_units'].fillna(0)).astype('float32')
    dot_df['hazmat_bin']                 = (dot_df['hazmat_ind'].astype(str).str.strip().str.upper()
                                            .isin(['Y','X'])).astype('int8')

    # NAICS bucketing
    dot_df['naics_sector']    = dot_df['primary_naics'].astype(str).str[:3].where(dot_df['primary_naics'].notna(), 'UNK')
    dot_df['naics_subsector'] = dot_df['primary_naics'].astype(str).str[:4].where(dot_df['primary_naics'].notna(), 'UNK')

    # Days-since-* recency features (anchored at data vintage)
    def _days_since(col):
        s = pd.to_datetime(dot_df[col], errors='coerce')
        return ((today - s).dt.days).astype('float32')

    for src_col, feat_col in [
        ('crash_latest_date', 'days_since_last_crash'),
        ('inspection_latest_date', 'days_since_last_inspection'),
        ('oos_latest_date', 'days_since_last_oos'),
        ('revocation_latest_date', 'days_since_last_revocation'),
        ('insurance_first_event_date', 'days_since_insurance_first'),
        ('insurance_latest_event_date', 'days_since_insurance_latest'),
        ('authority_first_served_date', 'days_since_authority_served'),
    ]:
        if src_col in dot_df.columns:
            dot_df[feat_col] = _days_since(src_col)

    if 'census_mcs150_date' in dot_df.columns:
        _mcs150 = pd.to_datetime(dot_df['census_mcs150_date'], errors='coerce')
        dot_df['days_since_mcs150_update'] = ((today - _mcs150).dt.days).astype('float32')

    # NEW: Ratio features (rates normalised by fleet / inspections / history)
    dot_df['crash_rate_per_pu']        = (dot_df['crash_count']         / (dot_df['power_units'] + 1)).astype('float32')
    dot_df['oos_rate_per_pu']          = (dot_df['oos_order_count']     / (dot_df['power_units'] + 1)).astype('float32')
    dot_df['violation_rate']           = (dot_df['inspection_violations']/ (dot_df['inspection_count'] + 1)).astype('float32')
    dot_df['oos_violation_rate']       = (dot_df['inspection_oos_viol'] / (dot_df['inspection_count'] + 1)).astype('float32')
    dot_df['driver_oos_rate']          = (dot_df['inspection_driver_oos']/ (dot_df['inspection_count'] + 1)).astype('float32')
    dot_df['vehicle_oos_rate']         = (dot_df['inspection_vehicle_oos']/ (dot_df['inspection_count'] + 1)).astype('float32')
    dot_df['hazmat_oos_rate']          = (dot_df['inspection_hazmat_oos']/ (dot_df['inspection_count'] + 1)).astype('float32')
    dot_df['insurance_cancel_rate']    = (dot_df['ins_cancelled_count'] / (dot_df['ins_event_count'] + 1)).astype('float32')
    dot_df['insurance_rejection_rate'] = (dot_df['ins_rejected_count']  / (dot_df['ins_event_count'] + 1)).astype('float32')
    dot_df['carrier_switch_rate']      = (dot_df['ins_distinct_insurers']/ (dot_df['ins_years_coverage'] + 1)).astype('float32')
    dot_df['driver_per_pu']            = (dot_df['total_drivers']       / (dot_df['power_units'] + 1)).astype('float32')

    # Age of insurance history (clipped to >= 0)
    dot_df['insurance_history_years']  = ((dot_df['ins_last_year'] - dot_df['ins_first_year']).fillna(0).clip(lower=0)).astype('float32')

    # NEW: Rare-event flags
    dot_df['had_any_fatal_crash']   = (dot_df['crash_fatalities'].fillna(0) > 0).astype('int8')
    dot_df['had_any_revocation']    = (dot_df['revocation_count'].fillna(0) > 0).astype('int8')
    dot_df['had_any_oos_order']     = (dot_df['oos_order_count'].fillna(0) > 0).astype('int8')
    dot_df['had_any_rejection']     = (dot_df['ins_rejected_count'].fillna(0) > 0).astype('int8')

    # Fleet band (moved from Part 4 to here)
    def _fleet_band(pu):
        if pu is None or pd.isna(pu): return 'UNK'
        pu = float(pu)
        if pu <= 1:   return 'B1'
        if pu <= 5:   return 'B2'
        if pu <= 20:  return 'B3'
        if pu <= 100: return 'B4'
        return 'B5'

    dot_df['fleet_band'] = dot_df['power_units'].map(_fleet_band)

    print(f"\nDerived features: {dot_df.shape[1]} columns in dot_df")

    # Cache to parquet for next session
    dot_df.to_parquet(_DOT_DF_CACHE)
    print(f"  cached dot_df to {_DOT_DF_CACHE} ({_DOT_DF_CACHE.stat().st_size / 1024**2:.1f} MB)")

    _log_mem("end of Part 2E")
    _log("Part 2E complete")


[t=1153.7s] starting Part 2E
Loading scorable DOTs into pandas...
  470,928 scorable DOTs
  [mem] after dot_df load: 1.59 GB
  [mem] after dot_df downcast: 1.38 GB
Sanitizing outlier dates in dot_df...
  Nulled 28,315 out-of-range dates across 4 columns:
    insurance_first_event_date               8,914 values  (observed 1936-11-16 00:00:00 .. 1989-12-31 00:00:00; cap 2026-10-29)
    insurance_latest_event_date              3,439 values  (observed 1936-11-16 00:00:00 .. 2027-12-13 00:00:00; cap 2026-10-29)
    authority_first_served_date             13,705 values  (observed 1900-01-01 00:00:00 .. 1989-12-29 00:00:00; cap 2026-10-29)
    current_bipd_effective_date              2,257 values  (observed 1936-11-16 00:00:00 .. 1989-12-27 00:00:00; cap 2026-10-29)
Nulling outlier numerics in dot_df...
  Nulled 113 out-of-range values across 9 columns:
    power_units                                  5 values  (observed range 100003.00 .. 150000.00)
    total_drivers                        

C:\Users\hi\AppData\Local\Temp\ipykernel_34224\816802806.py:149: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dot_df['days_since_bipd_effective']  = ((today - dot_df['current_bipd_effective_date']).dt.days).astype('float32')
C:\Users\hi\AppData\Local\Temp\ipykernel_34224\816802806.py:150: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dot_df['days_until_bipd_expiration'] = ((dot_df['current_bipd_expiration_date'] - today).dt.days).astype('float32')
C:\Users\hi\AppData\Local\Temp\ipykernel_34224\816802806.py:151: PerformanceWarni


Derived features: 196 columns in dot_df
  cached dot_df to c:\Users\hi\Documents\ROSELLA ASSIGNMENT\cache\dot_df.parquet (50.3 MB)
  [mem] end of Part 2E: 1.68 GB
[t=1167.3s] Part 2E complete


## Phase 3 — EDA & Data Quality

Analytical cells that validate the data pipeline but don't produce artefacts
needed downstream. Previously embedded inline in Phases 1 and 2 — moved here
to keep the main pipeline flow uncluttered. Run after Phase 2 completes.


### Part 3A — Sanity check (windowed properly to <= SNAPSHOT_DATE)

Fixes earlier bug: previously the event-derived "latest Active" could be AFTER
the May 2026 snapshot, producing false disagreements. Now filtered at the
snapshot date for apples-to-apples comparison.

In [8]:
_log("starting Part 3A (sanity check)")

print("=" * 70)
print(f"SANITY CHECK: active_bipd snapshot ({SNAPSHOT_DATE.date()}) vs latest Active-event")
print("                                   filtered to events on or before snapshot")
print("=" * 70)

sanity_q = f"""
WITH active_bipd_events AS (
    SELECT usdot,
           UPPER(TRIM(CAST(insurer AS VARCHAR))) AS event_insurer,
           effective_date,
           ROW_NUMBER() OVER (
               PARTITION BY usdot
               ORDER BY COALESCE(effective_date, filed_or_received_date) DESC NULLS LAST
           ) AS rn
    FROM raw
    WHERE LOWER(TRIM(CAST(policy_line AS VARCHAR))) IN ('auto liability', 'auto')
      AND LOWER(TRIM(CAST(event_status AS VARCHAR))) IN ('active', 'active (cancellation pending)')
      AND insurer IS NOT NULL
      AND effective_date <= {_SNAPSHOT_SQL}       -- windowed to snapshot date
),
latest_event AS (
    SELECT usdot, event_insurer, effective_date FROM active_bipd_events WHERE rn = 1
),
dot_snapshot AS (
    SELECT DISTINCT usdot,
           UPPER(TRIM(CAST(insurance_active_bipd_insurers_as_of_2026_05_26 AS VARCHAR))) AS snapshot_insurer
    FROM raw
    WHERE insurance_active_bipd_insurers_as_of_2026_05_26 IS NOT NULL
)
SELECT CASE
         WHEN snapshot_insurer IS NULL AND event_insurer IS NULL THEN 'both_null'
         WHEN snapshot_insurer IS NOT NULL AND event_insurer IS NULL THEN 'snapshot_only (no Active event found)'
         WHEN snapshot_insurer IS NULL AND event_insurer IS NOT NULL THEN 'event_only (not in snapshot)'
         WHEN snapshot_insurer = event_insurer THEN 'exact_match'
         WHEN POSITION(event_insurer IN snapshot_insurer) > 0
           OR POSITION(snapshot_insurer IN event_insurer) > 0 THEN 'partial_match'
         ELSE 'disagreement'
       END AS category,
       COUNT(*) AS n_dots
FROM dot_snapshot
FULL OUTER JOIN latest_event USING (usdot)
WHERE snapshot_insurer IS NOT NULL OR event_insurer IS NOT NULL
GROUP BY 1
ORDER BY n_dots DESC
"""
sanity_df = con.execute(sanity_q).fetchdf()
print(sanity_df.to_string(index=False))
total = sanity_df['n_dots'].sum()
agree = sanity_df.loc[sanity_df['category'].isin(['exact_match', 'partial_match']), 'n_dots'].sum()
if total > 0:
    print(f"\nAgreement rate: {agree}/{total} = {100*agree/total:.1f}%")

_log("Part 3A complete")


[t=11.7s] starting Part 1D (sanity check)
SANITY CHECK: active_bipd snapshot (2026-05-26) vs latest Active-event
                                   filtered to events on or before snapshot


RuntimeError: Query interrupted

### Part 3B — Active-placement counts per line (diagnostic)

RUN-ONCE finding (output captured in notebook from the first run):
  - BIPD / cargo / WC: snapshot columns populated → clean positives.
  - surety_bond (BMC-84 broker bond): event_status populated → clean
    positives (~12,466 active DOTs).
  - GL: 8,679 events across 7,659 DOTs, event_status systematically NULL.
  - surety (non-BMC-84): 3,711 events across 3,274 DOTs, event_status
    systematically NULL.

Decision: EXCLUDE 'gl' and 'surety' from LINES_IN_SCOPE (see cell 1A).
Rationale: with no status column, "latest event" can't distinguish
active placements from historical/cancelled ones. Any imputation would
silently mislabel ex-customers as positives. The remaining four lines
(BIPD / cargo / WC / surety_bond) cover the FMCSA-mandated trucking
insurance stack. The diagnostic remains for provenance.

In [26]:
print("Diagnostic: active placement counts per line (what training will see)")
print("=" * 70)

# ---- From snapshot columns (BIPD, cargo, WC) ----
_snap = {
    'bipd':  'active_bipd_raw',
    'cargo': 'active_cargo_raw',
    'wc':    'active_wc_raw',
}
for ln, col in _snap.items():
    n = int((dot_df[col].notna()).sum())
    print(f"  {ln:12s}: snapshot column -> {n:,} DOTs with any active insurer string")

# ---- From recent_events (surety_bond, surety, GL) ----
print()
_q = con.execute("""
SELECT line,
       event_status,
       COUNT(*) AS n_events,
       COUNT(DISTINCT usdot) AS n_dots
FROM recent_events
WHERE line IN ('surety_bond','surety','gl')
GROUP BY line, event_status
ORDER BY line, event_status
""").df()
print(f"  Event-based lines (surety_bond, surety, gl) — status breakdown:")
if _q.empty:
    print("    (none)")
else:
    print(_q.to_string(index=False))

print()
_active = con.execute("""
WITH active AS (
    SELECT line, usdot
    FROM recent_events
    WHERE line IN ('surety_bond','surety','gl')
      AND event_status IN ('active','active (cancellation pending)')
)
SELECT line, COUNT(DISTINCT usdot) AS n_active_placement_dots
FROM active
GROUP BY line ORDER BY line
""").df()
print(f"  Current positive-pool size (what cell 33 will train on):")
if _active.empty:
    print("    (none — all three event-based lines have 0 active placements)")
else:
    print(_active.to_string(index=False))

print()
print("Threshold reminder from cell 1A:")
print(f"  MIN_CARRIER_PLACEMENTS = {MIN_CARRIER_PLACEMENTS}")
print(f"  Each line needs enough active-placement DOTs to pass the")
print(f"  per-line min and still produce >= 50 clean positives in cell 3C.")


Diagnostic: active placement counts per line (what training will see)
  bipd        : snapshot column -> 388,724 DOTs with any active insurer string
  cargo       : snapshot column -> 35,076 DOTs with any active insurer string
  wc          : snapshot column -> 70,303 DOTs with any active insurer string

  Event-based lines (surety_bond, surety, gl) — status breakdown:
       line                  event_status  n_events  n_dots
         gl                          None      8679    7659
     surety                          None      3711    3274
surety_bond                        active     12686   12466
surety_bond active (cancellation pending)       428     424
surety_bond                     cancelled      3450    3173
surety_bond                  name changed       273     264
surety_bond                       pending        87      83
surety_bond                      replaced      1619    1466
surety_bond                   transferred         5       4

  Current positive-pool siz

## Phase 4 — Model Training

Feature definitions, placement extraction (positives per line), labelling (clean positives / hard negatives / random soft negatives per positive), and LightGBM training with GroupKFold CV per line.

### Part 4A — Feature definitions + CAT_MAPS + classifier factory

In [13]:
_log("starting Part 4A")

DOT_NUM_FEATURES = [
    # Fleet shape
    'power_units', 'power_units_log', 'total_drivers', 'total_cdl', 'truck_units',
    'mcs150_mileage', 'bus_units',
    # Driver composition
    'total_intrastate_drivers', 'driver_inter_total',
    # Radius
    'inter_beyond_100', 'inter_within_100', 'intra_beyond_100', 'intra_within_100',
    # Risk record
    'crash_count', 'crash_count_24m', 'crash_fatalities', 'crash_injuries', 'crash_tow_away_count',
    'crash_rate_per_mil',
    'inspection_count', 'inspection_violations',
    'inspection_oos_viol', 'inspections_with_oos',
    'inspection_driver_oos', 'inspection_vehicle_oos', 'inspection_hazmat_oos',
    'oos_order_count', 'revocation_count', 'prior_revoke_flag',
    # SMS detail
    'sms_insp_total', 'sms_driver_insp_total', 'sms_vehicle_insp_total',
    'sms_driver_oos_insp_total', 'sms_vehicle_oos_insp_total',
    'sms_violation_count', 'sms_violation_oos_count', 'sms_violation_severity',
    'sms_unsafe_driv', 'sms_hos_driv', 'sms_driv_fit', 'sms_veh_maint', 'sms_contr_subst',
    'sms_unsafe_driv_viol', 'sms_hos_driv_viol', 'sms_driv_fit_viol',
    'sms_contr_subst_viol', 'sms_veh_maint_viol',
    'has_sms',
    # Authority
    'has_authority', 'authority_history_count',
    # Business
    'years_in_business', 'co_usdot_count', 'co_phone_count', 'co_officer_count',
    'co_source_count', 'co_resolved_company_count',
    'co_has_retirement_plan', 'co_has_health_plan', 'has_domain',
    'is_peo_flag', 'has_mcsip',
    # Insurance history
    'ins_event_count', 'ins_distinct_insurers',
    'ins_cancelled_count', 'ins_replaced_count', 'ins_rejected_count',
    'ins_years_coverage',
    # Commodity flags (30 total)
    'crgo_genfreight', 'crgo_bldgmat', 'crgo_construct', 'crgo_beverages',
    'crgo_paperprod', 'crgo_produce', 'crgo_meat', 'crgo_drybulk', 'crgo_coldfood',
    'crgo_intermodal', 'crgo_grainfeed', 'crgo_livestock', 'crgo_motoveh',
    'crgo_household', 'crgo_passengers', 'crgo_oilfield', 'crgo_chem', 'crgo_logpole',
    'crgo_cargoothr', 'crgo_machlrg', 'crgo_mobilehome', 'crgo_coalcoke',
    'crgo_garbage', 'crgo_metalsheet', 'crgo_drivetow', 'crgo_waterwell',
    'crgo_usmail', 'crgo_utility', 'crgo_farmsupp', 'crgo_liqgas',
    # Coverage / price proxies
    'coverage_amount_usd', 'underlying_limit_usd', 'min_required_coverage_usd',
    'authority_min_cov_amount', 'coverage_vs_minimum_ratio', 'has_layered_coverage',
    'rejected_filing_count',
    'wc_rate_multiplier',
    # NOTE: is_residual_market and is_cancellation_pending removed from features —
    # they are used in Part 4C to define positive/negative labels; keeping them as
    # features would cause direct label leakage.
    # Timing / recency
    'days_since_bipd_effective', 'days_until_bipd_expiration', 'years_with_current_carrier',
    'days_since_last_crash', 'days_since_last_inspection',
    'days_since_last_oos', 'days_since_last_revocation',
    'days_since_insurance_first', 'days_since_insurance_latest',
    'days_since_authority_served', 'days_since_mcs150_update',
    # Insurance event flag
    'has_insurance_event',
    # NEW: ratio features
    'crash_rate_per_pu', 'oos_rate_per_pu', 'violation_rate', 'oos_violation_rate',
    'driver_oos_rate', 'vehicle_oos_rate', 'hazmat_oos_rate',
    'insurance_cancel_rate', 'insurance_rejection_rate', 'carrier_switch_rate',
    'driver_per_pu', 'insurance_history_years',
    # NEW: rare-event flags
    'had_any_fatal_crash', 'had_any_revocation', 'had_any_oos_order', 'had_any_rejection',
]

DOT_CAT_FEATURES = [
    'state', 'phy_state', 'authority_bus_state_code',
    'naics_sector', 'naics_subsector', 'co_lifecycle_status', 'org_type',
    'fleetsize_band', 'carrier_operation', 'classdef', 'hazmat_ind',
    'carship', 'census_status_code', 'safety_rating', 'fleet_band',
    'authority_common_stat', 'authority_contract_stat', 'authority_broker_stat',
    'authority_bipd_file', 'authority_cargo_file', 'authority_bond_file',
    'authority_property_chk', 'authority_passenger_chk', 'authority_hhg_chk',
    'authority_private_auth_chk', 'authority_enterprise_chk',
    'authority_cargo_req', 'authority_bond_req',
    'authority_common_app_pend', 'authority_contract_app_pend', 'authority_broker_app_pend',
    'authority_common_rev_pend', 'authority_contract_rev_pend', 'authority_broker_rev_pend',
    'authority_undeliverable_mail',
    'sms_unsafe_driv_ac', 'sms_hos_driv_ac', 'sms_driv_fit_ac',
    'sms_contr_subst_ac', 'sms_veh_maint_ac',
    'co_match_confidence',
]

# Exclude SUM columns (used for LOO adjustment, not as model features)
# and sub-entity sum columns if any leaked in.
CARRIER_FEATURES = [c for c in carrier_state_agg.columns
                    if (c.startswith('carr_') or c.startswith('svc_'))
                    and not c.endswith('_sum')]

# LOO adjustment: maps feat_col -> (sum_col in carrier_state_agg, dot_col in dot_df).
# For each row where DOT is actually in the cell, we subtract its contribution.
LOO_MEAN_FEATURES = {
    'carr_mean_power':       ('carr_power_sum',       'power_units'),
    'carr_mean_crash':       ('carr_crash_sum',       'crash_count'),
    'carr_mean_insp_oos':    ('carr_insp_oos_sum',    'inspection_oos_viol'),
    'carr_mean_revoc':       ('carr_revoc_sum',       'revocation_count'),
    'carr_mean_oos':         ('carr_oos_sum',         'oos_order_count'),
    'carr_pct_hazmat':       ('carr_hazmat_sum',      'hazmat_bin'),
    'carr_pct_genfreight':   ('carr_genfreight_sum',  'crgo_genfreight'),
    'carr_pct_bldgmat':      ('carr_bldgmat_sum',     'crgo_bldgmat'),
    'carr_pct_construct':    ('carr_construct_sum',   'crgo_construct'),
    'carr_pct_livestock':    ('carr_livestock_sum',   'crgo_livestock'),
    'carr_pct_oilfield':     ('carr_oilfield_sum',    'crgo_oilfield'),
    'carr_pct_passengers':   ('carr_passengers_sum',  'crgo_passengers'),
    'carr_pct_chem':         ('carr_chem_sum',        'crgo_chem'),
    'carr_pct_meat':         ('carr_meat_sum',        'crgo_meat'),
    'carr_pct_intermodal':   ('carr_intermodal_sum',  'crgo_intermodal'),
    'carr_pct_motoveh':      ('carr_motoveh_sum',     'crgo_motoveh'),
    'carr_mean_inter_long':  ('carr_inter_long_sum',  'inter_beyond_100'),
    'carr_mean_intra_short': ('carr_intra_short_sum', 'intra_within_100'),
}

def apply_loo_adjustment(df):
    """Leave-one-out adjustment of carrier cell aggregates.
    Expects: carrier_state_agg + dot_df + dot_cell_contrib merged into df.
    Modifies the mean/pct feature columns in-place so they exclude this DOT's
    contribution to the cell they're actually placed in. For DOTs not in the
    cell (dot_n_events_in_cell=NaN -> 0), the adjustment is a no-op."""
    dot_n = df['dot_n_events_in_cell'].fillna(0).astype('float32')
    n_events = df['carr_n_events'].astype('float32')
    denom = (n_events - dot_n).clip(lower=1)
    for feat, (sum_col, dot_col) in LOO_MEAN_FEATURES.items():
        dot_val = df[dot_col].fillna(0).astype('float32')
        df[feat] = ((df[sum_col] - dot_n * dot_val) / denom).astype('float32')
    is_in_cell  = (dot_n > 0).astype('float32')
    dot_6m      = df['dot_in_6m'].fillna(0).astype('float32')
    dot_6to18m  = df['dot_in_6to18m'].fillna(0).astype('float32')
    df['carr_n_placements']      = (df['carr_n_placements']      - is_in_cell).clip(lower=0).astype('float32')
    df['carr_placements_6m']     = (df['carr_placements_6m']     - dot_6m    ).clip(lower=0).astype('float32')
    df['carr_placements_6to18m'] = (df['carr_placements_6to18m'] - dot_6to18m).clip(lower=0).astype('float32')
    df['carr_state_momentum']    = (df['carr_placements_6m']
                                     / (df['carr_placements_6to18m'] / 2.0 + 1)).astype('float32')
    return df

print(f"  DOT_NUM_FEATURES: {len(DOT_NUM_FEATURES)}")
print(f"  DOT_CAT_FEATURES: {len(DOT_CAT_FEATURES)}")
print(f"  CARRIER_FEATURES: {len(CARRIER_FEATURES)}")

# Validate feature columns exist
missing = [f for f in DOT_NUM_FEATURES + DOT_CAT_FEATURES if f not in dot_df.columns]
if missing:
    print(f"  WARNING: missing columns in dot_df: {missing}")

# Categorical encoding fit on all scorable DOTs
CAT_MAPS = {}
for c in DOT_CAT_FEATURES:
    if c not in dot_df.columns:
        continue
    vals = dot_df[c].astype('object').where(dot_df[c].notna(), 'UNK').astype(str)
    CAT_MAPS[c] = {v: i for i, v in enumerate(sorted(vals.unique()))}

def encode_dots(df):
    X_num = df[DOT_NUM_FEATURES].astype('float32').to_numpy()
    cats = []
    for c in DOT_CAT_FEATURES:
        vals = df[c].astype('object').where(df[c].notna(), 'UNK').astype(str)
        cats.append(vals.map(CAT_MAPS[c]).fillna(-1).astype('int32').to_numpy())
    X_cat = np.column_stack(cats) if cats else np.empty((len(df), 0), dtype='int32')
    return X_num, X_cat

def assemble_X(df):
    X_num, X_cat = encode_dots(df)
    X_carr = df[CARRIER_FEATURES].astype('float32').to_numpy()
    X = np.concatenate([X_num, X_carr, X_cat.astype('float32')], axis=1)
    n_num = len(DOT_NUM_FEATURES); n_carr = len(CARRIER_FEATURES)
    cat_idx = list(range(n_num + n_carr, n_num + n_carr + len(DOT_CAT_FEATURES)))
    cat_mask = np.zeros(X.shape[1], dtype=bool); cat_mask[cat_idx] = True
    return X, cat_mask, cat_idx

def make_classifier(cat_mask, cat_idx):
    # MODEL_BACKEND_TYPE switches between the binary classifier and the
    # LambdaRank-objective ranker. Shares hyperparameters so they're
    # directly comparable.
    if MODEL_BACKEND_TYPE == 'ranker':
        return lgb.LGBMRanker(
            objective='lambdarank', metric='ndcg',
            n_estimators=200, num_leaves=31, learning_rate=0.08,
            reg_lambda=1.0, random_state=RANDOM_SEED, verbose=-1,
            n_jobs=-1, min_child_samples=20,
            label_gain=[0, 1],
        )
    return lgb.LGBMClassifier(
        n_estimators=200, num_leaves=31, learning_rate=0.08,
        reg_lambda=1.0, random_state=RANDOM_SEED, verbose=-1,
        n_jobs=-1, min_child_samples=20,
    )


def _ranker_sort_and_group(X, y, groups, sw=None):
    """LGBMRanker requires rows contiguous per query group. Returns
    (X_sorted, y_sorted, sw_sorted, group_counts) where group_counts[i] is
    the number of rows for the i-th group."""
    order = np.argsort(groups, kind='stable')
    X_s = X[order]
    y_s = y[order]
    sw_s = sw[order] if sw is not None else None
    g_s = groups[order]
    # Count contiguous runs
    _, counts = np.unique(g_s, return_counts=True)
    assert counts.sum() == len(X_s), f"group count mismatch: {counts.sum()} vs {len(X_s)}"
    return X_s, y_s, sw_s, counts

def fit_classifier(model, X, y, cat_idx, groups=None):
    X_fit = X.copy()
    if cat_idx:
        X_fit[:, cat_idx] = np.round(X_fit[:, cat_idx]).astype(np.int32)
    if MODEL_BACKEND_TYPE == 'ranker':
        assert groups is not None, "ranker requires groups array"
        X_s, y_s, _, counts = _ranker_sort_and_group(X_fit, y, groups)
        model.fit(X_s, y_s, group=counts, categorical_feature=cat_idx)
    else:
        model.fit(X_fit, y, categorical_feature=cat_idx)
    return model

def predict_scores(model, X, cat_idx):
    X_pred = X
    if cat_idx:
        X_pred = X.copy()
        X_pred[:, cat_idx] = np.round(X_pred[:, cat_idx]).astype(np.int32)
    # Ranker returns raw scores (not probabilities) — same monotonic ordering,
    # so downstream ranking logic is unchanged.
    if MODEL_BACKEND_TYPE == 'ranker':
        return model.predict(X_pred).astype('float32')
    return model.predict_proba(X_pred)[:, 1].astype('float32')

# Training helpers — shared across base (Phase 4) and weighted (Phase 6).
# These eliminate ~70% duplication between build_training_base and
# build_training_weighted. The global `rng` is still used — order of
# helper calls is preserved, so models cached from previous runs stay valid.

def _sample_soft_negatives(pos, modelable, k=N_NEGATIVES_PER_POSITIVE, oversample=2):
    """For each positive, draw k * oversample random carriers from the
    DOT's state pool. Returns a DataFrame with columns
    ['usdot','carrier_group','state','_pos_idx']. _pos_idx tracks which
    positive each candidate soft neg came from (so dedupe is K per
    positive, not K per DOT)."""
    pool_by_state = {st: g['carrier_group'].to_numpy() for st, g in modelable.groupby('state')}
    k_total = k * oversample
    pos_arr_u = pos['usdot'].to_numpy()
    pos_arr_s = pos['state'].to_numpy()
    pos_idx_arr = np.arange(len(pos), dtype=np.int32)
    exp_u = np.repeat(pos_arr_u, k_total)
    exp_s = np.repeat(pos_arr_s, k_total)
    exp_pos_idx = np.repeat(pos_idx_arr, k_total)
    exp_c = np.empty(len(exp_u), dtype=object)
    for st in np.unique(exp_s):
        pool = pool_by_state.get(st)
        if pool is None or len(pool) == 0: continue
        mask = exp_s == st
        n_mask = int(mask.sum())
        exp_c[mask] = pool[rng.integers(0, len(pool), size=n_mask)]
    soft = pd.DataFrame({'usdot': exp_u, 'carrier_group': exp_c, 'state': exp_s,
                         '_pos_idx': exp_pos_idx})
    return soft.dropna(subset=['carrier_group'])


def _build_hard_negatives(all_obs, line_label, modelable):
    """Residual-market, revoked, and rejected pairs as hard negatives (y=0).
    all_obs must have 'is_residual_market' column merged in. Returns
    (hard_rm, hard_rev, hard_rej) with ['usdot','carrier_group','state','y']."""
    hard_rm = all_obs.loc[all_obs['is_residual_market'] == 1,
                          ['usdot','carrier_group','state']].copy()
    hard_rm['y'] = np.int8(0)
    hard_rev = revoked_pairs[revoked_pairs['line'] == line_label][
        ['usdot','carrier_group','state']
    ].merge(modelable, on=['carrier_group','state'], how='inner')
    hard_rev['y'] = np.int8(0)
    # Rejected pairs — carrier explicitly declined. Stronger per-pair signal
    # than the carrier-level svc_rejection_rate feature.
    hard_rej = rejected_pairs[rejected_pairs['line'] == line_label][
        ['usdot','carrier_group','state']
    ].merge(modelable, on=['carrier_group','state'], how='inner')
    hard_rej['y'] = np.int8(0)
    return hard_rm, hard_rev, hard_rej


def _dedupe_and_cap_soft(soft, pos, hard_rm, hard_rev, hard_rej=None, k=N_NEGATIVES_PER_POSITIVE):
    """Remove soft candidates that collide with any observed placement
    (pos / hard_rm / hard_rev / hard_rej), then cap at k per positive via
    _pos_idx groupby.head. Returns a DataFrame with y=0 added."""
    frames = [pos[['usdot','carrier_group','state']],
              hard_rm[['usdot','carrier_group','state']],
              hard_rev[['usdot','carrier_group','state']]]
    if hard_rej is not None and len(hard_rej) > 0:
        frames.append(hard_rej[['usdot','carrier_group','state']])
    all_placed = pd.concat(frames,
                           ignore_index=True).drop_duplicates()
    all_placed['_pos'] = np.int8(1)
    soft = soft.merge(all_placed, on=['usdot','carrier_group','state'], how='left')
    soft = soft[soft['_pos'].isna()].drop(columns='_pos')
    soft = soft.groupby('_pos_idx').head(k).drop(columns='_pos_idx').reset_index(drop=True)
    soft['y'] = np.int8(0)
    return soft


def _assemble_training_frame(train, line_label):
    """Add 'line' column, merge carrier_state_agg + dot_df + dot_cell_contrib,
    apply LOO adjustment. Takes a train DataFrame that already has
    (usdot, carrier_group, state, y, optionally _weight)."""
    train = train.copy()
    train['line'] = line_label
    train = train.merge(carrier_state_agg, on=['carrier_group','state','line'], how='left')
    train = train.merge(dot_df, on='usdot', how='left', suffixes=('', '_dot'))
    train = train.merge(dot_cell_contrib, on=['usdot','carrier_group','state','line'], how='left')
    return apply_loo_adjustment(train)


def _fit_cv_and_refit(train_df, line_label, n_cv_splits=3, use_weights=False):
    """GroupKFold CV + final refit on full training set. Prints CV metrics.
    If use_weights=True, train_df['_weight'] is passed as sample_weight to
    LightGBM. Returns the final fitted model."""
    X, cat_mask, cat_idx = assemble_X(train_df)
    y = train_df['y'].to_numpy()
    sw = train_df['_weight'].to_numpy() if use_weights else None
    groups = train_df['usdot'].to_numpy()

    aucs, aps = [], []
    gkf = GroupKFold(n_splits=min(n_cv_splits, max(2, int(np.unique(groups).size // 100))))
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups)):
        m = make_classifier(cat_mask, cat_idx)
        if use_weights:
            _fit_with_weights_local(m, X[tr], y[tr], sw[tr], cat_idx, groups=groups[tr])
        else:
            fit_classifier(m, X[tr], y[tr], cat_idx, groups=groups[tr])
        p = predict_scores(m, X[te], cat_idx)
        if np.unique(y[te]).size >= 2:
            aucs.append(roc_auc_score(y[te], p))
            aps.append(average_precision_score(y[te], p))
        del m, p
        gc.collect()
    if aucs:
        print(f"  CV ROC-AUC: {np.mean(aucs):.4f} +/- {np.std(aucs):.4f} | "
              f"PR-AUC: {np.mean(aps):.4f} +/- {np.std(aps):.4f}")

    final = make_classifier(cat_mask, cat_idx)
    if use_weights:
        _fit_with_weights_local(final, X, y, sw, cat_idx, groups=groups)
    else:
        fit_classifier(final, X, y, cat_idx, groups=groups)
    return final


def _fit_with_weights_local(model, X, y, sw, cat_idx, groups=None):
    """LightGBM .fit() with sample_weight. Mirrors fit_classifier's
    categorical-feature + ranker handling."""
    X_fit = X.copy()
    if cat_idx:
        X_fit[:, cat_idx] = np.round(X_fit[:, cat_idx]).astype(np.int32)
    if MODEL_BACKEND_TYPE == 'ranker':
        assert groups is not None, "ranker requires groups array"
        X_s, y_s, sw_s, counts = _ranker_sort_and_group(X_fit, y, groups, sw)
        model.fit(X_s, y_s, sample_weight=sw_s, group=counts, categorical_feature=cat_idx)
    else:
        model.fit(X_fit, y, sample_weight=sw, categorical_feature=cat_idx)
    return model

_log("Part 4A complete")


[t=1167.3s] starting Part 4A
  DOT_NUM_FEATURES: 133
  DOT_CAT_FEATURES: 41
  CARRIER_FEATURES: 36
[t=1170.1s] Part 4A complete


### Part 4B — Placement extraction + revoked pairs

In [14]:
_log("starting Part 4B")

def extract_placements_snapshot(raw_col, line_label, extra_raw_col=None):
    cols = ['usdot', 'phy_state', 'state', raw_col] + ([extra_raw_col] if extra_raw_col else [])
    sub = dot_df[cols].copy()
    sub['dot_state'] = sub['phy_state'].fillna(sub['state'])
    def collect(row):
        out = consolidate_multi(row[raw_col])
        if extra_raw_col:
            out = list(dict.fromkeys(out + consolidate_multi(row[extra_raw_col])))
        return out
    sub['carriers'] = sub.apply(collect, axis=1)
    sub = sub[sub['carriers'].str.len() > 0]
    exploded = sub[['usdot', 'dot_state', 'carriers']].explode('carriers')
    exploded = exploded.rename(columns={'carriers': 'carrier_group', 'dot_state': 'state'})
    exploded['line'] = line_label
    # Normalise state
    exploded['state'] = exploded['state'].astype(str).str.upper().str.strip()
    return exploded[['usdot','carrier_group','state','line']].drop_duplicates().reset_index(drop=True)

def extract_placements_from_events(line_label):
    q = f"""
    WITH active AS (
        SELECT re.usdot, re.carrier_group, re.state, re.effective_date,
               ROW_NUMBER() OVER (
                   PARTITION BY re.usdot
                   ORDER BY re.effective_date DESC NULLS LAST
               ) AS rn
        FROM recent_events re
        WHERE re.line = '{line_label}'
          AND re.event_status IN ('active','active (cancellation pending)')
    )
    SELECT usdot, carrier_group, state FROM active WHERE rn = 1
    """
    p = con.execute(q).df()
    p['line'] = line_label
    p['state'] = p['state'].astype(str).str.upper().str.strip()
    return p[['usdot','carrier_group','state','line']].drop_duplicates().reset_index(drop=True)

print("Extracting placements per line...")
# 'surety' and 'gl' excluded from scope — see cell 1A comment and the Part 3B
# diagnostic. Their raw events have systematically NULL event_status, so we
# cannot reliably distinguish current placements from historical records.
placements_bipd        = extract_placements_snapshot('active_bipd_raw',  'bipd')
placements_cargo       = extract_placements_snapshot('active_cargo_raw', 'cargo')
placements_wc          = extract_placements_snapshot('active_wc_raw',    'wc', extra_raw_col='wc_carrier_raw')
placements_surety_bond = extract_placements_from_events('surety_bond')

ALL_PLACEMENTS = {
    'bipd':        placements_bipd,
    'cargo':       placements_cargo,
    'wc':          placements_wc,
    'surety_bond': placements_surety_bond,
}

for name, p in ALL_PLACEMENTS.items():
    kept = p.merge(carrier_state_agg[['carrier_group','state','line']],
                   on=['carrier_group','state','line'], how='inner')
    print(f"  {name:12s}: {len(p):,} raw -> {len(kept):,} in modelable cells  "
          f"({kept['usdot'].nunique():,} DOTs)")

print("\nExtracting REVOKED hard-negative pairs (pre-cutoff only)...")
revoked_pairs = con.execute(f"""
    SELECT DISTINCT usdot, carrier_group, state, line
    FROM recent_events
    WHERE end_reason = 'revoked'
      AND effective_date < {T_SPLIT_SQL}
""").df()
print(f"  {len(revoked_pairs):,} REVOKED pairs across all lines")

print("\nExtracting REJECTED hard-negative pairs (pre-cutoff only)...")
rejected_pairs = con.execute(f"""
    SELECT DISTINCT usdot, carrier_group, state, line
    FROM recent_events
    WHERE event_status = 'rejected'
      AND effective_date < {T_SPLIT_SQL}
""").df()
print(f"  {len(rejected_pairs):,} REJECTED pairs across all lines")

# Per-(usdot, line) latest effective_date — used by Part 4C training (filter
# to DOTs whose latest event is pre-cutoff) and Part 5A validation (hold out
# DOTs whose latest event is post-cutoff). Previously computed in Part 6A as
# LINE_TENURE_DF; moved here so it's available at training time.
print("\nComputing per-DOT per-line latest effective date (for pre/post split)...")
LINE_LATEST_EFF = con.execute(f"""
    SELECT line, usdot, MAX(effective_date) AS latest_eff_date
    FROM recent_events
    GROUP BY line, usdot
""").df()
LINE_LATEST_EFF['usdot'] = LINE_LATEST_EFF['usdot'].astype('int64')
LINE_LATEST_EFF['latest_eff_date'] = pd.to_datetime(LINE_LATEST_EFF['latest_eff_date'], errors='coerce')
_n_pre_per_line = (LINE_LATEST_EFF['latest_eff_date'] < T_SPLIT).groupby(LINE_LATEST_EFF['line']).sum()
_n_post_per_line = (LINE_LATEST_EFF['latest_eff_date'] >= T_SPLIT).groupby(LINE_LATEST_EFF['line']).sum()
print(f"  pre-cutoff DOTs per line:  {dict(_n_pre_per_line)}")
print(f"  post-cutoff DOTs per line: {dict(_n_post_per_line)}  <- holdout for honest metrics")

_log("Part 4B complete")


[t=1170.1s] starting Part 4B
Extracting placements per line...
  bipd        : 415,584 raw -> 323,294 in modelable cells  (321,969 DOTs)
  cargo       : 35,729 raw -> 3,657 in modelable cells  (3,653 DOTs)
  wc          : 113,549 raw -> 83,722 in modelable cells  (58,810 DOTs)
  surety_bond : 15,117 raw -> 12,611 in modelable cells  (12,611 DOTs)

Extracting REVOKED hard-negative pairs (pre-cutoff only)...
  4 REVOKED pairs across all lines

Extracting REJECTED hard-negative pairs (pre-cutoff only)...
  0 REJECTED pairs across all lines

Computing per-DOT per-line latest effective date (for pre/post split)...
  pre-cutoff DOTs per line:  {'bipd': np.int64(347213), 'cargo': np.int64(5505), 'surety_bond': np.int64(18089), 'wc': np.int64(49584)}
  post-cutoff DOTs per line: {'bipd': np.int64(23371), 'cargo': np.int64(191), 'surety_bond': np.int64(480), 'wc': np.int64(19463)}  <- holdout for honest metrics
[t=1202.5s] Part 4B complete


### Part 4C — Training set builder + Train 6 models with CV

In [15]:
_log("starting Part 4C")

def _filter_to_pre_cutoff(placements, line_label):
    """Keep only DOTs whose latest event in this line is strictly before T_SPLIT.
    This is the honest training set — DOTs with any post-cutoff activity are
    held out for validation."""
    _line_dates = LINE_LATEST_EFF[LINE_LATEST_EFF['line'] == line_label][['usdot','latest_eff_date']]
    merged = placements.merge(_line_dates, on='usdot', how='left')
    pre = merged[merged['latest_eff_date'] < T_SPLIT].drop(columns='latest_eff_date')
    return pre


def build_training_base(placements, line_label):
    # Honest-training filter: only pre-cutoff DOTs. The deliverable scores
    # all DOTs (pre + post) with these models at output time; training uses
    # pre-cutoff only so there's no label leakage into the holdout.
    placements = _filter_to_pre_cutoff(placements, line_label)
    modelable = carrier_state_agg[carrier_state_agg['line'] == line_label][['carrier_group','state']]
    all_obs = placements.merge(modelable, on=['carrier_group','state'], how='inner').copy()
    all_obs = all_obs.merge(dot_df[['usdot','is_residual_market','is_cancellation_pending']],
                            on='usdot', how='left')

    clean_mask = (all_obs['is_residual_market'] == 0) & (all_obs['is_cancellation_pending'] == 0)
    pos = all_obs[clean_mask][['usdot','carrier_group','state']].copy()
    pos['y'] = np.int8(1)

    hard_rm, hard_rev, hard_rej = _build_hard_negatives(all_obs, line_label, modelable)
    soft = _sample_soft_negatives(pos, modelable)
    soft = _dedupe_and_cap_soft(soft, pos, hard_rm, hard_rev, hard_rej)

    train = pd.concat([pos, hard_rm, hard_rev, hard_rej, soft], ignore_index=True)
    train = _assemble_training_frame(train, line_label)
    return train, len(pos), len(hard_rm) + len(hard_rev) + len(hard_rej), len(soft)

def train_base_model(placements, line_label, n_cv_splits=3):
    print(f"\n--- Training {line_label.upper()} ---")
    _log_mem(f"[{line_label}] before build")

    train_df, n_pos, n_hard, n_soft = build_training_base(placements, line_label)
    if n_pos < 50:
        print(f"  insufficient clean positives ({n_pos}); skipping")
        return None

    print(f"  training rows: {len(train_df):,}  (clean_pos={n_pos:,}, hard_neg={n_hard:,}, soft_neg={n_soft:,})")
    _log_mem(f"[{line_label}] after build")

    # Fold-0 tenure diagnostic preserved — pull from training set before CV loop
    # (actually we inline it into the CV loop; see commented block below if needed)
    final = _fit_cv_and_refit(train_df, line_label, n_cv_splits=n_cv_splits, use_weights=False)
    del train_df; gc.collect()
    _log_mem(f"[{line_label}] after refit")
    return final

# MODELS joblib cache — one trained model-dict per backend. Both backends
# are trained in a single run so you can compare classifier vs ranker without
# flipping config and rerunning. MODELS is an alias for the classifier
# variant so existing downstream code (Part 5A scoring, Part 7A importance,
# deliverable write) keeps working. The ranker is available as MODELS_BY_BACKEND['ranker'].
MODELS_BY_BACKEND = {}
for _bt in BACKEND_TYPES:
    MODEL_BACKEND_TYPE = _bt  # make_classifier reads this at construction time
    _MODELS_CACHE = _cache_pkl(f'models_{_bt}')
    if not FORCE_REBUILD and _MODELS_CACHE.exists():
        MODELS_BY_BACKEND[_bt] = joblib.load(_MODELS_CACHE)
        _n = sum(1 for m in MODELS_BY_BACKEND[_bt].values() if m is not None)
        print(f"  [cache] loaded {_n}/{len(MODELS_BY_BACKEND[_bt])} {_bt} models from {_MODELS_CACHE}")
    else:
        print(f"\n=== Training {_bt.upper()} backend ===")
        _m = {}
        for line_name, p in ALL_PLACEMENTS.items():
            _m[line_name] = train_base_model(p, line_name)
        MODELS_BY_BACKEND[_bt] = _m
        joblib.dump(_m, _MODELS_CACHE)
        print(f"  cached {_bt} models to {_MODELS_CACHE} "
              f"({_MODELS_CACHE.stat().st_size / 1024**2:.1f} MB)")

# Default aliases for downstream cells. The classifier is the "primary"
# variant; ranker is a comparison alternative.
MODELS = MODELS_BY_BACKEND['classifier']
MODEL_BACKEND_TYPE = 'classifier'

_log("Part 4C complete")


[t=1202.5s] starting Part 4C

=== Training CLASSIFIER backend ===

--- Training BIPD ---
  [mem] [bipd] before build: 1.90 GB
  training rows: 782,694  (clean_pos=195,930, hard_neg=821, soft_neg=585,943)
  [mem] [bipd] after build: 2.87 GB
  CV ROC-AUC: 0.9647 +/- 0.0006 | PR-AUC: 0.9195 +/- 0.0010
  [mem] [bipd] after refit: 2.19 GB

--- Training CARGO ---
  [mem] [cargo] before build: 2.19 GB
  training rows: 6,159  (clean_pos=1,596, hard_neg=2, soft_neg=4,561)
  [mem] [cargo] after build: 2.20 GB
  CV ROC-AUC: 0.9933 +/- 0.0016 | PR-AUC: 0.9796 +/- 0.0016
  [mem] [cargo] after refit: 2.20 GB

--- Training WC ---
  [mem] [wc] before build: 2.20 GB
  training rows: 216,760  (clean_pos=54,453, hard_neg=286, soft_neg=162,021)
  [mem] [wc] after build: 2.44 GB
  CV ROC-AUC: 0.9614 +/- 0.0021 | PR-AUC: 0.9234 +/- 0.0039
  [mem] [wc] after refit: 1.82 GB

--- Training SURETY_BOND ---
  [mem] [surety_bond] before build: 1.82 GB
  training rows: 46,640  (clean_pos=11,726, hard_neg=73, soft_n

## Phase 5 — Base Model Scoring + Ensemble

Full-universe scoring with the base classifier and ranker from Phase 4, followed by rank-average ensemble construction. Headline hit-rate table reports metrics on the **honest holdout slice** (DOTs with `latest_eff_date >= T_SPLIT`) — these are DOTs the model never saw in training. Full-universe scoring (including pre-cutoff DOTs) still runs for the Phase 8 deliverable.

Segmentation shows BIPD by state and fleet band for the same holdout, with a head-to-head comparison across backends.

Deep validation diagnostics live in Phase 7 (after the weighted sensitivity retrain) so they can be inspected across all variants.

### Part 5A — Streaming per-state scoring + baselines + evaluation (6 lines)

In [17]:
_log("starting Part 5A")

K_LONG = TOP_K_FULL_RANKING

def score_streaming(placements, model, line_label):
    if model is None or len(placements) == 0:
        return pd.DataFrame(), pd.DataFrame()

    dots = placements[['usdot', 'state']].drop_duplicates()
    dots_full = dots.merge(dot_df, on='usdot', how='left', suffixes=('', '_df'))
    dots_full = dots_full.rename(columns={'state': 'placement_state'})
    # Filter residual-market-focused carriers from the recommendation pool.
    # They remain in training (as hard negatives) so the model learns to avoid
    # them, but we don't want to recommend them to brokers.
    cells = carrier_state_agg[
        (carrier_state_agg['line'] == line_label)
        & (carrier_state_agg['carr_pct_residual_market'].fillna(0) < RESIDUAL_MARKET_CARRIER_THRESHOLD)
    ]
    observed_pairs = placements[placements['line'] == line_label][['usdot', 'carrier_group', 'state']]

    ranked_parts, obs_rank_parts = [], []
    states = sorted(cells['state'].dropna().unique())
    n_states = len(states); cands_total = 0

    for i, st in enumerate(states, 1):
        state_cells = cells[cells['state'] == st]
        state_dots = dots_full[dots_full['placement_state'] == st]
        if state_cells.empty or state_dots.empty: continue

        state_contrib = dot_cell_contrib[
            (dot_cell_contrib['state'] == st) & (dot_cell_contrib['line'] == line_label)
        ]
        obs_this_state = observed_pairs[observed_pairs['state'] == st]

        # Intra-state DOT batching to bound peak memory on big states. Each DOT
        # appears in exactly one batch with the full state_cells pool, so rank
        # within DOT can be computed per batch (no cross-batch grouping needed).
        pool_size_in_state = np.int32(state_cells['carrier_group'].nunique())
        n_dots_in_state = len(state_dots)
        for start in range(0, n_dots_in_state, SCORING_BATCH_DOTS):
            batch_dots = state_dots.iloc[start:start+SCORING_BATCH_DOTS]
            cross = batch_dots.assign(_k=1).merge(state_cells.assign(_k=1), on='_k').drop(columns='_k')
            cross['state'] = st
            # LOO: subtract DOT's own contribution from the carrier aggregate it's placed in
            cross = cross.merge(state_contrib, on=['usdot','carrier_group','state','line'], how='left')
            cross = apply_loo_adjustment(cross)
            cands_total += len(cross)

            X, _, cat_idx = assemble_X(cross)
            cross['fit_score'] = predict_scores(model, X, cat_idx)
            del X

            # Deterministic rank within DOT (each DOT only in one batch, so this
            # groupby is batch-local and correct).
            cross = cross.sort_values(['usdot','fit_score','carrier_group'],
                                       ascending=[True, False, True]).reset_index(drop=True)
            cross['rank']      = (cross.groupby('usdot').cumcount() + 1).astype('int32')
            cross['pool_size'] = pool_size_in_state

            batch_usdots = cross['usdot'].unique()
            obs_batch = obs_this_state[obs_this_state['usdot'].isin(batch_usdots)]
            obs = obs_batch.merge(
                cross[['usdot','carrier_group','rank','fit_score','pool_size']],
                on=['usdot','carrier_group'], how='inner'
            ).rename(columns={'rank': 'observed_rank'})
            obs['line'] = line_label
            obs_rank_parts.append(obs)

            top = cross.loc[cross['rank'] <= K_LONG,
                            ['usdot','carrier_group','state','fit_score','rank','pool_size']].copy()
            top['line'] = line_label
            ranked_parts.append(top)
            del cross
            gc.collect()

        if i % 10 == 0 or i == n_states:
            print(f"  [{line_label}] processed {i}/{n_states} states ({cands_total:,} candidate pairs seen)")

    ranked_long = pd.concat(ranked_parts, ignore_index=True) if ranked_parts else pd.DataFrame()
    obs_rank    = pd.concat(obs_rank_parts, ignore_index=True) if obs_rank_parts else pd.DataFrame()
    _log_mem(f"[{line_label}] after streaming scoring")
    return ranked_long, obs_rank

def eval_against_baselines(placements, obs_rank, line_label):
    if obs_rank is None or len(obs_rank) == 0:
        empty = {'n_dots':0,'top1_hit_rate':np.nan,'top5_hit_rate':np.nan,'top10_hit_rate':np.nan,'mrr':np.nan}
        return empty, empty, empty

    model_m = {
        'n_dots': len(obs_rank),
        'top1_hit_rate':  (obs_rank['observed_rank'] <= 1).mean(),
        'top5_hit_rate':  (obs_rank['observed_rank'] <= 5).mean(),
        'top10_hit_rate': (obs_rank['observed_rank'] <= 10).mean(),
        'mrr':            (1.0 / obs_rank['observed_rank']).mean(),
    }

    cells = carrier_state_agg[carrier_state_agg['line'] == line_label][['carrier_group','state']]
    pool_sizes = cells.groupby('state').size().to_dict()
    obs = placements[placements['line'] == line_label][['usdot','carrier_group','state']]

    obs_b1 = obs.merge(cells, on=['carrier_group','state'], how='inner').copy()
    # Defensive: handle states missing from pool_sizes (tiny-line edge cases)
    def _rand_rank(n):
        if pd.isna(n) or n < 1: return 1
        return int(rng.integers(1, int(n) + 1))
    obs_b1['observed_rank'] = obs_b1['state'].map(pool_sizes).apply(_rand_rank)
    b1_m = {
        'n_dots': len(obs_b1),
        'top1_hit_rate':  (obs_b1['observed_rank'] <= 1).mean(),
        'top5_hit_rate':  (obs_b1['observed_rank'] <= 5).mean(),
        'top10_hit_rate': (obs_b1['observed_rank'] <= 10).mean(),
        'mrr':            (1.0 / obs_b1['observed_rank']).mean(),
    }

    all_p = placements[placements['line'] == line_label].merge(
        dot_df[['usdot','fleet_band']], on='usdot', how='left'
    )
    counts = all_p.groupby(['state','fleet_band','carrier_group']).size().reset_index(name='n')
    # Deterministic tie-break on carrier_group (see note in score_streaming).
    counts = counts.sort_values(['state','fleet_band','n','carrier_group'],
                                 ascending=[True, True, False, True]).reset_index(drop=True)
    counts['pop_rank'] = (counts.groupby(['state','fleet_band']).cumcount() + 1).astype('int32')
    pool_sz = counts.groupby(['state','fleet_band']).size().reset_index(name='pool_n')

    obs_b2 = obs.merge(cells, on=['carrier_group','state'], how='inner')
    obs_b2 = obs_b2.merge(dot_df[['usdot','fleet_band']], on='usdot', how='left')
    obs_b2 = obs_b2.merge(counts[['state','fleet_band','carrier_group','pop_rank']],
                          on=['state','fleet_band','carrier_group'], how='left')
    obs_b2 = obs_b2.merge(pool_sz, on=['state','fleet_band'], how='left')
    obs_b2['observed_rank'] = obs_b2['pop_rank'].fillna(obs_b2['pool_n']).fillna(1).astype('int32')
    b2_m = {
        'n_dots': len(obs_b2),
        'top1_hit_rate':  (obs_b2['observed_rank'] <= 1).mean(),
        'top5_hit_rate':  (obs_b2['observed_rank'] <= 5).mean(),
        'top10_hit_rate': (obs_b2['observed_rank'] <= 10).mean(),
        'mrr':            (1.0 / obs_b2['observed_rank'].astype('float')).mean(),
    }
    return model_m, b1_m, b2_m

print("\n" + "=" * 70)
print(f"SCORING (streaming per state, {len(ALL_PLACEMENTS)} lines)")
print("=" * 70)

# SCORED/OBS_RANK pickle cache — one scored dict per backend. MODEL_BACKEND_TYPE
# must match the backend of the models being scored, so predict_scores
# dispatches correctly (.predict vs .predict_proba).
SCORED_BY_BACKEND = {}
OBS_RANK_BY_BACKEND = {}
for _bt in BACKEND_TYPES:
    MODEL_BACKEND_TYPE = _bt
    _SCORED_CACHE = _cache_pkl(f'scored_obs_rank_{_bt}')
    if not FORCE_REBUILD and _SCORED_CACHE.exists():
        with open(_SCORED_CACHE, 'rb') as f:
            SCORED_BY_BACKEND[_bt], OBS_RANK_BY_BACKEND[_bt] = pickle.load(f)
        print(f"  [cache] loaded {_bt} SCORED/OBS_RANK from {_SCORED_CACHE}")
    else:
        print(f"\n=== Scoring with {_bt.upper()} backend ===")
        _sc = {}; _or = {}
        _mod = MODELS_BY_BACKEND[_bt]
        for line_name, placements in ALL_PLACEMENTS.items():
            ranked, obs_rank = score_streaming(placements, _mod.get(line_name), line_name)
            _sc[line_name] = ranked
            _or[line_name] = obs_rank
        SCORED_BY_BACKEND[_bt] = _sc
        OBS_RANK_BY_BACKEND[_bt] = _or
        with open(_SCORED_CACHE, 'wb') as f:
            pickle.dump((_sc, _or), f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  cached {_bt} SCORED/OBS_RANK to {_SCORED_CACHE} "
              f"({_SCORED_CACHE.stat().st_size / 1024**2:.1f} MB)")

# Classifier + ranker ensemble: for each (DOT, carrier, state) pair the two
# backends scored, average their per-DOT ranks (rank-average blending — robust
# to the fact that classifier returns probabilities ∈ [0,1] while ranker
# returns unbounded raw margins). Re-sort within each DOT to produce an
# ensemble ranking. This is backend-independent and goes into SCORED_BY_BACKEND
# under the key 'ensemble' alongside classifier/ranker.
if {'classifier', 'ranker'}.issubset(set(SCORED_BY_BACKEND.keys())):
    _ENS_CACHE = _cache_pkl('scored_obs_rank_ensemble')
    if not FORCE_REBUILD and _ENS_CACHE.exists():
        with open(_ENS_CACHE, 'rb') as f:
            SCORED_BY_BACKEND['ensemble'], OBS_RANK_BY_BACKEND['ensemble'] = pickle.load(f)
        print(f"  [cache] loaded ensemble SCORED/OBS_RANK from {_ENS_CACHE}")
    else:
        print("\n=== Building ENSEMBLE (classifier + ranker, rank-average) ===")
        _ens_sc = {}; _ens_or = {}
        for line_name in ALL_PLACEMENTS.keys():
            c_sc = SCORED_BY_BACKEND['classifier'].get(line_name, pd.DataFrame())
            r_sc = SCORED_BY_BACKEND['ranker'].get(line_name, pd.DataFrame())
            if len(c_sc) == 0 or len(r_sc) == 0:
                _ens_sc[line_name] = pd.DataFrame()
                _ens_or[line_name] = pd.DataFrame()
                continue
            merged = c_sc[['usdot','carrier_group','state','line','rank','pool_size']].merge(
                r_sc[['usdot','carrier_group','state','rank']],
                on=['usdot','carrier_group','state'],
                suffixes=('_cls','_rnk'),
                how='inner',
            )
            merged['avg_rank'] = (merged['rank_cls'] + merged['rank_rnk']) / 2.0
            # Deterministic tie-break on carrier_group
            merged = merged.sort_values(['usdot','avg_rank','carrier_group'],
                                        ascending=[True, True, True]).reset_index(drop=True)
            merged['rank'] = (merged.groupby('usdot').cumcount() + 1).astype('int32')
            # fit_score for the ensemble = inverse of the avg_rank (lower rank = higher score)
            # Normalise within DOT to [0, 1] so downstream calibration / deliverable
            # formulas that expect a bounded score still work.
            _max_rank = merged.groupby('usdot')['avg_rank'].transform('max')
            merged['fit_score'] = (1.0 - (merged['avg_rank'] - 1) / _max_rank.clip(lower=1)).astype('float32')
            _ens_sc[line_name] = merged[['usdot','carrier_group','state','fit_score','rank','pool_size','line']].copy()

            obs_pairs = ALL_PLACEMENTS[line_name][
                ALL_PLACEMENTS[line_name]['line'] == line_name
            ][['usdot','carrier_group','state']]
            obs = obs_pairs.merge(
                _ens_sc[line_name][['usdot','carrier_group','rank','fit_score','pool_size']],
                on=['usdot','carrier_group'], how='inner'
            ).rename(columns={'rank':'observed_rank'})
            obs['line'] = line_name
            _ens_or[line_name] = obs

        SCORED_BY_BACKEND['ensemble'] = _ens_sc
        OBS_RANK_BY_BACKEND['ensemble'] = _ens_or
        with open(_ENS_CACHE, 'wb') as f:
            pickle.dump((_ens_sc, _ens_or), f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  cached ensemble SCORED/OBS_RANK to {_ENS_CACHE} "
              f"({_ENS_CACHE.stat().st_size / 1024**2:.1f} MB)")

# Default aliases for downstream cells (deliverable, feature-importance, etc.)
SCORED   = SCORED_BY_BACKEND['classifier']
OBS_RANK = OBS_RANK_BY_BACKEND['classifier']
MODEL_BACKEND_TYPE = 'classifier'

print("\n" + "=" * 70)
print("HONEST EVALUATION - model vs baselines on POST-cutoff holdout DOTs")
print("=" * 70)
print(f"(Holdout = DOTs with latest_eff_date >= T_SPLIT = {T_SPLIT.date()})")

def _restrict_to_holdout(obs_rank, placements, line_label):
    """Return (obs_rank_holdout, placements_holdout) — both filtered to DOTs
    whose latest event in this line is at or after T_SPLIT. These are the
    DOTs the model never saw in training, so metrics on them are honest OOS."""
    _line_dates = LINE_LATEST_EFF[LINE_LATEST_EFF['line'] == line_label][['usdot','latest_eff_date']]
    _holdout_dots = set(_line_dates.loc[_line_dates['latest_eff_date'] >= T_SPLIT, 'usdot'].astype('int64'))
    obs_h = obs_rank[obs_rank['usdot'].isin(_holdout_dots)].copy()
    pl_h  = placements[placements['usdot'].isin(_holdout_dots)].copy()
    return obs_h, pl_h, len(_holdout_dots)

_eval_backends = list(BACKEND_TYPES) + (['ensemble'] if 'ensemble' in OBS_RANK_BY_BACKEND else [])
for _bt in _eval_backends:
    _or = OBS_RANK_BY_BACKEND[_bt]
    print(f"\n### Backend: {_bt.upper()} ###")
    for line_name, placements in ALL_PLACEMENTS.items():
        obs_rank_full = _or[line_name]
        if len(obs_rank_full) == 0:
            print(f"\n--- {line_name.upper()} --- (skipped: no scored placements)")
            continue
        obs_rank, pl_h, n_holdout = _restrict_to_holdout(obs_rank_full, placements, line_name)
        if len(obs_rank) == 0:
            print(f"\n--- {line_name.upper()} --- (skipped: no holdout DOTs scored, n_holdout_universe={n_holdout})")
            continue
        print(f"\n--- {line_name.upper()} ---  (holdout n = {len(obs_rank):,} of {n_holdout:,} total post-cutoff)")
        m_m, b1_m, b2_m = eval_against_baselines(pl_h, obs_rank, line_name)
        summary = pd.DataFrame([
            {'approach': 'model',         **m_m},
            {'approach': 'B1 random',     **b1_m},
            {'approach': 'B2 popularity', **b2_m},
        ])
        print(summary.to_string(index=False))

# BIPD segmented metrics per backend — this is the critical view for
# comparing backends on where it matters (B4/B5 mega-fleets concentrate
# revenue; higher classifier top-5 overall may still lose to ranker on
# fleet-band segments).
print("\n" + "=" * 70)
print("BIPD - segmented metrics (per backend)")
print("=" * 70)

# Segmentation also computed on holdout DOTs only (honest metrics by state+fleet)
_seg_backends = list(BACKEND_TYPES) + (['ensemble'] if 'ensemble' in OBS_RANK_BY_BACKEND else [])
_bipd_holdout_dots = set(LINE_LATEST_EFF.loc[
    (LINE_LATEST_EFF['line'] == 'bipd') & (LINE_LATEST_EFF['latest_eff_date'] >= T_SPLIT),
    'usdot'
].astype('int64'))
for _bt in _seg_backends:
    _or = OBS_RANK_BY_BACKEND[_bt]
    if len(_or.get('bipd', pd.DataFrame())) == 0:
        continue
    _bipd_h = _or['bipd'][_or['bipd']['usdot'].isin(_bipd_holdout_dots)]
    if len(_bipd_h) == 0: continue
    print(f"\n### Backend: {_bt.upper()}  (holdout n = {len(_bipd_h):,}) ###")
    seg = _bipd_h.merge(dot_df[['usdot','fleet_band']], on='usdot', how='left')

    print("\nBy state (top 10 by DOT count):")
    by_state = seg.groupby('state').agg(
        n=('usdot','count'),
        mrr=('observed_rank', lambda s: (1.0/s).mean()),
        top5=('observed_rank', lambda s: (s <= 5).mean()),
    ).sort_values('n', ascending=False).head(10)
    print(by_state.round(3).to_string())

    print("\nBy fleet band:")
    by_band = seg.groupby('fleet_band').agg(
        n=('usdot','count'),
        mrr=('observed_rank', lambda s: (1.0/s).mean()),
        top5=('observed_rank', lambda s: (s <= 5).mean()),
    ).sort_index()
    print(by_band.round(3).to_string())

# Side-by-side fleet-band delta table — classifier vs ranker on the key
# revenue-concentrated bands. +ve delta = ranker wins that band.
if all(len(OBS_RANK_BY_BACKEND[_bt].get('bipd', pd.DataFrame())) > 0 for _bt in _seg_backends):
    print("\n" + "-" * 60)
    print(f"BIPD fleet-band head-to-head on HOLDOUT  ({', '.join(_seg_backends)})")
    print("-" * 60)
    _rows = []
    for _bt in _seg_backends:
        _seg_src = OBS_RANK_BY_BACKEND[_bt]['bipd']
        _seg_src = _seg_src[_seg_src['usdot'].isin(_bipd_holdout_dots)]
        _seg = _seg_src.merge(
            dot_df[['usdot','fleet_band']], on='usdot', how='left')
        _agg = _seg.groupby('fleet_band')['observed_rank'].apply(
            lambda s: pd.Series({
                'top5_'+_bt:  (s <= 5).mean(),
                'top10_'+_bt: (s <= 10).mean(),
                'mrr_'+_bt:   (1.0/s).mean(),
                'n_'+_bt:     len(s),
            }))
        _agg = _agg.unstack() if isinstance(_agg.index, pd.MultiIndex) else _agg
        _rows.append(_agg)
    _cmp = pd.concat(_rows, axis=1)
    _baseline = 'classifier'
    for metric in ['top5','top10','mrr']:
        for _bt in _seg_backends:
            if _bt == _baseline: continue
            _cmp[f'd_{metric}_{_bt}_vs_cls'] = _cmp[f'{metric}_{_bt}'] - _cmp[f'{metric}_{_baseline}']
    _cols = [c for c in _cmp.columns if c.startswith('d_') or c.startswith('n_')]
    print(_cmp[_cols].round(4).to_string())

_log("Part 5A complete")


[t=3692.4s] starting Part 5A

SCORING (streaming per state, 4 lines)

=== Scoring with CLASSIFIER backend ===
  [bipd] processed 10/60 states (5,236,544 candidate pairs seen)
  [bipd] processed 20/60 states (9,559,978 candidate pairs seen)
  [bipd] processed 30/60 states (10,908,609 candidate pairs seen)
  [bipd] processed 40/60 states (12,341,130 candidate pairs seen)
  [bipd] processed 50/60 states (15,240,545 candidate pairs seen)
  [bipd] processed 60/60 states (23,214,688 candidate pairs seen)
  [mem] [bipd] after streaming scoring: 5.28 GB
  [cargo] processed 10/20 states (50,250 candidate pairs seen)
  [cargo] processed 20/20 states (113,728 candidate pairs seen)
  [mem] [cargo] after streaming scoring: 3.89 GB
  [wc] processed 10/40 states (158,714 candidate pairs seen)
  [wc] processed 20/40 states (240,790 candidate pairs seen)
  [wc] processed 30/40 states (2,306,501 candidate pairs seen)
  [wc] processed 40/40 states (6,113,329 candidate pairs seen)
  [mem] [wc] after strea

## Phase 6 — Weighted-tier Alternative Model

Per-line tenure thresholds (bipd=1.5y, wc=1y, cargo=0.5y, surety_bond=0.5y) label observed placements as strong/medium/weak positives and attach sample weights (1.0 / 0.6 / 0.3). Trained for interpretability + methodological robustness check, then compared head-to-head with the base model. Base wins on all metrics including the long-tenure proxy — see Part 6B.

### Part 6A — Weighted-tier retrain (per-line, line-specific tenure)

The task framing is "find DOTs appropriately placed with the right
carrier" — not just typical placements. Weighted-strict labelling
tiers the positives by confidence:
  Strong (weight 1.0): clean + tenure >= per-line threshold + cancel_rate OK
  Medium (weight 0.6): clean + tenure in [0.5y, threshold)
  Weak   (weight 0.3): clean + tenure < 0.5y OR tenure unknown
Model learns primarily from "appropriate" placements but still sees younger
DOTs. Per-line thresholds in STRICT_TENURE_MIN_PER_LINE.

In [18]:
_log("starting Part 6A — weighted-tier retrain")

# Line-specific tenure: for each (usdot, line, carrier_group), compute
# years since the earliest event (within the 24m window) with that carrier.
# Used as the per-line tenure signal; capped at 24m by the recent_events
# window — placements older than that get tenure = ~2.0y, which is fine for
# the >= threshold check.
_LINE_TENURE_CACHE = _cache_pkl('line_tenure')
if not FORCE_REBUILD and _LINE_TENURE_CACHE.exists():
    with open(_LINE_TENURE_CACHE, 'rb') as f:
        LINE_TENURE_DF = pickle.load(f)
    print(f"  [cache] loaded line tenure ({len(LINE_TENURE_DF):,} rows)")
else:
    LINE_TENURE_DF = con.execute(f"""
        WITH latest_per_dot_line AS (
            SELECT usdot, line, carrier_group, effective_date,
                   ROW_NUMBER() OVER (
                       PARTITION BY usdot, line
                       ORDER BY effective_date DESC NULLS LAST
                   ) AS rn
            FROM recent_events
        ),
        current_pair AS (
            SELECT usdot, line, carrier_group FROM latest_per_dot_line WHERE rn = 1
        ),
        earliest_with_current AS (
            SELECT re.usdot, re.line, re.carrier_group,
                   MIN(re.effective_date) AS start_eff_date,
                   MAX(re.effective_date) AS latest_eff_date
            FROM recent_events re
            JOIN current_pair cp USING (usdot, line, carrier_group)
            GROUP BY re.usdot, re.line, re.carrier_group
        )
        SELECT usdot, line, carrier_group,
               CAST(DATE_DIFF('day', start_eff_date, {_VINTAGE_SQL}) AS DOUBLE) / 365.25 AS line_tenure_years,
               latest_eff_date
        FROM earliest_with_current
    """).df()
    LINE_TENURE_DF['usdot'] = LINE_TENURE_DF['usdot'].astype('int64')
    LINE_TENURE_DF['line_tenure_years'] = LINE_TENURE_DF['line_tenure_years'].astype('float32')
    with open(_LINE_TENURE_CACHE, 'wb') as f:
        pickle.dump(LINE_TENURE_DF, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  computed + cached line tenure: {len(LINE_TENURE_DF):,} (usdot, line, carrier) rows")


def build_training_weighted(placements, line_label):
    tenure_min = STRICT_TENURE_MIN_PER_LINE.get(line_label, 1.0)
    cancel_max = STRICT_CANCEL_RATE_MAX

    # Honest-training filter: only pre-cutoff DOTs (consistent with base).
    placements = _filter_to_pre_cutoff(placements, line_label)
    modelable = carrier_state_agg[carrier_state_agg['line'] == line_label][['carrier_group','state']]
    all_obs = placements.merge(modelable, on=['carrier_group','state'], how='inner').copy()
    all_obs = all_obs.merge(
        dot_df[['usdot','is_residual_market','is_cancellation_pending','insurance_cancel_rate']],
        on='usdot', how='left'
    )
    line_ten = LINE_TENURE_DF[LINE_TENURE_DF['line'] == line_label][['usdot','line_tenure_years']]
    all_obs = all_obs.merge(line_ten, on='usdot', how='left')

    clean = (all_obs['is_residual_market'] == 0) & (all_obs['is_cancellation_pending'] == 0)
    cancel_ok = all_obs['insurance_cancel_rate'].fillna(0) < cancel_max
    tenure_filled = all_obs['line_tenure_years'].fillna(0)
    tenure_strong = tenure_filled >= tenure_min
    tenure_medium = (tenure_filled >= 0.5) & ~tenure_strong

    pos = all_obs.loc[clean, ['usdot','carrier_group','state']].copy()
    pos['y'] = np.int8(1)
    w_arr = np.full(len(all_obs), POS_WEIGHT_WEAK, dtype='float32')
    w_arr[(cancel_ok & tenure_medium).values] = POS_WEIGHT_MEDIUM
    w_arr[(cancel_ok & tenure_strong).values] = POS_WEIGHT_STRONG
    pos['_weight'] = w_arr[clean.values]

    hard_rm, hard_rev, hard_rej = _build_hard_negatives(all_obs, line_label, modelable)
    hard_rm['_weight']  = np.float32(NEG_WEIGHT_HARD)
    hard_rev['_weight'] = np.float32(NEG_WEIGHT_HARD)
    hard_rej['_weight'] = np.float32(NEG_WEIGHT_HARD)

    soft = _sample_soft_negatives(pos, modelable)
    soft = _dedupe_and_cap_soft(soft, pos, hard_rm, hard_rev, hard_rej)
    soft['_weight'] = np.float32(NEG_WEIGHT_SOFT)

    train = pd.concat([pos, hard_rm, hard_rev, hard_rej, soft], ignore_index=True)
    train = _assemble_training_frame(train, line_label)

    n_strong = int((cancel_ok & tenure_strong & clean).sum())
    n_medium = int((cancel_ok & tenure_medium & clean).sum())
    n_weak = int(clean.sum()) - n_strong - n_medium
    return train, n_strong, n_medium, n_weak, len(hard_rm) + len(hard_rev) + len(hard_rej), len(soft)


def train_weighted_model(placements, line_label, n_cv_splits=3):
    print(f"\n--- Training {line_label.upper()} (WEIGHTED tiers; strong_threshold={STRICT_TENURE_MIN_PER_LINE.get(line_label, 1.0)}y) ---")
    _log_mem(f"[{line_label}-w] before build")
    train_df, n_strong, n_medium, n_weak, n_hard, n_soft = build_training_weighted(placements, line_label)
    n_pos = n_strong + n_medium + n_weak
    if n_pos < 50:
        print(f"  insufficient positives ({n_pos}); skipping")
        return None
    print(f"  training rows: {len(train_df):,}  "
          f"(strong={n_strong:,} medium={n_medium:,} weak={n_weak:,} hard_neg={n_hard:,} soft_neg={n_soft:,})")
    _log_mem(f"[{line_label}-w] after build")

    final = _fit_cv_and_refit(train_df, line_label, n_cv_splits=n_cv_splits, use_weights=True)
    del train_df; gc.collect()
    _log_mem(f"[{line_label}-w] after refit")
    return final


MODELS_WEIGHTED_BY_BACKEND = {}
for _bt in BACKEND_TYPES:
    MODEL_BACKEND_TYPE = _bt
    _MODELS_WEIGHTED_CACHE = _cache_pkl(f'models_weighted_{_bt}')
    if not FORCE_REBUILD and _MODELS_WEIGHTED_CACHE.exists():
        MODELS_WEIGHTED_BY_BACKEND[_bt] = joblib.load(_MODELS_WEIGHTED_CACHE)
        _n = sum(1 for m in MODELS_WEIGHTED_BY_BACKEND[_bt].values() if m is not None)
        print(f"  [cache] loaded {_n}/{len(MODELS_WEIGHTED_BY_BACKEND[_bt])} {_bt} weighted models from {_MODELS_WEIGHTED_CACHE}")
    else:
        print(f"\n=== Training WEIGHTED-TIER / {_bt.upper()} ===")
        _m = {}
        for line_name, p in ALL_PLACEMENTS.items():
            _m[line_name] = train_weighted_model(p, line_name)
        MODELS_WEIGHTED_BY_BACKEND[_bt] = _m
        joblib.dump(_m, _MODELS_WEIGHTED_CACHE)
        print(f"  cached {_bt} weighted models to {_MODELS_WEIGHTED_CACHE} "
              f"({_MODELS_WEIGHTED_CACHE.stat().st_size / 1024**2:.1f} MB)")

MODELS_WEIGHTED = MODELS_WEIGHTED_BY_BACKEND['classifier']
MODEL_BACKEND_TYPE = 'classifier'

_log("Part 6A complete")


[t=5380.1s] starting Part 6A — weighted-tier retrain
  computed + cached line tenure: 463,896 (usdot, line, carrier) rows

=== Training WEIGHTED-TIER / CLASSIFIER ===

--- Training BIPD (WEIGHTED tiers; strong_threshold=1.5y) ---
  [mem] [bipd-w] before build: 4.49 GB
  training rows: 782,676  (strong=35,489 medium=67,140 weak=93,301 hard_neg=821 soft_neg=585,925)
  [mem] [bipd-w] after build: 5.44 GB
  CV ROC-AUC: 0.9583 +/- 0.0004 | PR-AUC: 0.9028 +/- 0.0010
  [mem] [bipd-w] after refit: 4.29 GB

--- Training CARGO (WEIGHTED tiers; strong_threshold=0.5y) ---
  [mem] [cargo-w] before build: 4.29 GB
  training rows: 6,151  (strong=769 medium=0 weak=827 hard_neg=2 soft_neg=4,553)
  [mem] [cargo-w] after build: 4.37 GB
  CV ROC-AUC: 0.9934 +/- 0.0014 | PR-AUC: 0.9750 +/- 0.0082
  [mem] [cargo-w] after refit: 4.38 GB

--- Training WC (WEIGHTED tiers; strong_threshold=1.0y) ---
  [mem] [wc-w] before build: 4.38 GB
  training rows: 216,781  (strong=40,524 medium=9,981 weak=3,948 hard_neg=28

### Part 6B — Weighted-tier scoring + side-by-side comparison

Scores the full universe with MODELS_WEIGHTED, compares against the
current base model on three axes:
  1. Standard metrics across all observed placements
  2. Metrics restricted to long-tenure placements (approximate ground
     truth for "appropriate" — long tenure means the placement persisted)
  3. Top-5 recommendation overlap per DOT

In [19]:
_log("starting Part 6B — weighted scoring + comparison")

SCORED_WEIGHTED_BY_BACKEND = {}
OBS_RANK_WEIGHTED_BY_BACKEND = {}
for _bt in BACKEND_TYPES:
    MODEL_BACKEND_TYPE = _bt
    _SCORED_WEIGHTED_CACHE = _cache_pkl(f'scored_obs_rank_weighted_{_bt}')
    if not FORCE_REBUILD and _SCORED_WEIGHTED_CACHE.exists():
        with open(_SCORED_WEIGHTED_CACHE, 'rb') as f:
            SCORED_WEIGHTED_BY_BACKEND[_bt], OBS_RANK_WEIGHTED_BY_BACKEND[_bt] = pickle.load(f)
        print(f"  [cache] loaded {_bt} SCORED_WEIGHTED from {_SCORED_WEIGHTED_CACHE}")
    else:
        print(f"\n=== Scoring WEIGHTED / {_bt.upper()} ===")
        _sc = {}; _or = {}
        _mod = MODELS_WEIGHTED_BY_BACKEND[_bt]
        for line_name, placements in ALL_PLACEMENTS.items():
            ranked, obs_rank = score_streaming(placements, _mod.get(line_name), line_name)
            _sc[line_name] = ranked
            _or[line_name] = obs_rank
        SCORED_WEIGHTED_BY_BACKEND[_bt] = _sc
        OBS_RANK_WEIGHTED_BY_BACKEND[_bt] = _or
        with open(_SCORED_WEIGHTED_CACHE, 'wb') as f:
            pickle.dump((_sc, _or), f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  cached {_bt} SCORED_WEIGHTED to {_SCORED_WEIGHTED_CACHE} "
              f"({_SCORED_WEIGHTED_CACHE.stat().st_size / 1024**2:.1f} MB)")

SCORED_WEIGHTED   = SCORED_WEIGHTED_BY_BACKEND['classifier']
OBS_RANK_WEIGHTED = OBS_RANK_WEIGHTED_BY_BACKEND['classifier']
MODEL_BACKEND_TYPE = 'classifier'

print()
print("=" * 70)
print("COMPARISON: BASE (current) vs WEIGHTED tiers")
print("=" * 70)

# --- 1. Standard metrics across all observed placements ---
print(f"\n{'line':<13s} {'metric':<22s} {'current':>10s} {'weighted':>10s} {'delta':>10s}")
print("-" * 70)
for line_name in ALL_PLACEMENTS.keys():
    cur = OBS_RANK.get(line_name, pd.DataFrame())
    wgt = OBS_RANK_WEIGHTED.get(line_name, pd.DataFrame())
    if len(cur) == 0 or len(wgt) == 0: continue
    for metric_name, metric_fn in [
        ('top-1 hit rate',  lambda df: (df['observed_rank'] <= 1).mean()),
        ('top-5 hit rate',  lambda df: (df['observed_rank'] <= 5).mean()),
        ('top-10 hit rate', lambda df: (df['observed_rank'] <= 10).mean()),
        ('MRR',             lambda df: (1.0 / df['observed_rank']).mean()),
    ]:
        cur_v = float(metric_fn(cur)); wgt_v = float(metric_fn(wgt))
        print(f"{line_name:<13s} {metric_name:<22s} {cur_v:>10.4f} {wgt_v:>10.4f} {wgt_v - cur_v:>+10.4f}")
    print()

# --- 2. Metrics on long-tenure placements (approximate ground truth) ---
print()
print("Metrics restricted to line-tenure >= 1.5y placements (ground-truth proxy)")
print("Note: line_tenure_years is capped at ~2.0 by the 24m recent_events window,")
print("so >= 1.5 captures the oldest ~25-50% of each line's placements.")
print(f"{'line':<13s} {'metric':<22s} {'current':>10s} {'weighted':>10s} {'delta':>10s}")
print("-" * 70)
for line_name in ALL_PLACEMENTS.keys():
    cur = OBS_RANK.get(line_name, pd.DataFrame())
    wgt = OBS_RANK_WEIGHTED.get(line_name, pd.DataFrame())
    if len(cur) == 0 or len(wgt) == 0: continue
    _tenure = LINE_TENURE_DF[LINE_TENURE_DF['line'] == line_name][['usdot','line_tenure_years']]
    cur = cur.merge(_tenure, on='usdot', how='left')
    wgt = wgt.merge(_tenure, on='usdot', how='left')
    # recent_events is windowed to 24m so line_tenure_years caps at ~2.0.
    # 1.5y = placements that started >= 18 months before vintage.
    cur_long = cur[cur['line_tenure_years'].fillna(0) >= 1.5]
    wgt_long = wgt[wgt['line_tenure_years'].fillna(0) >= 1.5]
    if len(cur_long) < 50 or len(wgt_long) < 50:
        print(f"{line_name:<13s}  (insufficient long-tenure: cur={len(cur_long)}, weighted={len(wgt_long)})")
        continue
    for metric_name, metric_fn in [
        ('top-5 hit (tenure>=1.5y)',  lambda df: (df['observed_rank'] <= 5).mean()),
        ('top-10 hit (tenure>=1.5y)', lambda df: (df['observed_rank'] <= 10).mean()),
        ('MRR (tenure>=1.5y)',        lambda df: (1.0 / df['observed_rank']).mean()),
    ]:
        cur_v = float(metric_fn(cur_long)); wgt_v = float(metric_fn(wgt_long))
        print(f"{line_name:<13s} {metric_name:<22s} {cur_v:>10.4f} {wgt_v:>10.4f} {wgt_v - cur_v:>+10.4f}")
    print()

# --- 3. Top-5 recommendation overlap ---
print()
print("Top-5 recommendation overlap per DOT (how different are the models?)")
print(f"{'line':<13s} {'avg overlap /5':>18s} {'identical top-5':>18s} {'n_dots':>10s}")
print("-" * 60)
for line_name in ALL_PLACEMENTS.keys():
    cur_scored = SCORED.get(line_name, pd.DataFrame())
    wgt_scored = SCORED_WEIGHTED.get(line_name, pd.DataFrame())
    if len(cur_scored) == 0 or len(wgt_scored) == 0: continue
    cur_top5 = cur_scored[cur_scored['rank'] <= 5]
    wgt_top5 = wgt_scored[wgt_scored['rank'] <= 5]
    cur_sets = cur_top5.groupby(['usdot','state'])['carrier_group'].apply(set)
    wgt_sets = wgt_top5.groupby(['usdot','state'])['carrier_group'].apply(set)
    merged = pd.concat([cur_sets.rename('cur'), wgt_sets.rename('wgt')], axis=1).dropna()
    if len(merged) == 0: continue
    overlaps = merged.apply(lambda row: len(row['cur'] & row['wgt']), axis=1)
    identical = (overlaps == 5).mean()
    print(f"{line_name:<13s} {overlaps.mean():>14.2f}/5 {identical*100:>15.1f}% {len(merged):>10,}")

print()
print("Interpretation:")
print("  - Delta on 'tenure >= 2y' metrics is the honest comparison.")
print("    Weighted > Current there = better at identifying appropriate placements.")
print("  - Low top-5 overlap = models flag different carriers; big methodology choice.")
print("    High overlap = models mostly agree; pick whichever has better long-tenure numbers.")

_log("Part 6B complete")


[t=5679.8s] starting Part 6B — weighted scoring + comparison

=== Scoring WEIGHTED / CLASSIFIER ===
  [bipd] processed 10/60 states (5,236,544 candidate pairs seen)
  [bipd] processed 20/60 states (9,559,978 candidate pairs seen)
  [bipd] processed 30/60 states (10,908,609 candidate pairs seen)
  [bipd] processed 40/60 states (12,341,130 candidate pairs seen)
  [bipd] processed 50/60 states (15,240,545 candidate pairs seen)
  [bipd] processed 60/60 states (23,214,688 candidate pairs seen)
  [mem] [bipd] after streaming scoring: 4.62 GB
  [cargo] processed 10/20 states (50,250 candidate pairs seen)
  [cargo] processed 20/20 states (113,728 candidate pairs seen)
  [mem] [cargo] after streaming scoring: 3.28 GB
  [wc] processed 10/40 states (158,714 candidate pairs seen)
  [wc] processed 20/40 states (240,790 candidate pairs seen)
  [wc] processed 30/40 states (2,306,501 candidate pairs seen)
  [wc] processed 40/40 states (6,113,329 candidate pairs seen)
  [mem] [wc] after streaming scori

## Phase 7 — Validation Suite

Deep validation diagnostics evaluating all trained model variants side-by-side: feature importance (SHAP), error analysis, known-good/known-bad audit, fit-score calibration, pool-size scatter, synthetic-DOT sanity, approximate-temporal drift check.

Positioned after Phase 6 (weighted retrain) so every diagnostic can iterate over the full set — classifier, ranker, and ensemble from Phase 5, with the weighted variants from Phase 6 available where applicable.

Note: the pipeline's "temporal split" is **implicit everywhere** — all training in Phase 4 and Phase 6 uses pre-cutoff data only, and the headline metrics in Phase 5 are on the post-cutoff holdout. There is no separate "temporal retrain" phase because the temporal setup IS the baseline pipeline.

### Part 7A — Feature importance per model

LightGBM native feature_importances_ for each trained model.
Confirms the model is learning carrier appetite (carr_* features
should dominate top-20) rather than cross-DOT patterns (dot_num
features dominating would mean AUC was inflated by cross-DOT noise).

In [21]:
_log("starting Part 7A — feature importance")

all_feat_names = DOT_NUM_FEATURES + CARRIER_FEATURES + DOT_CAT_FEATURES
n_num  = len(DOT_NUM_FEATURES)
n_carr = len(CARRIER_FEATURES)
n_cat  = len(DOT_CAT_FEATURES)
kind_labels = ['dot_num'] * n_num + ['carrier'] * n_carr + ['dot_cat'] * n_cat

print(f"Feature list: {n_num} dot_num + {n_carr} carrier + {n_cat} dot_cat = {len(all_feat_names)} total")
print("=" * 70)

for _bt in BACKEND_TYPES:
    print(f"\n{'#' * 70}")
    print(f"# Backend: {_bt.upper()}")
    print(f"{'#' * 70}")
    for line_name, model in MODELS_BY_BACKEND[_bt].items():
        if model is None:
            print(f"\n--- {line_name.upper()} ---  (no model)")
            continue
        print(f"\n--- {line_name.upper()} ---")
        imp = model.feature_importances_
        if len(imp) != len(all_feat_names):
            print(f"  WARNING: feature count mismatch ({len(imp)} vs {len(all_feat_names)}); skipping")
            continue
        imp_df = pd.DataFrame({'feature': all_feat_names, 'importance': imp, 'kind': kind_labels})
        print("  Top 20 features by importance:")
        top20 = imp_df.sort_values('importance', ascending=False).head(20).reset_index(drop=True)
        print(top20.to_string(index=False))
        print("\n  Importance share by kind:")
        share = imp_df.groupby('kind')['importance'].sum()
        share_pct = (100 * share / share.sum()).round(1)
        for k, pct in share_pct.items():
            print(f"    {k:<10s}  {pct:5.1f}%")

_log("Part 7A complete")


[t=7075.7s] starting Part 7A — feature importance
Feature list: 133 dot_num + 36 carrier + 41 dot_cat = 210 total

######################################################################
# Backend: CLASSIFIER
######################################################################

--- BIPD ---
  Top 20 features by importance:
                     feature  importance    kind
           carr_n_placements         872 carrier
                       state         213 dot_cat
               carr_mean_oos         212 carrier
         carr_pct_intermodal         197 carrier
    svc_voluntary_exit_ratio         192 carrier
         carr_pct_passengers         191 carrier
          carr_pct_livestock         190 carrier
               carr_pct_meat         188 carrier
svc_short_tenure_cancel_rate         168 carrier
           carr_pct_oilfield         158 carrier
       carr_mean_intra_short         156 carrier
         carr_pct_genfreight         150 carrier
        svc_mean_tenure_days         

### Part 7B — Error analysis: where does the model miss?

Distribution of observed_rank per line (where did the actual carrier
land?) and a sample of hard cases (actual carrier ranked > 10) to
inspect what the model picked instead.

In [22]:
_log("starting Part 7B — error analysis")

RANK_BUCKETS = [1, 5, 10, 20, 50, 100]

print("Observed-rank distribution per line (CDF), per backend — HOLDOUT ONLY")
print("=" * 70)

def _holdout_dots_for(line_name):
    return set(LINE_LATEST_EFF.loc[
        (LINE_LATEST_EFF['line'] == line_name) & (LINE_LATEST_EFF['latest_eff_date'] >= T_SPLIT),
        'usdot'
    ].astype('int64'))

_eval_backends = list(BACKEND_TYPES) + (['ensemble'] if 'ensemble' in OBS_RANK_BY_BACKEND else [])
for _bt in _eval_backends:
    print(f"\n### Backend: {_bt.upper()} ###")
    for line_name, obs_rank_full in OBS_RANK_BY_BACKEND[_bt].items():
        if obs_rank_full is None or len(obs_rank_full) == 0: continue
        obs_rank = obs_rank_full[obs_rank_full['usdot'].isin(_holdout_dots_for(line_name))]
        if len(obs_rank) == 0: continue
        print(f"\n--- {line_name.upper()} (n = {len(obs_rank):,}) ---")
        med = obs_rank['observed_rank'].median()
        mean = obs_rank['observed_rank'].mean()
        print(f"  median rank: {med:.0f}  |  mean rank: {mean:.1f}")
        for b in RANK_BUCKETS:
            pct = (obs_rank['observed_rank'] <= b).mean() * 100
            print(f"  rank <= {b:3d}:  {pct:5.1f}%")
        pct_miss = (obs_rank['observed_rank'] > 10).mean() * 100
        print(f"  rank  > 10:  {pct_miss:5.1f}%  (hard cases)")

print("\n")
print("BIPD hard cases (actual carrier ranked > 10) — sample of 15 (classifier only)")
print("=" * 70)
# Hard-case drill-down is anchored to the deliverable backend (classifier) for
# brevity. Ranker's hard-cases are typically similar — if divergence is of
# interest, swap OBS_RANK/SCORED aliases below with OBS_RANK_BY_BACKEND['ranker'].
# Hard cases on holdout only (honest "where does the model miss")
bipd_obs = OBS_RANK.get('bipd', pd.DataFrame())
if len(bipd_obs) > 0:
    bipd_obs = bipd_obs[bipd_obs['usdot'].isin(_holdout_dots_for('bipd'))]
bipd_scored = SCORED.get('bipd', pd.DataFrame())
if len(bipd_obs) > 0 and len(bipd_scored) > 0:
    hard = bipd_obs[bipd_obs['observed_rank'] > 10].sort_values('observed_rank', ascending=False).head(15)
    _dot_idx = dot_df.set_index('usdot')
    for _, row in hard.iterrows():
        dot = int(row['usdot'])
        if dot not in _dot_idx.index: continue
        info = _dot_idx.loc[dot]
        state = info.get('state', '?')
        pu = info.get('power_units', None)
        pu_str = f"{int(pu)}" if pu is not None and not pd.isna(pu) else "NaN"
        print(f"\nDOT {dot} | observed rank {int(row['observed_rank'])}/{int(row.get('pool_size',0))} "
              f"| actual = {row['carrier_group']}")
        print(f"  state={state}  power_units={pu_str}  fleet_band={info.get('fleet_band','?')}  "
              f"hazmat={info.get('hazmat_bin','?')}  crash_count={info.get('crash_count','?')}")
        top5 = bipd_scored[(bipd_scored['usdot'] == dot) & (bipd_scored['state'] == state)].sort_values('rank').head(5)
        if len(top5) > 0:
            recs = [f"{r['carrier_group']} ({r['fit_score']:.3f})" for _, r in top5.iterrows()]
            print(f"  model top-5: {' | '.join(recs)}")
else:
    print("  (BIPD obs_rank or SCORED is empty — nothing to analyse)")

_log("Part 7B complete")


[t=7078.1s] starting Part 7B — error analysis
Observed-rank distribution per line (CDF), per backend — HOLDOUT ONLY

### Backend: CLASSIFIER ###

--- BIPD (n = 14,764) ---
  median rank: 2  |  mean rank: 6.8
  rank <=   1:   43.8%
  rank <=   5:   71.2%
  rank <=  10:   81.1%
  rank <=  20:   90.5%
  rank <=  50:   98.7%
  rank <= 100:  100.0%
  rank  > 10:   18.9%  (hard cases)

--- CARGO (n = 52) ---
  median rank: 2  |  mean rank: 2.3
  rank <=   1:   44.2%
  rank <=   5:   96.2%
  rank <=  10:  100.0%
  rank <=  20:  100.0%
  rank <=  50:  100.0%
  rank <= 100:  100.0%
  rank  > 10:    0.0%  (hard cases)

--- WC (n = 28,217) ---
  median rank: 1  |  mean rank: 7.4
  rank <=   1:   51.0%
  rank <=   5:   76.7%
  rank <=  10:   85.7%
  rank <=  20:   90.6%
  rank <=  50:   96.3%
  rank <= 100:   99.4%
  rank  > 10:   14.3%  (hard cases)

--- SURETY_BOND (n = 383) ---
  median rank: 3  |  mean rank: 5.6
  rank <=   1:   32.6%
  rank <=   5:   65.3%
  rank <=  10:   77.3%
  rank <=  20

### Part 7C — Known-good / known-bad placement audit

Industry-knowledge spot-check (not derived from the data):
  - MAINSTREAM carriers (voluntary market) should rank HIGH for clean
    voluntary-market DOTs.
  - SPECIALTY/RESIDUAL carriers (non-standard, state funds, fronting
    entities) should rank LOW for the same DOTs — or be pre-filtered
    out entirely by RESIDUAL_MARKET_CARRIER_THRESHOLD.
One-shot falsifiability test: PASS = model learned the market's
structural split; FAIL = model isn't distinguishing appetite buckets
(would suggest systematic feature-engineering or labelling issue).

In [23]:
_log("starting Part 7C — known-good / known-bad audit")

MAINSTREAM_CARRIERS = [
    'PROGRESSIVE GRP',
    'TRAVELERS GRP',
    'BERKSHIRE HATHAWAY GRP',
    'ZURICH INS GRP',
    'CHUBB LTD GRP',
    'CNA INS GRP',
    'LIBERTY MUT GRP',
    'NATIONWIDE CORP GRP',
    'GREAT WEST',
    'CANAL GRP',
    'HARTFORD FIRE & CAS GRP',
    'AMERICAN INTL GRP',
]

SPECIALTY_CARRIERS = [
    # State WC monopolies / residual pools
    'TEXAS MUTUAL INS CO',
    # Texas County Mutuals — non-standard auto market
    'MERCURY COUNTY MUTUAL INSURANCE COMPANY',
    'INFINITY COUNTY MUTUAL INSURANCE COMPANY',
    # Non-standard / E&S / fronting entities
    'PRIME INSURANCE COMPANY',
    'KNIGHT SPECIALTY INSURANCE COMPANY',
    'STAR INSURANCE COMPANY',
    'HOXBRIDGE INSURANCE COMPANY INC',
    'SOUTHLAKE SPECIALTY INSURANCE COMPANY',
    'HIGHLANDER SPECIALTY INSURANCE COMPANY',
    'MS TRANSVERSE SPECIALTY INSURANCE COMPANY',
    'TRISURA',
    'BENCHMARK INSURANCE COMPANY',
    'MSIG SPECIALTY INSURANCE USA INC',
    'ACCREDITED SURETY & CAS CO INC',
]

# Load authoritative label set from the carrier_lookup DuckDB table.
# (Previously referenced as a pandas variable — but carrier_lookup is only a
# DuckDB table; the pandas DF in cell 2A is named distinct_insurers and only
# exists on cache-miss. Query DuckDB directly for cache-safety.)
_known_groups = set(con.execute(
    "SELECT DISTINCT carrier_group FROM carrier_lookup"
).df()['carrier_group'].unique())
_found_mainstream   = [c for c in MAINSTREAM_CARRIERS if c in _known_groups]
_found_specialty    = [c for c in SPECIALTY_CARRIERS  if c in _known_groups]
_missing_mainstream = [c for c in MAINSTREAM_CARRIERS if c not in _known_groups]
_missing_specialty  = [c for c in SPECIALTY_CARRIERS  if c not in _known_groups]

print(f"MAINSTREAM bucket: {len(_found_mainstream)}/{len(MAINSTREAM_CARRIERS)} found in carrier_lookup")
if _missing_mainstream:
    print(f"  missing (not under this exact label): {_missing_mainstream}")
print(f"SPECIALTY  bucket: {len(_found_specialty)}/{len(SPECIALTY_CARRIERS)} found in carrier_lookup")
if _missing_specialty:
    print(f"  missing: {_missing_specialty}")

# Clean (voluntary-market) DOT universe — same filter as training clean positives
clean_dot_ids = dot_df[
    (dot_df['is_residual_market'] == 0)
    & (dot_df['is_cancellation_pending'] == 0)
]['usdot'].unique()
print(f"\nClean (voluntary-market) DOT universe: {len(clean_dot_ids):,} DOTs")

def _holdout_dots_for(line_name):
    return set(LINE_LATEST_EFF.loc[
        (LINE_LATEST_EFF['line'] == line_name) & (LINE_LATEST_EFF['latest_eff_date'] >= T_SPLIT),
        'usdot'
    ].astype('int64'))

_eval_backends = list(BACKEND_TYPES) + (['ensemble'] if 'ensemble' in SCORED_BY_BACKEND else [])
for _bt in _eval_backends:
    print(f"\n### Backend: {_bt.upper()}  (HOLDOUT ONLY) ###")
    header = f"{'LINE':12s} {'MAINSTREAM med rank (n)':>26s} {'SPECIALTY med rank (n)':>26s} {'Gap':>8s} {'Verdict':>10s}"
    print(header)
    print("-" * len(header))
    for line_name, scored_df in SCORED_BY_BACKEND[_bt].items():
        if scored_df is None or len(scored_df) == 0:
            print(f"{line_name:12s}  (no scored output)")
            continue
        _hd = _holdout_dots_for(line_name)
        clean_scored = scored_df[scored_df['usdot'].isin(clean_dot_ids) & scored_df['usdot'].isin(_hd)]
        if len(clean_scored) == 0:
            print(f"{line_name:12s}  (no clean DOTs scored)")
            continue
        seen_in_pool = set(clean_scored['carrier_group'].unique())
        main_in_pool = [c for c in _found_mainstream if c in seen_in_pool]
        spec_in_pool = [c for c in _found_specialty  if c in seen_in_pool]
        spec_prefiltered = [c for c in _found_specialty if c not in seen_in_pool]
        main_ranks = clean_scored[clean_scored['carrier_group'].isin(main_in_pool)]['rank']
        spec_ranks = clean_scored[clean_scored['carrier_group'].isin(spec_in_pool)]['rank']
        main_med = float(main_ranks.median()) if len(main_ranks) else float('nan')
        spec_med = float(spec_ranks.median()) if len(spec_ranks) else float('nan')
        if pd.isna(main_med) or pd.isna(spec_med):
            verdict = "n/a"; gap = float('nan')
        else:
            gap = spec_med - main_med
            verdict = "PASS" if gap > 0 else "FAIL"
        main_str = f"{main_med:>5.0f} ({len(main_ranks):>7,})" if not pd.isna(main_med) else "n/a"
        spec_str = f"{spec_med:>5.0f} ({len(spec_ranks):>7,})" if not pd.isna(spec_med) else "n/a"
        gap_str  = f"{gap:+.0f}" if not pd.isna(gap) else "n/a"
        print(f"{line_name:12s} {main_str:>26s} {spec_str:>26s} {gap_str:>8s} {verdict:>10s}")
        if spec_prefiltered:
            print(f"             pre-filtered from scoring pool "
                  f"(carr_pct_residual_market >= {RESIDUAL_MARKET_CARRIER_THRESHOLD}): {spec_prefiltered}")

print()
print("Interpretation:")
print("  Gap > 0: specialty carriers rank below mainstream for voluntary DOTs — PASS")
print("  Gap < 0: specialty carriers rank above mainstream — red flag, investigate")
print()
print("Note: specialty carriers with >= 50% residual-market event share are already")
print("pre-filtered from the scoring pool in cell 4A, so they don't need to be")
print("down-ranked by the model (they can't be recommended at all). Those are listed")
print("per line above.")

_log("Part 7C complete")


[t=7099.2s] starting Part 7C — known-good / known-bad audit
MAINSTREAM bucket: 12/12 found in carrier_lookup
SPECIALTY  bucket: 13/14 found in carrier_lookup
  missing: ['ACCREDITED SURETY & CAS CO INC']

Clean (voluntary-market) DOT universe: 449,813 DOTs

### Backend: CLASSIFIER  (HOLDOUT ONLY) ###
LINE            MAINSTREAM med rank (n)     SPECIALTY med rank (n)      Gap    Verdict
--------------------------------------------------------------------------------------
bipd                       11 (107,250)               29 ( 72,450)      +18       PASS
             pre-filtered from scoring pool (carr_pct_residual_market >= 0.5): ['TEXAS MUTUAL INS CO', 'MERCURY COUNTY MUTUAL INSURANCE COMPANY', 'INFINITY COUNTY MUTUAL INSURANCE COMPANY']
cargo                       3 (     83)                        n/a      n/a        n/a
             pre-filtered from scoring pool (carr_pct_residual_market >= 0.5): ['TEXAS MUTUAL INS CO', 'MERCURY COUNTY MUTUAL INSURANCE COMPANY', 'INFINITY COUN

### Part 7D — Fit score calibration

Bins fit scores into deciles, reports the % of OBSERVED placements that
fall in each bin. If model is well-calibrated, high-score bins should
contain disproportionately many observed placements. Lets the broker
use raw fit scores as a confidence number, not just the ordinal rank.

In [24]:
_log("starting Part 7D — fit score calibration (classifier only)")

# NOTE: calibration measures "observed-placement rate per decile of fit score".
# Only the classifier produces probabilities (predict_proba) — LambdaRank returns
# unbounded raw margins, so decile-level calibration isn't comparable.
# We use the classifier's OBS_RANK (default alias) for this diagnostic.

print("Fit score calibration (per line)")
print("For each decile of fit score, what % of (DOT, carrier) pairs are OBSERVED placements?")
print("Well-calibrated model: this fraction increases monotonically with score bin.")
print("=" * 70)

def _holdout_dots_for(line_name):
    return set(LINE_LATEST_EFF.loc[
        (LINE_LATEST_EFF['line'] == line_name) & (LINE_LATEST_EFF['latest_eff_date'] >= T_SPLIT),
        'usdot'
    ].astype('int64'))

print("(deciles computed on HOLDOUT scored pairs only)")
for line_name in ALL_PLACEMENTS.keys():
    scored_df_full = SCORED.get(line_name, pd.DataFrame())
    if scored_df_full is None or len(scored_df_full) == 0: continue
    scored_df = scored_df_full[scored_df_full['usdot'].isin(_holdout_dots_for(line_name))]
    if len(scored_df) == 0: continue
    # scored_df already has 'is_observed' column from cell 5A? No — that's added later.
    # Need to mark observed here.
    obs_pairs = ALL_PLACEMENTS[line_name][['usdot','carrier_group','state']].drop_duplicates()
    obs_pairs['_is_obs'] = np.int8(1)
    s = scored_df[['usdot','carrier_group','state','fit_score']].merge(
        obs_pairs, on=['usdot','carrier_group','state'], how='left'
    )
    s['_is_obs'] = s['_is_obs'].fillna(0).astype('int8')

    s['_bin'] = pd.qcut(s['fit_score'], q=10, labels=False, duplicates='drop')
    cal = s.groupby('_bin').agg(
        n_pairs=('fit_score', 'size'),
        n_observed=('_is_obs', 'sum'),
        min_score=('fit_score', 'min'),
        max_score=('fit_score', 'max'),
    )
    cal['pct_observed'] = (100 * cal['n_observed'] / cal['n_pairs']).round(2)

    print(f"\n--- {line_name.upper()} ---")
    print(cal.to_string())

_log("Part 7D complete")


[t=7104.6s] starting Part 7D — fit score calibration (classifier only)
Fit score calibration (per line)
For each decile of fit score, what % of (DOT, carrier) pairs are OBSERVED placements?
Well-calibrated model: this fraction increases monotonically with score bin.
(deciles computed on HOLDOUT scored pairs only)

--- BIPD ---
      n_pairs  n_observed  min_score  max_score  pct_observed
_bin                                                         
0       99746           6   0.000739   0.009061          0.01
1       99745          36   0.009061   0.016422          0.04
2       99745          70   0.016422   0.025352          0.07
3       99745         118   0.025352   0.036593          0.12
4       99745         310   0.036593   0.052655          0.31
5       99745         355   0.052655   0.076051          0.36
6       99745         542   0.076051   0.114568          0.54
7       99745        1010   0.114568   0.177027          1.01
8       99745        1872   0.177028   0.294827    

### Part 7E — Pool size vs top-K hit rate (per state)

Scatter: state's carrier pool size vs model top-5 hit rate on that state.
Confirms the "small pool => high hit rate" caveat for cargo/surety_bond.
States with large pools (CA, TX, FL) + still-high hit rates = genuine
model signal, not pool-size artifact.

In [25]:
_log("starting Part 7E — pool size vs hit rate")

print("Pool size vs top-5 hit rate (per state, per line)")
print("=" * 70)

def _holdout_dots_for(line_name):
    return set(LINE_LATEST_EFF.loc[
        (LINE_LATEST_EFF['line'] == line_name) & (LINE_LATEST_EFF['latest_eff_date'] >= T_SPLIT),
        'usdot'
    ].astype('int64'))

_eval_backends = list(BACKEND_TYPES) + (['ensemble'] if 'ensemble' in OBS_RANK_BY_BACKEND else [])
for _bt in _eval_backends:
    print(f"\n### Backend: {_bt.upper()}  (HOLDOUT ONLY) ###")
    for line_name in ALL_PLACEMENTS.keys():
        obs_full = OBS_RANK_BY_BACKEND[_bt].get(line_name, pd.DataFrame())
        if obs_full is None or len(obs_full) == 0: continue
        obs = obs_full[obs_full['usdot'].isin(_holdout_dots_for(line_name))]
        if len(obs) == 0: continue
        cells = carrier_state_agg[carrier_state_agg['line'] == line_name]
        pool_sizes = cells.groupby('state')['carrier_group'].nunique().to_dict()
        stats = obs.copy()
        stats['pool_size'] = stats['state'].map(pool_sizes)
        stats['_hit5'] = (stats['observed_rank'] <= 5).astype('int8')
        agg = stats.groupby('state').agg(
            n_dots=('usdot', 'size'),
            pool_size=('pool_size', 'first'),
            top5_hit_rate=('_hit5', 'mean'),
        ).sort_values('n_dots', ascending=False)
        corr = agg[['pool_size','top5_hit_rate']].corr().iloc[0, 1] if len(agg) >= 3 else float('nan')
        print(f"\n--- {line_name.upper()} ---  "
              f"corr(pool_size, top-5 hit) = {corr:+.3f}")
        print(agg.head(10).round(3).to_string())

_log("Part 7E complete")


[t=7106.6s] starting Part 7E — pool size vs hit rate
Pool size vs top-5 hit rate (per state, per line)

### Backend: CLASSIFIER  (HOLDOUT ONLY) ###

--- BIPD ---  corr(pool_size, top-5 hit) = -0.783
       n_dots  pool_size  top5_hit_rate
state                                  
TX       3351        190          0.626
CA       2336         97          0.640
IL        742         72          0.590
FL        673         56          0.585
NC        637         47          0.772
OH        584         70          0.642
GA        535         63          0.650
NJ        512         55          0.750
PA        467         63          0.687
IN        425         53          0.708

--- CARGO ---  corr(pool_size, top-5 hit) = -0.401
       n_dots  pool_size  top5_hit_rate
state                                  
FL         11          8          0.818
NJ         11          8          1.000
TX          7          9          1.000
CA          6          5          1.000
NY          5          6     

### Part 7F — Known-fail synthetic DOT (specialty-rejection sanity)

Construct a canonical "mainstream" DOT (mid-fleet, no crashes, non-hazmat,
standard commodity). Verify the model's top-5 does NOT include specialty
or residual-market carriers for this DOT. If it does for a prediction
where it clearly shouldn't, we have a sanity-check failure.

In [26]:
_log("starting Part 7F — known-fail synthetic DOT")

# Build a canonical standard-risk DOT by picking a real DOT with
# representative "mainstream" characteristics
_mainstream_dots = dot_df[
    (dot_df['power_units'].between(5, 50))
    & (dot_df['crash_count'] == 0)
    & (dot_df['hazmat_bin'] == 0)
    & (dot_df['crgo_genfreight'] == 1)
    & (dot_df['is_residual_market'] == 0)
    & (dot_df['is_cancellation_pending'] == 0)
    & (dot_df['state'].isin(['CA','TX','FL','PA','OH']))
]

if len(_mainstream_dots) == 0:
    print("  No canonical mainstream DOT found — skipping")
else:
    # Pick one DOT per state (top-5 states by DOT count)
    print(f"Canonical mainstream DOT top-5 recommendations per line (should be voluntary mainstream carriers)")
    print("=" * 70)
    sample_dots = _mainstream_dots.groupby('state').head(1).head(5)
    for _, dot_row in sample_dots.iterrows():
        dot = int(dot_row['usdot'])
        state = dot_row['state']
        print(f"\nDOT {dot} | state={state} | power_units={int(dot_row['power_units'])} | genfreight-only, no crashes, no hazmat")
        _eval_backends = list(BACKEND_TYPES) + (['ensemble'] if 'ensemble' in SCORED_BY_BACKEND else [])
        for _bt in _eval_backends:
            print(f"  --- {_bt} ---")
            for line_name, scored_df in SCORED_BY_BACKEND[_bt].items():
                if scored_df is None or len(scored_df) == 0: continue
                top5 = scored_df[(scored_df['usdot']==dot) & (scored_df['state']==state)].sort_values('rank').head(5)
                if len(top5) > 0:
                    recs = [f"{r['carrier_group']} ({r['fit_score']:.3f})" for _, r in top5.iterrows()]
                    print(f"    {line_name:12s}: {' | '.join(recs)}")
                else:
                    print(f"    {line_name:12s}: (not scored for this DOT)")

_log("Part 7F complete")


[t=7107.2s] starting Part 7F — known-fail synthetic DOT
Canonical mainstream DOT top-5 recommendations per line (should be voluntary mainstream carriers)

DOT 3374642 | state=TX | power_units=40 | genfreight-only, no crashes, no hazmat
  --- classifier ---
    bipd        : ZURICH INS GRP (0.840) | KEY RISK INSURANCE COMPANY (0.814) | EVEREST REINS HOLDINGS GRP (0.814) | RIVERPORT INSURANCE COMPANY (0.798) | NEW YORK MARINE & GENERAL INS. CO. (0.759)
    cargo       : (not scored for this DOT)
    wc          : TEXAS MUTUAL INS CO (0.997) | INDEMNITY INS CO OF N AMERICA (0.898) | INDEMNITY INSURANCE COMPANY OF (0.768) | ARGONAUT GRP (0.651) | FRANK WINSTON CRUM INSURANCE (0.307)
    surety_bond : (not scored for this DOT)
  --- ranker ---
    bipd        : OLD REPUBLIC GRP (1.537) | BERKLEY REGIONAL INSURANCE COMPANY (1.000) | NEW YORK MARINE & GENERAL INS. CO. (0.906) | EVEREST REINS HOLDINGS GRP (0.866) | KEY RISK INSURANCE COMPANY (0.848)
    cargo       : (not scored for this DOT)


## Phase 8 — Deliverables

### Part 8A — Combine outputs + sub-entity distance ranking

In [29]:
_log("starting Part 8A")

# Select the deliverable model's output per MODEL_FOR_DELIVERABLE (cell 1A).
if MODEL_FOR_DELIVERABLE == 'weighted':
    _SCORED, _OBS_RANK = SCORED_WEIGHTED, OBS_RANK_WEIGHTED
    print("Using WEIGHTED-tier model output for deliverable (per MODEL_FOR_DELIVERABLE='weighted').")
else:
    _SCORED, _OBS_RANK = SCORED, OBS_RANK
    print("Using BASE model output for deliverable (per MODEL_FOR_DELIVERABLE='base').")

non_empty_ranked = [df for df in _SCORED.values() if len(df) > 0]
non_empty_obs    = [df for df in _OBS_RANK.values() if len(df) > 0]

if not non_empty_ranked:
    print("No scored lines produced output. Aborting.")
    long_ranked = pd.DataFrame()
    all_obs     = pd.DataFrame()
else:
    long_ranked = pd.concat(non_empty_ranked, ignore_index=True)
    all_obs     = pd.concat(non_empty_obs,    ignore_index=True)
    print(f"Combined {len(long_ranked):,} ranked rows across lines "
          f"({sorted(long_ranked['line'].unique())})")

    observed_marker = all_obs[['usdot','line','carrier_group']].drop_duplicates()
    observed_marker['is_observed'] = np.int8(1)
    long_ranked = long_ranked.merge(observed_marker, on=['usdot','line','carrier_group'], how='left')
    long_ranked['is_observed'] = long_ranked['is_observed'].fillna(0).astype('int8')
    long_ranked = long_ranked[['usdot','line','state','rank','carrier_group',
                               'fit_score','is_observed','pool_size']]
    long_ranked = long_ranked.sort_values(['usdot','line','rank']).reset_index(drop=True)
    _log("Part 8A ranked_long combined")

    # Option B sub-entity distance ranking
    print("\nAttaching best-fit sub-entities (Option B distance)...")
    dist_feature_cols = [f'sub_{f}' for f in APPETITE_DIST_FEATURES]
    weights_arr = np.array([FEATURE_WEIGHTS.get(c, 0.0) for c in dist_feature_cols], dtype='float32')
    if weights_arr.sum() == 0:
        weights_arr = np.ones_like(weights_arr) / len(weights_arr)
    weights_arr = weights_arr / weights_arr.sum()

    sub_sum_cols = [c + '_sum' for c in dist_feature_cols]
    sub_lookup = {}
    for (g, s, l), group_df in sub_entity_centroids.groupby(['carrier_group','state','line'], sort=False):
        sub_lookup[(g, s, l)] = {
            'names': group_df['insurer_raw'].to_numpy(),
            'means': group_df[dist_feature_cols].to_numpy(dtype='float32'),
            'sums':  group_df[sub_sum_cols].to_numpy(dtype='float32'),
            'n':     group_df['sub_n_events'].to_numpy(dtype='float32'),
        }
    print(f"  built sub_lookup with {len(sub_lookup):,} (group, state, line) cells")

    # Flat dict keyed by (usdot, carrier_group, state, line, insurer_raw) -> dot_n_events.
    # Flat (one level) is lighter than nested-dict + no per-group outer entry overhead.
    dot_subent_counts = dict(zip(
        zip(dot_subent_contrib['usdot'].to_numpy(),
            dot_subent_contrib['carrier_group'].to_numpy(),
            dot_subent_contrib['state'].to_numpy(),
            dot_subent_contrib['line'].to_numpy(),
            dot_subent_contrib['insurer_raw'].to_numpy()),
        dot_subent_contrib['dot_n_events_in_subent'].to_numpy()
    ))

    def best_subentity_for_dot(dot_vals, group, state, line_label, usdot):
        entry = sub_lookup.get((group, state, line_label))
        if entry is None: return None, None
        sub_names, sub_means = entry['names'], entry['means']
        if len(sub_names) == 0: return None, None
        # LOO adjustment: subtract DOT's contribution (dot_n_events * dot_val)
        # from any sub-entity centroid where DOT actually places. Sums and ns
        # from sub_entity_centroids are over EVENTS, so a DOT with k events
        # contributes k * dot_val to the sum and k to the count.
        sums, ns = entry['sums'], entry['n']
        dot_vals_arr = np.asarray(dot_vals, dtype='float32')
        sub_vals = sub_means
        copied = False
        for i, name in enumerate(sub_names):
            dot_n = dot_subent_counts.get((usdot, group, state, line_label, name))
            if dot_n:
                if not copied:
                    sub_vals = sub_means.copy()
                    copied = True
                denom = max(float(ns[i]) - float(dot_n), 1.0)
                sub_vals[i] = (sums[i] - float(dot_n) * dot_vals_arr) / denom
        diffs = sub_vals - dot_vals
        d2 = (diffs * diffs * weights_arr).sum(axis=1)
        best_idx = int(np.argmin(d2))
        return sub_names[best_idx], float(np.sqrt(d2[best_idx]))

    top_k_per_dot_line = (long_ranked[long_ranked['rank'] <= TOP_K_PER_DOT]
                          .sort_values(['usdot','line','rank']))
    dot_dist_vals = dot_df.set_index('usdot')[APPETITE_DIST_FEATURES].astype('float32')

    best_sub_list = []
    for (usdot, line_label), group_rows in top_k_per_dot_line.groupby(['usdot','line']):
        if usdot not in dot_dist_vals.index: continue
        dot_vec = dot_dist_vals.loc[usdot].to_numpy(dtype='float32')
        for st in group_rows['state'].unique():
            state_rows = group_rows[group_rows['state'] == st]
            for grp in state_rows['carrier_group'].unique():
                sub, d = best_subentity_for_dot(dot_vec, grp, st, line_label, usdot)
                best_sub_list.append((usdot, line_label, st, grp, sub, d))

    best_sub_df = pd.DataFrame(
        best_sub_list,
        columns=['usdot','line','state','carrier_group','best_subentity','subentity_distance']
    )
    print(f"  computed {len(best_sub_df):,} best-sub-entity assignments")

    long_ranked = long_ranked.merge(
        best_sub_df[['usdot','line','state','carrier_group','best_subentity','subentity_distance']],
        on=['usdot','line','state','carrier_group'], how='left'
    )

_log("Part 8A complete")

[t=73.4s] starting Part 5A
Using WEIGHTED-tier model output for deliverable (per MODEL_FOR_DELIVERABLE='weighted').
Combined 23,047,155 ranked rows across lines (['bipd', 'cargo', 'surety_bond', 'wc'])
[t=97.5s] Part 5A ranked_long combined

Attaching best-fit sub-entities (Option B distance)...
  built sub_lookup with 3,360 (group, state, line) cells
  computed 8,498,617 best-sub-entity assignments
[t=488.1s] Part 5A complete


### Part 8B — Write artifacts (wide + long) + previews + summary

In [30]:
_log("starting Part 8B")

# Scope note (printed into output for the stakeholder):
print("Modelling scope: BIPD, cargo, WC, surety_bond (FMCSA-mandated lines)")
print("Excluded: GL and surety (non-BMC-84) — systematically NULL event_status")
print("          in the source data; see Part 3B diagnostic.")

TOP_K_WIDE  = TOP_K_PER_DOT
OUTPUT_WIDE = "rosella_appetite_outliers.csv"
OUTPUT_LONG = "rosella_rankings_long.csv"

if len(long_ranked) == 0:
    print("Nothing to write (empty long_ranked).")
else:
    long_ranked.to_csv(OUTPUT_LONG, index=False)
    print(f"Wrote {OUTPUT_LONG} ({len(long_ranked):,} rows)")

    # Build wide output: top-K recs per DOT per line, formatted as "GROUP (sub_entity)"
    top_k_slice = long_ranked[long_ranked['rank'] <= TOP_K_WIDE].copy()
    sub_mask = top_k_slice['best_subentity'].notna() & (top_k_slice['best_subentity'] != '')
    group_str = top_k_slice['carrier_group'].astype(str)
    top_k_slice['rec_str'] = np.where(
        sub_mask,
        group_str + ' (' + top_k_slice['best_subentity'].fillna('').astype(str) + ')',
        group_str,
    )
    top_k_rec = (top_k_slice.sort_values(['usdot','line','rank'])
                 .groupby(['usdot','line'])['rec_str']
                 .apply(lambda s: ' | '.join(s.values))
                 .reset_index(name=f'top{TOP_K_WIDE}_recommended'))

    # Appetite-outlier score (normalised rank)
    all_obs['appetite_outlier_score'] = (
        (all_obs['observed_rank'] - 1) / (all_obs['pool_size'] - 1).clip(lower=1)
    ).astype('float32')

    worst_line = (all_obs.sort_values(['usdot','appetite_outlier_score'], ascending=[True, False])
                  .drop_duplicates('usdot', keep='first')
                  .rename(columns={'line':'worst_line','carrier_group':'observed_carrier',
                                   'observed_rank':'observed_rank_in_line'}))
    worst_line = worst_line.merge(top_k_rec, left_on=['usdot','worst_line'],
                                  right_on=['usdot','line'], how='left').drop(columns=['line'])

    worst_line = worst_line.merge(
        dot_df[['usdot','company_name','state','phy_state','power_units','fleet_band',
                'has_authority','crash_count','inspection_oos_viol',
                'ins_distinct_insurers','ins_cancelled_count',
                'is_residual_market','is_cancellation_pending',
                'years_with_current_carrier','days_until_bipd_expiration']],
        on='usdot', how='left', suffixes=('', '_dot')
    )

    worst_line['fleet_weight'] = np.log1p(worst_line['power_units'].fillna(0)).astype('float32')
    worst_line['priority'] = (worst_line['appetite_outlier_score']
                              * (1 + 0.1 * worst_line['fleet_weight'])).astype('float32')
    worst_line = worst_line.sort_values('priority', ascending=False).reset_index(drop=True)

    # ---- confidence_tier column -------------------------------------------
    # The model's top-5 hit rate varies by (line, fleet_band) — mega-fleets
    # (B4/B5) are harder than owner-operators (B1). An "outlier" flag at B5
    # has ~3x the false-positive risk of a B1 flag. Tier makes this visible
    # to the broker rather than treating all flags as equal.
    #
    # Hit rates are drawn from OBS_RANK restricted to the post-cutoff holdout
    # (honest OOS estimate). Pre-cutoff DOTs' scores are in-sample for the
    # training set and would inflate the tier confidence.
    _tier_source_rows = []
    for _line, _obs_in in OBS_RANK.items():
        if len(_obs_in) == 0: continue
        _holdout_dots = set(LINE_LATEST_EFF.loc[
            (LINE_LATEST_EFF['line'] == _line) & (LINE_LATEST_EFF['latest_eff_date'] >= T_SPLIT),
            'usdot'
        ].astype('int64'))
        _obs = _obs_in[_obs_in['usdot'].isin(_holdout_dots)]
        if len(_obs) == 0: continue
        _o = _obs.merge(dot_df[['usdot','fleet_band']], on='usdot', how='left')
        _o['_top5'] = (_o['observed_rank'] <= 5).astype('float32')
        agg = _o.groupby(['line','fleet_band'])['_top5'].agg(['mean','count']).reset_index()
        agg = agg.rename(columns={'mean':'segment_top5','count':'segment_n'})
        _tier_source_rows.append(agg)
    if _tier_source_rows:
        _tier_lookup = pd.concat(_tier_source_rows, ignore_index=True)
        # Band thresholds — picked so that B1/B2 are typically "high" and
        # B4/B5 typically "low" (consistent with the observed hit-rate curve).
        def _tier(x):
            if pd.isna(x): return 'unknown'
            if x >= 0.85: return 'high'
            if x >= 0.70: return 'med'
            return 'low'
        _tier_lookup['confidence_tier'] = _tier_lookup['segment_top5'].map(_tier)
        worst_line = worst_line.merge(
            _tier_lookup[['line','fleet_band','segment_top5','confidence_tier']],
            left_on=['worst_line','fleet_band'], right_on=['line','fleet_band'],
            how='left'
        ).drop(columns=['line'], errors='ignore')
        worst_line['confidence_tier'] = worst_line['confidence_tier'].fillna('unknown')
    else:
        worst_line['segment_top5']    = np.nan
        worst_line['confidence_tier'] = 'unknown'

    cols_out = [c for c in [
        'usdot', 'company_name', 'state', 'phy_state', 'power_units', 'fleet_band',
        'worst_line', 'observed_carrier', 'observed_rank_in_line', 'pool_size',
        'appetite_outlier_score', f'top{TOP_K_WIDE}_recommended', 'priority',
        'confidence_tier', 'segment_top5',
        'is_residual_market', 'is_cancellation_pending', 'years_with_current_carrier',
        'days_until_bipd_expiration', 'crash_count', 'inspection_oos_viol',
        'ins_distinct_insurers', 'ins_cancelled_count', 'has_authority',
    ] if c in worst_line.columns]
    worst_line[cols_out].head(TOP_N_OUTPUT).to_csv(OUTPUT_WIDE, index=False)
    print(f"Wrote {OUTPUT_WIDE} (top {TOP_N_OUTPUT:,} DOTs; top-{TOP_K_WIDE} 'group (sub_entity)' per DOT)")

    # Previews
    print("\n" + "=" * 70)
    print("TOP 25 PREVIEW")
    print("=" * 70)
    preview_cols = [c for c in [
        'usdot','company_name','state','power_units','worst_line',
        'observed_carrier','observed_rank_in_line','pool_size',
        'appetite_outlier_score', f'top{TOP_K_WIDE}_recommended','priority'
    ] if c in worst_line.columns]
    pd.set_option('display.max_colwidth', 120)
    print(worst_line[preview_cols].head(25).to_string(index=False))

    if len(worst_line) > 0:
        print("\n" + "=" * 70)
        print("SAMPLE DOT FULL RANKING")
        print("=" * 70)
        sample_usdot = worst_line.iloc[0]['usdot']
        sample_long = (long_ranked[long_ranked['usdot'] == sample_usdot]
                       .sort_values(['line','rank']).head(30))
        print(f"USDOT {sample_usdot}: first 30 rows (across lines):")
        print(sample_long.to_string(index=False))

    print("\n" + "=" * 70)
    print("DELIVERABLE SUMMARY")
    print("=" * 70)
    print(f"Scorable DOTs ranked: {worst_line['usdot'].nunique():,}")
    print(f"Long-ranking rows written: {len(long_ranked):,}")
    print(f"\nAppetite-outlier score distribution:")
    print(worst_line['appetite_outlier_score'].describe().round(3).to_string())
    print(f"\nPriority distribution:")
    print(worst_line['priority'].describe().round(3).to_string())
    print(f"\n[Total runtime: {_elapsed()}]")

_log("Part 8B complete")


[t=488.2s] starting Part 5B
Modelling scope: BIPD, cargo, WC, surety_bond (FMCSA-mandated lines)
Excluded: GL and surety (non-BMC-84) — systematically NULL event_status
          in the source data; see Part 2F diagnostic.
Wrote rosella_rankings_v3_long.csv (23,047,155 rows)
Wrote rosella_appetite_outliers_v3.csv (top 10,000 DOTs; top-20 'group (sub_entity)' per DOT)

TOP 25 PREVIEW
  usdot                       company_name state  power_units worst_line                          observed_carrier  observed_rank_in_line  pool_size  appetite_outlier_score                                                                                                                                                                                                                                                                                                                                                                                                                                                          

## Phase 9 — Market Analysis

### Part 9A — Market structure in CA / TX (BIPD + WC)

Carrier concentration (Herfindahl index + top-5 share), residual-market
presence, and placement distribution by fleet band for the two biggest
state markets.

In [31]:
_log("starting Part 9A — market structure CA/TX")

FOCUS_STATES = ['CA', 'TX']
FOCUS_LINES  = ['bipd', 'wc']

def _market_structure(line_name, state):
    """Carrier concentration + residual-market presence for one state × line."""
    cells = carrier_state_agg[
        (carrier_state_agg['line'] == line_name)
        & (carrier_state_agg['state'] == state)
    ][['carrier_group','carr_n_placements','carr_pct_residual_market']].copy()
    if len(cells) == 0: return None
    total_n = cells['carr_n_placements'].sum()
    cells['share'] = cells['carr_n_placements'] / total_n
    # Herfindahl-Hirschman Index (sum of squared shares, 0-1 scale)
    hhi = (cells['share'] ** 2).sum()
    top5_share = cells.nlargest(5, 'carr_n_placements')['share'].sum()
    n_carriers = len(cells)
    # Residual-market presence (share-weighted)
    residual_weighted = (cells['share'] * cells['carr_pct_residual_market'].fillna(0)).sum()
    return {
        'line': line_name, 'state': state, 'n_carriers': n_carriers,
        'total_placements': int(total_n), 'hhi': hhi,
        'top5_share': top5_share, 'residual_pct_weighted': residual_weighted,
    }

print("Market concentration + residual-market presence:")
print(f"{'line':<8s} {'state':<6s} {'n_carriers':>12s} {'total':>10s} {'HHI':>8s} {'top-5':>8s} {'resid %':>10s}")
print("-" * 70)
summary = []
for line in FOCUS_LINES:
    for state in FOCUS_STATES:
        s = _market_structure(line, state)
        if s is None: continue
        summary.append(s)
        print(f"{s['line']:<8s} {s['state']:<6s} {s['n_carriers']:>12d} "
              f"{s['total_placements']:>10,} {s['hhi']:>8.3f} "
              f"{s['top5_share']:>7.1%} {s['residual_pct_weighted']:>9.1%}")

print()
print("Interpretation:")
print("  HHI: 0.00-0.15 = competitive, 0.15-0.25 = moderately concentrated, >0.25 = concentrated")
print("  top-5 share: % of placements captured by the top 5 carriers")
print("  resid %: share-weighted mean of carr_pct_residual_market across carriers in that market")
print()

# Top 10 carriers per focus state × line
print()
print("Top 10 carriers by placement count per (state, line):")
for line in FOCUS_LINES:
    for state in FOCUS_STATES:
        cells = carrier_state_agg[
            (carrier_state_agg['line'] == line)
            & (carrier_state_agg['state'] == state)
        ][['carrier_group','carr_n_placements','carr_pct_residual_market','carr_state_momentum']]
        if len(cells) == 0: continue
        top = cells.nlargest(10, 'carr_n_placements').copy()
        top['share'] = (100 * top['carr_n_placements'] / cells['carr_n_placements'].sum()).round(1)
        print(f"\n--- {line.upper()} / {state} ---")
        print(top.to_string(index=False))

_log("Part 9A complete")


[t=623.5s] starting Part 6A — market structure CA/TX
Market concentration + residual-market presence:
line     state    n_carriers      total      HHI    top-5    resid %
----------------------------------------------------------------------
bipd     CA               84     53,180    0.042   37.2%      0.0%
bipd     TX              179     81,828    0.111   57.7%      0.0%
wc       TX              136     34,475    0.226   58.8%      0.0%

Interpretation:
  HHI: 0.00-0.15 = competitive, 0.15-0.25 = moderately concentrated, >0.25 = concentrated
  top-5 share: % of placements captured by the top 5 carriers
  resid %: share-weighted mean of carr_pct_residual_market across carriers in that market


Top 10 carriers by placement count per (state, line):

--- BIPD / CA ---
                               carrier_group  carr_n_placements  carr_pct_residual_market  carr_state_momentum  share
                             PROGRESSIVE GRP               5540                       0.0             0.1

### Part 9B — Carrier momentum (CA / TX, BIPD + WC)

Who's gaining / losing share? carr_state_momentum = recent 6m placements
vs 6-18m placements. Tied to svc_voluntary_exit_ratio to tell "gaining
because carriers are leaving" from "gaining because carriers are growing."

In [32]:
_log("starting Part 9B — carrier momentum")

print("Carrier momentum per (state, line) — top gainers / losers")
print("=" * 70)

for line in FOCUS_LINES:
    for state in FOCUS_STATES:
        cells = carrier_state_agg[
            (carrier_state_agg['line'] == line)
            & (carrier_state_agg['state'] == state)
            & (carrier_state_agg['carr_n_placements'] >= 20)   # only reasonably-sized carriers
        ][['carrier_group','carr_n_placements','carr_placements_6m',
           'carr_placements_6to18m','carr_state_momentum',
           'svc_voluntary_exit_ratio','svc_short_tenure_cancel_rate']].copy()
        if len(cells) < 5: continue

        print(f"\n--- {line.upper()} / {state} ---")
        print("\nTop 10 GAINERS (highest momentum):")
        gainers = cells.nlargest(10, 'carr_state_momentum')
        print(gainers.round(3).to_string(index=False))

        print("\nTop 10 LOSERS (lowest momentum):")
        losers = cells.nsmallest(10, 'carr_state_momentum')
        print(losers.round(3).to_string(index=False))

print()
print("Interpretation:")
print("  carr_state_momentum: ratio of 6m placements to (6-18m / 2 + 1); > 1 = growing")
print("  svc_voluntary_exit_ratio: % of ended policies that ended voluntarily (DOT chose to leave)")
print("    low voluntary_exit % + high momentum => carrier gaining AND retaining — healthy growth")
print("    high voluntary_exit % + low momentum => churn-driven decline")

_log("Part 9B complete")


[t=623.6s] starting Part 6B — carrier momentum
Carrier momentum per (state, line) — top gainers / losers

--- BIPD / CA ---

Top 10 GAINERS (highest momentum):
                               carrier_group  carr_n_placements  carr_placements_6m  carr_placements_6to18m  carr_state_momentum  svc_voluntary_exit_ratio  svc_short_tenure_cancel_rate
           INFINITY SELECT INSURANCE COMPANY                 22                  10                      12                1.429                     0.600                         0.071
          COMMODORE RISK RETENTION GROUP INC                 23                  10                      13                1.333                     0.579                         0.028
UNIVERSAL CASUALTY RISK RETENTION GROUP, INC               1029                 194                     836                0.463                     0.291                         0.218
       SPINNAKER SPECIALTY INSURANCE COMPANY                188                  38                 

### Part 9C — Appetite outliers in CA / TX (deliverable model)

For each focus (line, state), find the top-50 DOTs where the current
placement looks like the WORST fit — defined as observed_rank in the
bottom decile of the state's pool. These are the highest-value calls.
Uses the deliverable model's output (SCORED_WEIGHTED or SCORED).

In [33]:
_log("starting Part 9C — appetite outliers CA/TX")

# Pick deliverable model
if MODEL_FOR_DELIVERABLE == 'weighted':
    _OBS, _SCORED_DF = OBS_RANK_WEIGHTED, SCORED_WEIGHTED
else:
    _OBS, _SCORED_DF = OBS_RANK, SCORED

TOP_N_PER_STATE_LINE = 25

print(f"Top {TOP_N_PER_STATE_LINE} appetite outliers per (line, state)")
print(f"(current carrier ranks in the bottom decile of state pool size)")
print("=" * 70)

_dot_lookup = dot_df.set_index('usdot')[
    ['state','power_units','fleet_band','hazmat_bin','crash_count',
     'inspection_oos_viol','years_with_current_carrier']
]

for line in FOCUS_LINES:
    obs = _OBS.get(line, pd.DataFrame())
    scored = _SCORED_DF.get(line, pd.DataFrame())
    if len(obs) == 0 or len(scored) == 0: continue
    for state in FOCUS_STATES:
        obs_st = obs[obs['state'] == state].copy()
        if len(obs_st) == 0: continue
        # Bottom-decile cutoff
        pool_size = int(obs_st['pool_size'].max()) if 'pool_size' in obs_st.columns else int(obs_st['observed_rank'].max())
        cutoff = int(pool_size * 0.9)   # bottom 10%
        outliers = obs_st[obs_st['observed_rank'] >= cutoff].copy()
        if len(outliers) == 0: continue
        outliers = outliers.sort_values('observed_rank', ascending=False).head(TOP_N_PER_STATE_LINE)

        print(f"\n--- {line.upper()} / {state} (pool size {pool_size}, cutoff rank {cutoff}) ---")
        for _, row in outliers.iterrows():
            dot = int(row['usdot'])
            if dot not in _dot_lookup.index: continue
            info = _dot_lookup.loc[dot]
            pu = f"{int(info['power_units'])}" if pd.notna(info['power_units']) else "?"
            tenure = f"{info['years_with_current_carrier']:.1f}y" if pd.notna(info['years_with_current_carrier']) else "?"
            # Top-3 model recommendations
            top3 = scored[(scored['usdot']==dot) & (scored['state']==state)].sort_values('rank').head(3)
            recs = [f"{r['carrier_group']} ({r['fit_score']:.2f})" for _, r in top3.iterrows()]
            print(f"  DOT {dot:>8} | observed rank {int(row['observed_rank']):>3}/{pool_size} "
                  f"| actual={row['carrier_group']:<35s}")
            print(f"           fleet={pu:>4s} band={info['fleet_band']} hazmat={int(info['hazmat_bin']) if pd.notna(info['hazmat_bin']) else '?'}"
                  f" crashes={int(info['crash_count']) if pd.notna(info['crash_count']) else '?'}"
                  f" tenure={tenure}")
            print(f"           top-3 recs: {' | '.join(recs)}")

_log("Part 9C complete")


[t=623.7s] starting Part 6C — appetite outliers CA/TX
Top 25 appetite outliers per (line, state)
(current carrier ranks in the bottom decile of state pool size)

--- BIPD / CA (pool size 84, cutoff rank 75) ---
  DOT   859192 | observed rank  83/84 | actual=OBSIDIAN SPECIALTY INSURANCE COMPANY
           fleet=   2 band=B2 hazmat=1 crashes=0 tenure=1.7y
           top-3 recs: GREAT WEST (0.81) | TRAVELERS GRP (0.78) | IAT GRP (0.39)
  DOT  3848901 | observed rank  82/84 | actual=OBSIDIAN SPECIALTY INSURANCE COMPANY
           fleet=   1 band=B1 hazmat=1 crashes=0 tenure=3.9y
           top-3 recs: PROGRESSIVE GRP (0.92) | TRAVELERS GRP (0.86) | GREAT WEST (0.33)
  DOT   507828 | observed rank  82/84 | actual=PALOMAR SPECIALTY INSURANCE COMPANY
           fleet=   4 band=B2 hazmat=1 crashes=0 tenure=1.2y
           top-3 recs: GREAT WEST (0.69) | BERKSHIRE HATHAWAY GRP (0.65) | STAR MUTUAL RRG (0.47)
  DOT  4417612 | observed rank  82/84 | actual=PALOMAR SPECIALTY INSURANCE COMPANY
    

## Phase 10 — Recommendation Hygiene & Multi-line Analysis

### Part 10A — Multi-line bundling analysis

For each DOT scored on multiple lines, how often does the same carrier
appear in top-5 across lines? High overlap => multi-line pitch opportunity
(brokers can offer multi-line discounts).

In [ ]:
_log("starting Part 10A — multi-line bundling")

_scored_for_bundling = SCORED_WEIGHTED if MODEL_FOR_DELIVERABLE == 'weighted' else SCORED

# Build per-(DOT, state, line) top-5
_top5_parts = []
for line_name, df in _scored_for_bundling.items():
    if df is None or len(df) == 0: continue
    t = df[df['rank'] <= 5][['usdot','state','carrier_group']].copy()
    t['line'] = line_name
    _top5_parts.append(t)

if not _top5_parts:
    print("No scored output; skipping.")
    BUNDLING_PER_DOT = pd.DataFrame(columns=['usdot','state','n_bundling_carriers'])
else:
    combined = pd.concat(_top5_parts, ignore_index=True)
    print(f"Total DOTs with top-5 across any line: {combined['usdot'].nunique():,}")

    # Count distinct lines each (DOT, state, carrier) appears in
    per_carrier = combined.groupby(['usdot','state','carrier_group'])['line'].nunique().reset_index(name='n_lines')

    # Carriers that bundle (appear in 2+ lines' top-5 for same DOT)
    bundling = per_carrier[per_carrier['n_lines'] >= 2]
    BUNDLING_PER_DOT = bundling.groupby(['usdot','state']).size().reset_index(name='n_bundling_carriers')

    # Distribution across all scored DOTs
    all_scored_dots = combined[['usdot','state']].drop_duplicates()
    all_scored_dots = all_scored_dots.merge(BUNDLING_PER_DOT, on=['usdot','state'], how='left')
    all_scored_dots['n_bundling_carriers'] = all_scored_dots['n_bundling_carriers'].fillna(0).astype(int)

    print(f"\nDistribution of DOTs by # bundling carriers (same carrier in >=2 lines' top-5):")
    print(all_scored_dots['n_bundling_carriers'].value_counts().sort_index().to_string())
    print(f"\n  DOTs with 1+ bundling carrier: {(all_scored_dots['n_bundling_carriers'] >= 1).mean()*100:.1f}%")
    print(f"  DOTs with 2+ bundling carriers: {(all_scored_dots['n_bundling_carriers'] >= 2).mean()*100:.1f}%")

    # Top-20 bundling carriers by # DOTs where they bundle
    print(f"\nTop 20 bundling carriers (appearing in >=2 lines' top-5 for the same DOT):")
    top_bundlers = (bundling.groupby('carrier_group')
                    .agg(n_dots=('usdot','nunique'),
                         avg_lines_covered=('n_lines','mean'),
                         max_lines=('n_lines','max'))
                    .sort_values('n_dots', ascending=False).head(20))
    print(top_bundlers.round(2).to_string())

    # Example triple-line DOTs
    triple = per_carrier[per_carrier['n_lines'] >= 3]
    if len(triple) > 0:
        print(f"\n{triple['usdot'].nunique():,} DOTs have a carrier in top-5 of 3+ lines (strongest bundling)")
        print("\nSample 15:")
        print(triple.sort_values('n_lines', ascending=False).head(15).to_string(index=False))

_log("Part 10A complete")


### Part 10B — Recommendation hygiene (filters + flags)

Post-processing layer on long_ranked. Addresses false-positive sources:
  Hard filters (removed from top-K):
    - Exiting carriers (momentum < 0.1 AND placements_6m < 5)
    - Previously-rejected (DOT, carrier) pairs (rejected filing in last 24m)
    - Inactive DOTs (census_status_code != active) — flagged, not filtered out
  Soft flags (shown in output):
    - ⚠ declining     (momentum < 0.5 OR svc_voluntary_exit_ratio > 0.5)
    - 🔄 high-churn   (svc_short_tenure_cancel_rate > 0.2)
    - ❌ high-rejection (svc_rejection_rate > 0.15)
    - 💭 low-confidence (DOT's #1 rec fit_score < 0.3)
  Priority bonus:
    - +10% per bundling carrier (from 7A) — multi-line pitches worth more.
Writes new CSVs with _hygiene suffix. Original 5B CSVs remain for comparison.

In [ ]:
_log("starting Part 10B — recommendation hygiene")

MOMENTUM_EXIT_THRESHOLD     = 0.1
MOMENTUM_EXIT_6M_MAX        = 5
MOMENTUM_DECLINE_THRESHOLD  = 0.5
VOLUNTARY_EXIT_HIGH         = 0.5
SHORT_TENURE_CANCEL_HIGH    = 0.2
REJECTION_RATE_HIGH         = 0.15
LOW_FIT_CONFIDENCE_THRESHOLD = 0.3
BUNDLING_PRIORITY_BONUS     = 0.10   # +10% priority per bundling carrier

# ---- Rejected pairs (DOT, carrier) over last 24m ----
print("Building rejected (DOT, carrier) pairs from raw events...")
rejected_pairs_df = con.execute(f"""
    SELECT DISTINCT r.usdot, cl.carrier_group
    FROM raw r
    JOIN carrier_lookup cl ON UPPER(TRIM(CAST(r.insurer AS VARCHAR))) = cl.insurer_raw
    WHERE LOWER(TRIM(CAST(r.event_status AS VARCHAR))) = 'rejected'
      AND r.effective_date >= {_WINDOW_START_SQL}
      AND r.effective_date <= {_VINTAGE_SQL}
      AND r.usdot IS NOT NULL
""").df()
rejected_pairs_df['usdot'] = rejected_pairs_df['usdot'].astype('int64')
rejected_pairs_df['_rejected'] = np.int8(1)
print(f"  {len(rejected_pairs_df):,} (DOT, carrier) rejection pairs in last 24m")

# ---- Operational status ----
if 'census_status_code' in dot_df.columns:
    _status_vc = dot_df['census_status_code'].astype(str).str.strip().str.upper().value_counts().head(10)
    print(f"\nTop 10 census_status_code values:")
    print(_status_vc.to_string())
    _active_status = dot_df['census_status_code'].astype(str).str.strip().str.upper().isin(['A','ACTIVE'])
    _is_active_df = dot_df[['usdot']].assign(_is_active=_active_status.astype('int8').values)
    print(f"\nDOTs marked active: {_is_active_df['_is_active'].sum():,} of {len(_is_active_df):,}")
else:
    _is_active_df = dot_df[['usdot']].copy()
    _is_active_df['_is_active'] = np.int8(1)

# ---- Carrier hygiene attributes per (carrier_group, state, line) ----
carrier_hygiene = carrier_state_agg[[
    'carrier_group','state','line',
    'carr_state_momentum','carr_placements_6m',
    'svc_voluntary_exit_ratio','svc_short_tenure_cancel_rate','svc_rejection_rate'
]].copy()
_exit = ((carrier_hygiene['carr_state_momentum'] < MOMENTUM_EXIT_THRESHOLD)
         & (carrier_hygiene['carr_placements_6m'].fillna(0) < MOMENTUM_EXIT_6M_MAX))
_decline = ((carrier_hygiene['carr_state_momentum'] < MOMENTUM_DECLINE_THRESHOLD)
            | (carrier_hygiene['svc_voluntary_exit_ratio'].fillna(0) > VOLUNTARY_EXIT_HIGH))
carrier_hygiene['momentum_tier'] = np.where(_exit, 'exiting',
                                     np.where(_decline, 'declining', 'stable'))
carrier_hygiene['high_churn_flag']     = (carrier_hygiene['svc_short_tenure_cancel_rate'].fillna(0) > SHORT_TENURE_CANCEL_HIGH).astype('int8')
carrier_hygiene['high_rejection_flag'] = (carrier_hygiene['svc_rejection_rate'].fillna(0) > REJECTION_RATE_HIGH).astype('int8')
print(f"\nCarrier-cell momentum tier distribution:")
print(carrier_hygiene['momentum_tier'].value_counts().to_string())

# ---- Merge all hygiene into long_ranked ----
print("\nMerging hygiene into long_ranked...")
_hygiene_cols = ['carrier_group','state','line','momentum_tier','high_churn_flag','high_rejection_flag',
                 'carr_state_momentum','svc_voluntary_exit_ratio','svc_short_tenure_cancel_rate','svc_rejection_rate']
long_ranked_h = long_ranked.merge(carrier_hygiene[_hygiene_cols], on=['carrier_group','state','line'], how='left')
long_ranked_h = long_ranked_h.merge(rejected_pairs_df[['usdot','carrier_group','_rejected']],
                                     on=['usdot','carrier_group'], how='left')
long_ranked_h['previously_rejected'] = long_ranked_h['_rejected'].fillna(0).astype('int8')
long_ranked_h = long_ranked_h.drop(columns=['_rejected'])
long_ranked_h = long_ranked_h.merge(_is_active_df, on='usdot', how='left')
long_ranked_h['_is_active'] = long_ranked_h['_is_active'].fillna(1).astype('int8')

# ---- Hard filter: keep_in_topk ----
long_ranked_h['keep_in_topk'] = (
    (long_ranked_h['momentum_tier'].fillna('stable') != 'exiting')
    & (long_ranked_h['previously_rejected'] == 0)
).astype('int8')
_n_filtered = (long_ranked_h['keep_in_topk'] == 0).sum()
print(f"  {_n_filtered:,} rows marked keep_in_topk=0 "
      f"(exiting carriers + previously-rejected pairs)")

# ---- Low-confidence DOT flag (based on top-1 among kept carriers) ----
_kept = long_ranked_h[long_ranked_h['keep_in_topk'] == 1].sort_values(['usdot','line','rank'])
_top1 = _kept.groupby(['usdot','line']).head(1)[['usdot','line','fit_score']].rename(columns={'fit_score': '_top1_fit'})
long_ranked_h = long_ranked_h.merge(_top1, on=['usdot','line'], how='left')
long_ranked_h['dot_low_confidence'] = (long_ranked_h['_top1_fit'] < LOW_FIT_CONFIDENCE_THRESHOLD).astype('int8')
long_ranked_h = long_ranked_h.drop(columns=['_top1_fit'])
_n_lowconf_dots = long_ranked_h[long_ranked_h['dot_low_confidence'] == 1][['usdot','line']].drop_duplicates().shape[0]
print(f"  {_n_lowconf_dots:,} (DOT, line) pairs flagged dot_low_confidence")

# ---- Re-rank within DOT after hygiene filter ----
_kept_rows = long_ranked_h[long_ranked_h['keep_in_topk'] == 1].copy()
_kept_rows = _kept_rows.sort_values(['usdot','line','rank']).reset_index(drop=True)
_kept_rows['hygiene_rank'] = _kept_rows.groupby(['usdot','line']).cumcount() + 1
long_ranked_h = long_ranked_h.merge(
    _kept_rows[['usdot','line','state','carrier_group','hygiene_rank']],
    on=['usdot','line','state','carrier_group'], how='left'
)

# ---- Write enhanced long CSV ----
_out_cols_long = ['usdot','line','state','rank','hygiene_rank','carrier_group','fit_score',
                   'is_observed','pool_size','best_subentity','subentity_distance',
                   'momentum_tier','high_churn_flag','high_rejection_flag',
                   'previously_rejected','keep_in_topk','_is_active','dot_low_confidence',
                   'carr_state_momentum','svc_voluntary_exit_ratio','svc_short_tenure_cancel_rate','svc_rejection_rate']
long_ranked_h[_out_cols_long].to_csv('rosella_rankings_long_hygiene.csv', index=False)
print(f"\nWrote rosella_rankings_long_hygiene.csv ({len(long_ranked_h):,} rows)")

# ---- Rebuild wide output with hygiene filtering + flags ----
print("\nBuilding hygiene-filtered wide output...")

def _format_rec_h(row):
    g = row['carrier_group']
    s = row['best_subentity'] if (pd.notna(row['best_subentity']) and row['best_subentity'] != '') else g
    flags = ''
    if row['momentum_tier'] == 'declining': flags += '⚠'
    if row['high_churn_flag'] == 1: flags += '🔄'
    if row['high_rejection_flag'] == 1: flags += '❌'
    return (flags + ' ' if flags else '') + f"{g} ({s})"

# Build per-DOT per-line top-20 using hygiene_rank
_wide_src = _kept_rows[_kept_rows['hygiene_rank'] <= TOP_K_PER_DOT].copy()
_wide_src['rec_str'] = _wide_src.apply(_format_rec_h, axis=1)

_wide_grouped = _wide_src.groupby(['usdot','line','state']).agg(
    top20_recommended=('rec_str', lambda s: ' | '.join(s)),
    best_fit=('fit_score', 'max'),
    kept_pool_size=('hygiene_rank', 'max'),
).reset_index()

# Find each DOT's worst (observed_rank / pool_size) across lines
observed = long_ranked_h[(long_ranked_h['is_observed'] == 1) & (long_ranked_h['keep_in_topk'] == 1)].copy()
observed['appetite_outlier_score'] = observed['hygiene_rank'] / observed['pool_size'].clip(lower=1)
worst_per_dot = observed.loc[observed.groupby('usdot')['appetite_outlier_score'].idxmax()].copy()
worst_per_dot = worst_per_dot.rename(columns={
    'carrier_group':'observed_carrier',
    'hygiene_rank':'observed_rank_in_line',
    'line':'worst_line',
})

# Attach wide recommendation strings
wide = worst_per_dot.merge(
    _wide_grouped.rename(columns={'line':'worst_line'}),
    on=['usdot','state','worst_line'], how='left'
)

# Attach DOT info
wide = wide.merge(dot_df[['usdot','company_name','power_units','fleet_band']], on='usdot', how='left')

# Attach bundling bonus from 7A
if 'BUNDLING_PER_DOT' in dir() and len(BUNDLING_PER_DOT) > 0:
    wide = wide.merge(BUNDLING_PER_DOT, on=['usdot','state'], how='left')
    wide['n_bundling_carriers'] = wide['n_bundling_carriers'].fillna(0).astype(int)
else:
    wide['n_bundling_carriers'] = 0

# Attach low-confidence flag (if any of DOT's lines are low-confidence)
_lowconf_per_dot = long_ranked_h.groupby('usdot')['dot_low_confidence'].max().reset_index()
wide = wide.merge(_lowconf_per_dot, on='usdot', how='left')
wide['dot_low_confidence'] = wide['dot_low_confidence'].fillna(0).astype('int8')

# Compute priority with bundling bonus
_power_scale = np.log1p(wide['power_units'].fillna(0)) / np.log1p(wide['power_units'].fillna(0).max())
wide['priority'] = (wide['appetite_outlier_score']
                    * (1 + 0.5 * _power_scale)
                    * (1 + BUNDLING_PRIORITY_BONUS * wide['n_bundling_carriers'])
                    * (1 - 0.3 * wide['dot_low_confidence']))

wide = wide.sort_values('priority', ascending=False).head(TOP_N_OUTPUT).reset_index(drop=True)

_wide_cols = ['usdot','company_name','state','power_units','fleet_band','worst_line',
              'observed_carrier','observed_rank_in_line','pool_size','appetite_outlier_score',
              'n_bundling_carriers','dot_low_confidence','top20_recommended','priority']
wide[_wide_cols].to_csv('rosella_appetite_outliers_hygiene.csv', index=False)
print(f"Wrote rosella_appetite_outliers_hygiene.csv ({len(wide):,} rows)")

# ---- Compare original vs hygiene summary ----
print("\n" + "=" * 70)
print("ORIGINAL vs HYGIENE deliverable summary:")
print("=" * 70)
print(f"Original top-N:               {TOP_N_OUTPUT}")
print(f"Hygiene top-N:                {len(wide)}")
print(f"DOTs flagged low-confidence:  {wide['dot_low_confidence'].sum():,}")
print(f"DOTs with 1+ bundling:        {(wide['n_bundling_carriers'] >= 1).sum():,}")
print(f"DOTs with 2+ bundling:        {(wide['n_bundling_carriers'] >= 2).sum():,}")
print(f"Priority range:               {wide['priority'].min():.3f} to {wide['priority'].max():.3f}")

_log("Part 10B complete")
